In [ ]:
def load_charmap(filepath):
    decode_map = {}
    encode_map = {}

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            # Skip comments and blank lines
            if not line or line.startswith("//"):
                continue

            # Handle lines like: 'A' = BB
            if '=' in line:
                parts = line.split('=')
                if len(parts) != 2:
                    continue

                left = parts[0].strip()
                right = parts[1].strip().split()[0]  # Only take the first byte if there are multiple

                # Extract character between quotes, e.g., 'A'
                if left.startswith("'") and left.endswith("'"):
                    char = left[1:-1]
                else:
                    continue  # Skip things like VERSION = ...

                try:
                    byte = int(right, 16)
                    decode_map[byte] = char
                    encode_map[char] = byte
                except ValueError:
                    continue  # Skip invalid hex values

    return decode_map, encode_map




In [ ]:
import json
d_map, e_map = load_charmap('charmap.txt')
print(e_map)
json.dump(e_map, open('char_encode.json', 'w'))
json.dump(d_map, open('char_decode.json', 'w'))

{' ': 0, 'À': 1, 'Á': 2, 'Â': 3, 'Ç': 4, 'È': 5, 'É': 6, 'Ê': 7, 'Ë': 8, 'Ì': 9, 'Î': 11, 'Ï': 12, 'Ò': 13, 'Ó': 14, 'Ô': 15, 'Œ': 16, 'Ù': 17, 'Ú': 18, 'Û': 19, 'Ñ': 20, 'ß': 21, 'à': 22, 'á': 23, 'ç': 25, 'è': 26, 'é': 27, 'ê': 28, 'ë': 29, 'ì': 30, 'î': 32, 'ï': 33, 'ò': 34, 'ó': 35, 'ô': 36, 'œ': 37, 'ù': 38, 'ú': 39, 'û': 40, 'ñ': 41, 'º': 42, 'ª': 43, '&': 45, '+': 46, ';': 54, '¿': 81, '¡': 82, 'Í': 90, '%': 91, '(': 92, ')': 93, 'â': 104, 'í': 111, '<': 133, '>': 134, '0': 161, '1': 162, '2': 163, '3': 164, '4': 165, '5': 166, '6': 167, '7': 168, '8': 169, '9': 170, '!': 171, '?': 172, '.': 173, '-': 174, '·': 175, '…': 176, '“': 177, '”': 178, '‘': 179, '’': 180, "\\'": 180, '♂': 181, '♀': 182, '¥': 183, ',': 184, '×': 185, '/': 186, 'A': 187, 'B': 188, 'C': 189, 'D': 190, 'E': 191, 'F': 192, 'G': 193, 'H': 194, 'I': 195, 'J': 196, 'K': 197, 'L': 198, 'M': 199, 'N': 200, 'O': 201, 'P': 202, 'Q': 203, 'R': 204, 'S': 205, 'T': 206, 'U': 207, 'V': 208, 'W': 209, 'X': 210, 'Y': 21

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
!pip install pokebase -q

In [ ]:
import pandas as pd
import numpy as np
datasheets = pd.read_excel("/content/Pokemon Gen 3 Trainers.xlsx", sheet_name=["Emerald Swampert", "FRLG Charizard", "FRLG Blastoise", "FRLG Venusaur", "RubySapphire Swampert"])


In [ ]:
sheet_names = list(datasheets.keys())
sheet_names

['Emerald Swampert',
 'FRLG Charizard',
 'FRLG Blastoise',
 'FRLG Venusaur',
 'RubySapphire Swampert']

In [ ]:
data = datasheets[sheet_names[1]]
data["IVS"] = [{"hp": 0, "atk": 0, "def": 0, "spa": 0, "spd": 0, "spe": 0}]*len(data)
data["EVS"] = [{"hp": 0, "atk": 0, "def": 0, "spa": 0, "spd": 0, "spe": 0}]*len(data)
data["Nature"] = ['Hardy']*len(data)

In [ ]:
data.drop(
    [
    'Location on Route',
    'HP',
    'Attack',
    'Defense',
    'Sp. Attack',
    'Sp Defense',
    'Speed'
    ], axis=1, inplace=True)

In [ ]:
import numpy as np
for i in range(len(data['Opponent'])):
  if pd.isna(data['Opponent'][i]):
    data.loc[i,'Opponent'] = data['Opponent'][i-1]


In [ ]:
# setting ivs for trainers

standard = {
    "Pokemaniac":30,
    "Psychic": 50,
    "Tamer": 40,
    "Black Belt": 100,
    "22 Rival":50,
    "Giovanni": 250,
    "Cooltrainer": 100,
    "Elite 4": 250,
    "Tower Rival": 100,
    "Champ" : 255,
    "Cool Couple": 100,
    "Crush": 50,
    "Pokemon Breeder": 30,
    "Pokemon Ranger" : 100,
    "Painter": 50,
    "Team Rocket Admin 1": 150,
    "Team Rocket Admin 2": 200,
    "Elite 4 Rematch": 255,
    }
ace = {
    "24 Rival": [50, 50 , 50, 100],
    "S.S Anne Rival": [50, 50 , 50, 100],
    "Silph Rival": [100, 100, 100, 100, 150],
    "22 Rival 2": [150, 150, 150, 150, 150, 250]
    }
exceptions = {
    "Psychic Laura": 100,
    "Psychic Jaclyn": 100,
    "Psychic Rodette": 100,
    "Psychic Dario": 100,
    "Cue Ball Paxton": 50
    }

In [ ]:

def get_ivs(iv_mod):
  iv = int(iv_mod * 31/255)
  return {"hp": iv, "atk": iv, "def": iv, "spa": iv, "spd": iv, "spe": iv}


In [ ]:
count = 0
for index, row in data.iterrows():
  if index != 0:
    if str(data.loc[index]['Opponent']) == data.loc[index - 1]['Opponent']:
      count += 1
    else:
      count = 0
  for i in list(exceptions.keys()):
    if i == str(row['Opponent']):
      data.loc[index, ('IVS','EVS')] = (get_ivs(exceptions[i]), {"hp": 0, "atk": 0, "def": 0, "spa": 0, "spd": 0, "spe": 0})
      break
  for i in list(standard.keys()):
    if i == str(row['Opponent'])[:len(i)]:
      data.loc[index, ('IVS','EVS')] = (get_ivs(standard[i]), {"hp": 0, "atk": 0, "def": 0, "spa": 0, "spd": 0, "spe": 0})
      break
  for i in list(ace.keys()):
    if i == str(row['Opponent'])[:len(i)]:
      data.loc[index, ('IVS','EVS')] = (get_ivs(ace[i][count]), {"hp": 0, "atk": 0, "def": 0, "spa": 0, "spd": 0, "spe": 0})
      break

In [ ]:
data

In [ ]:
data.to_csv('FRLG Charizard.csv')

In [ ]:
natures = [
  'Hardy', 'Lonely', 'Brave', 'Adamant', 'Naughty',
  'Bold', 'Docile', 'Relaxed', 'Impish', 'Lax',
  'Timid', 'Hasty', 'Serious', 'Jolly', 'Naive',
  'Modest', 'Mild', 'Quiet', 'Bashful', 'Rash',
  'Calm', 'Gentle', 'Sassy', 'Careful', 'Quirky'
]

test_data = ['BROCK', 'GEODUDE', 'BROCK', 'ONIX']


In [ ]:
import json

# Open and read the JSON file
with open('char_encode.json', 'r') as file:
    data = json.load(file)

# Print the data
print(data)

{' ': 0, 'À': 1, 'Á': 2, 'Â': 3, 'Ç': 4, 'È': 5, 'É': 6, 'Ê': 7, 'Ë': 8, 'Ì': 9, 'Î': 11, 'Ï': 12, 'Ò': 13, 'Ó': 14, 'Ô': 15, 'Œ': 16, 'Ù': 17, 'Ú': 18, 'Û': 19, 'Ñ': 20, 'ß': 21, 'à': 22, 'á': 23, 'ç': 25, 'è': 26, 'é': 27, 'ê': 28, 'ë': 29, 'ì': 30, 'î': 32, 'ï': 33, 'ò': 34, 'ó': 35, 'ô': 36, 'œ': 37, 'ù': 38, 'ú': 39, 'û': 40, 'ñ': 41, 'º': 42, 'ª': 43, '&': 45, '+': 46, ';': 54, '¿': 81, '¡': 82, 'Í': 90, '%': 91, '(': 92, ')': 93, 'â': 104, 'í': 111, '<': 133, '>': 134, '0': 161, '1': 162, '2': 163, '3': 164, '4': 165, '5': 166, '6': 167, '7': 168, '8': 169, '9': 170, '!': 171, '?': 172, '.': 173, '-': 174, '·': 175, '…': 176, '“': 177, '”': 178, '‘': 179, '’': 180, "\\'": 180, '♂': 181, '♀': 182, '¥': 183, ',': 184, '×': 185, '/': 186, 'A': 187, 'B': 188, 'C': 189, 'D': 190, 'E': 191, 'F': 192, 'G': 193, 'H': 194, 'I': 195, 'J': 196, 'K': 197, 'L': 198, 'M': 199, 'N': 200, 'O': 201, 'P': 202, 'Q': 203, 'R': 204, 'S': 205, 'T': 206, 'U': 207, 'V': 208, 'W': 209, 'X': 210, 'Y': 21

In [ ]:
counter = 0
for i in range(len(test_data)):
    for j in test_data[i]:
        counter += data[j]
    if i%2==1:
        counter*=256
        counter += 136
        print(test_data[i], natures[counter%25])
        counter = ((counter-136)//256)%25


GEODUDE Jolly
ONIX Careful


In [ ]:
true = True
false = False
move_data = {
  "1": {
    "name": "Pound",
    "type": "Normal",
    "power": 40,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "2": {
    "name": "Karate Chop",
    "type": "Fighting",
    "power": 50,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "3": {
    "name": "Double Slap",
    "type": "Normal",
    "power": 15,
    "accuracy": 85,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": true,
    "minHits": 2,
    "maxHits": 5
  },
  "4": {
    "name": "Comet Punch",
    "type": "Normal",
    "power": 18,
    "accuracy": 85,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": true,
    "minHits": 2,
    "maxHits": 5
  },
  "5": {
    "name": "Mega Punch",
    "type": "Normal",
    "power": 80,
    "accuracy": 85,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "6": {
    "name": "Pay Day",
    "type": "Normal",
    "power": 40,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "7": {
    "name": "Fire Punch",
    "type": "Fire",
    "power": 75,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "8": {
    "name": "Ice Punch",
    "type": "Ice",
    "power": 75,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "9": {
    "name": "Thunder Punch",
    "type": "Electric",
    "power": 75,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "10": {
    "name": "Scratch",
    "type": "Normal",
    "power": 40,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "11": {
    "name": "Vice Grip",
    "type": "Normal",
    "power": 55,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "12": {
    "name": "Guillotine",
    "type": "Normal",
    "power": " ",
    "accuracy": 30,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "13": {
    "name": "Razor Wind",
    "type": "Normal",
    "power": 80,
    "accuracy": 100,
    "category": "Physical",
    "target": "multiple",
    "contact": false,
    "multiple": false
  },
  "14": {
    "name": "Swords Dance",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "15": {
    "name": "Cut",
    "type": "Normal",
    "power": 50,
    "accuracy": 95,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "16": {
    "name": "Gust",
    "type": "Flying",
    "power": 40,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "multiple": false
  },
  "17": {
    "name": "Wing Attack",
    "type": "Flying",
    "power": 60,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "18": {
    "name": "Whirlwind",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "19": {
    "name": "Fly",
    "type": "Flying",
    "power": 70,
    "accuracy": 95,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "20": {
    "name": "Bind",
    "type": "Normal",
    "power": 15,
    "accuracy": 75,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "21": {
    "name": "Slam",
    "type": "Normal",
    "power": 80,
    "accuracy": 75,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "22": {
    "name": "Vine Whip",
    "type": "Grass",
    "power": 35,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "23": {
    "name": "Stomp",
    "type": "Normal",
    "power": 65,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "24": {
    "name": "Double Kick",
    "type": "Fighting",
    "power": 30,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": true,
    "minHits": 2,
    "maxHits": 2
  },
  "25": {
    "name": "Mega Kick",
    "type": "Normal",
    "power": 120,
    "accuracy": 75,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "26": {
    "name": "Jump Kick",
    "type": "Fighting",
    "power": 70,
    "accuracy": 95,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "27": {
    "name": "Rolling Kick",
    "type": "Fighting",
    "power": 60,
    "accuracy": 85,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "28": {
    "name": "Sand Attack",
    "type": "Ground",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "29": {
    "name": "Headbutt",
    "type": "Normal",
    "power": 70,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "30": {
    "name": "Horn Attack",
    "type": "Normal",
    "power": 65,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "31": {
    "name": "Fury Attack",
    "type": "Normal",
    "power": 15,
    "accuracy": 85,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": true,
    "minHits": 2,
    "maxHits": 5
  },
  "32": {
    "name": "Horn Drill",
    "type": "Normal",
    "power": " ",
    "accuracy": 30,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "33": {
    "name": "Tackle",
    "type": "Normal",
    "power": 35,
    "accuracy": 95,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "34": {
    "name": "Body Slam",
    "type": "Normal",
    "power": 85,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "35": {
    "name": "Wrap",
    "type": "Normal",
    "power": 15,
    "accuracy": 85,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "36": {
    "name": "Take Down",
    "type": "Normal",
    "power": 90,
    "accuracy": 85,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "37": {
    "name": "Thrash",
    "type": "Normal",
    "power": 90,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "38": {
    "name": "Double Edge",
    "type": "Normal",
    "power": 120,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "39": {
    "name": "Tail Whip",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "multiple",
    "multiple": false
  },
  "40": {
    "name": "Poison Sting",
    "type": "Poison",
    "power": 15,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "41": {
    "name": "Twineedle",
    "type": "Bug",
    "power": 25,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false,
    "multiple": true,
    "minHits": 2,
    "maxHits": 2
  },
  "42": {
    "name": "Pin Missile",
    "type": "Bug",
    "power": 14,
    "accuracy": 85,
    "category": "Physical",
    "target": "single",
    "contact": false,
    "multiple": true,
    "minHits": 2,
    "maxHits": 5
  },
  "43": {
    "name": "Leer",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "multiple",
    "multiple": false
  },
  "44": {
    "name": "Bite",
    "type": "Dark",
    "power": 60,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "45": {
    "name": "Growl",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "multiple",
    "multiple": false
  },
  "46": {
    "name": "Roar",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "47": {
    "name": "Sing",
    "type": "Normal",
    "power": " ",
    "accuracy": 55,
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "48": {
    "name": "Supersonic",
    "type": "Normal",
    "power": " ",
    "accuracy": 55,
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "49": {
    "name": "Sonic Boom",
    "type": "Normal",
    "power": " ",
    "accuracy": 90,
    "category": "Physical",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "50": {
    "name": "Disable",
    "type": "Normal",
    "power": " ",
    "accuracy": 55,
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "51": {
    "name": "Acid",
    "type": "Poison",
    "power": 40,
    "accuracy": 100,
    "category": "Physical",
    "target": "multiple",
    "contact": false,
    "multiple": false
  },
  "52": {
    "name": "Ember",
    "type": "Fire",
    "power": 40,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "53": {
    "name": "Flamethrower",
    "type": "Fire",
    "power": 95,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "54": {
    "name": "Mist",
    "type": "Ice",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "55": {
    "name": "Water Gun",
    "type": "Water",
    "power": 40,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "56": {
    "name": "Hydro Pump",
    "type": "Water",
    "power": 120,
    "accuracy": 80,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "57": {
    "name": "Surf",
    "type": "Water",
    "power": 95,
    "accuracy": 100,
    "category": "Special",
    "target": "multiple",
    "contact": false,
    "multiple": false
  },
  "58": {
    "name": "Ice Beam",
    "type": "Ice",
    "power": 95,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "59": {
    "name": "Blizzard",
    "type": "Ice",
    "power": 120,
    "accuracy": 70,
    "category": "Special",
    "target": "multiple",
    "contact": false,
    "multiple": false
  },
  "60": {
    "name": "Psybeam",
    "type": "Psychic",
    "power": 65,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "61": {
    "name": "Bubble Beam",
    "type": "Water",
    "power": 65,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "62": {
    "name": "Aurora Beam",
    "type": "Ice",
    "power": 65,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "63": {
    "name": "Hyper Beam",
    "type": "Normal",
    "power": 150,
    "accuracy": 90,
    "category": "Physical",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "64": {
    "name": "Peck",
    "type": "Flying",
    "power": 35,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "65": {
    "name": "Drill Peck",
    "type": "Flying",
    "power": 80,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "66": {
    "name": "Submission",
    "type": "Fighting",
    "power": 80,
    "accuracy": 80,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "67": {
    "name": "Low Kick",
    "type": "Fighting",
    "power": " ",
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "68": {
    "name": "Counter",
    "type": "Fighting",
    "power": " ",
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "69": {
    "name": "Seismic Toss",
    "type": "Fighting",
    "power": " ",
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "70": {
    "name": "Strength",
    "type": "Normal",
    "power": 80,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "71": {
    "name": "Absorb",
    "type": "Grass",
    "power": 20,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "72": {
    "name": "Mega Drain",
    "type": "Grass",
    "power": 75,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "73": {
    "name": "Leech Seed",
    "type": "Grass",
    "power": " ",
    "accuracy": 90,
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "74": {
    "name": "Growth",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "75": {
    "name": "Razor Leaf",
    "type": "Grass",
    "power": 55,
    "accuracy": 95,
    "category": "Special",
    "target": "multiple",
    "contact": false,
    "multiple": false
  },
  "76": {
    "name": "Solar Beam",
    "type": "Grass",
    "power": 200,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "77": {
    "name": "Poison Powder",
    "type": "Poison",
    "power": " ",
    "accuracy": 75,
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "78": {
    "name": "Stun Spore",
    "type": "Grass",
    "power": " ",
    "accuracy": 75,
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "79": {
    "name": "Sleep Powder",
    "type": "Grass",
    "power": " ",
    "accuracy": 75,
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "80": {
    "name": "Petal Dance",
    "type": "Grass",
    "power": 90,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "81": {
    "name": "String Shot",
    "type": "Bug",
    "power": " ",
    "accuracy": 95,
    "category": "Status",
    "target": "multiple",
    "multiple": false
  },
  "82": {
    "name": "Dragon Rage",
    "type": "Dragon",
    "power": " ",
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "83": {
    "name": "Fire Spin",
    "type": "Fire",
    "power": 15,
    "accuracy": 70,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "84": {
    "name": "Thunder Shock",
    "type": "Electric",
    "power": 40,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "85": {
    "name": "Thunderbolt",
    "type": "Electric",
    "power": 95,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "86": {
    "name": "Thunder Wave",
    "type": "Electric",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "87": {
    "name": "Thunder",
    "type": "Electric",
    "power": 120,
    "accuracy": 70,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "88": {
    "name": "Rock Throw",
    "type": "Rock",
    "power": 50,
    "accuracy": 90,
    "category": "Physical",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "89": {
    "name": "Earthquake",
    "type": "Ground",
    "power": 100,
    "accuracy": 100,
    "category": "Physical",
    "target": "multiple",
    "contact": false,
    "multiple": false
  },
  "90": {
    "name": "Fissure",
    "type": "Ground",
    "power": " ",
    "accuracy": 30,
    "category": "Physical",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "91": {
    "name": "Dig",
    "type": "Ground",
    "power": 60,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "92": {
    "name": "Toxic",
    "type": "Poison",
    "power": " ",
    "accuracy": 85,
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "93": {
    "name": "Confusion",
    "type": "Psychic",
    "power": 50,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "94": {
    "name": "Psychic",
    "type": "Psychic",
    "power": 90,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "95": {
    "name": "Hypnosis",
    "type": "Psychic",
    "power": " ",
    "accuracy": 60,
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "96": {
    "name": "Meditate",
    "type": "Psychic",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "97": {
    "name": "Agility",
    "type": "Psychic",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "98": {
    "name": "Quick Attack",
    "type": "Normal",
    "power": 40,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "99": {
    "name": "Rage",
    "type": "Normal",
    "power": 20,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": false
  },
  "100": {
    "name": "Teleport",
    "type": "Psychic",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "101": {
    "name": "Night Shade",
    "type": "Ghost",
    "power": " ",
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false,
    "multiple": false
  },
  "102": {
    "name": "Mimic",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "103": {
    "name": "Screech",
    "type": "Normal",
    "power": " ",
    "accuracy": 85,
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "104": {
    "name": "Double Team",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "105": {
    "name": "Recover",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "106": {
    "name": "Harden",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "107": {
    "name": "Minimize",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "108": {
    "name": "Smokescreen",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "109": {
    "name": "Confuse Ray",
    "type": "Ghost",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "110": {
    "name": "Withdraw",
    "type": "Water",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "111": {
    "name": "Defense Curl",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "112": {
    "name": "Barrier",
    "type": "Psychic",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "113": {
    "name": "Light Screen",
    "type": "Psychic",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single",
    "multiple": false
  },
  "114": {
    "name": "Haze",
    "type": "Ice",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "115": {
    "name": "Reflect",
    "type": "Psychic",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "116": {
    "name": "Focus Energy",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "117": {
    "name": "Bide",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "118": {
    "name": "Metronome",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "119": {
    "name": "Mirror Move",
    "type": "Flying",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "120": {
    "name": "Self Destruct",
    "type": "Normal",
    "power": 200,
    "accuracy": 100,
    "category": "Physical",
    "target": "multiple",
    "contact": false
  },
  "121": {
    "name": "Egg Bomb",
    "type": "Normal",
    "power": 100,
    "accuracy": 75,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "122": {
    "name": "Lick",
    "type": "Ghost",
    "power": 20,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "123": {
    "name": "Smog",
    "type": "Poison",
    "power": 20,
    "accuracy": 70,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "124": {
    "name": "Sludge",
    "type": "Poison",
    "power": 65,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "125": {
    "name": "Bone Club",
    "type": "Ground",
    "power": 65,
    "accuracy": 85,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "126": {
    "name": "Fire Blast",
    "type": "Fire",
    "power": 120,
    "accuracy": 85,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "127": {
    "name": "Waterfall",
    "type": "Water",
    "power": 80,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "128": {
    "name": "Clamp",
    "type": "Water",
    "power": 35,
    "accuracy": 75,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "129": {
    "name": "Swift",
    "type": "Normal",
    "power": 60,
    "accuracy": " ",
    "category": "Physical",
    "target": "multiple",
    "contact": false
  },
  "130": {
    "name": "Skull Bash",
    "type": "Normal",
    "power": 100,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "131": {
    "name": "Spike Cannon",
    "type": "Normal",
    "power": 20,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false,
    "multiple": true,
    "minHits": 2,
    "maxHits": 5
  },
  "132": {
    "name": "Constrict",
    "type": "Normal",
    "power": 10,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "133": {
    "name": "Amnesia",
    "type": "Psychic",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "134": {
    "name": "Kinesis",
    "type": "Psychic",
    "power": " ",
    "accuracy": 80,
    "category": "Status",
    "target": "single"
  },
  "135": {
    "name": "Soft Boiled",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "136": {
    "name": "High Jump Kick",
    "type": "Fighting",
    "power": 85,
    "accuracy": 90,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "137": {
    "name": "Glare",
    "type": "Normal",
    "power": " ",
    "accuracy": 75,
    "category": "Status",
    "target": "single"
  },
  "138": {
    "name": "Dream Eater",
    "type": "Psychic",
    "power": 100,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "139": {
    "name": "Poison Gas",
    "type": "Poison",
    "power": " ",
    "accuracy": 55,
    "category": "Status",
    "target": "multiple"
  },
  "140": {
    "name": "Barrage",
    "type": "Normal",
    "power": 15,
    "accuracy": 85,
    "category": "Physical",
    "target": "single",
    "contact": false,
    "multiple": true,
    "minHits": 2,
    "maxHits": 5
  },
  "141": {
    "name": "Leech Life",
    "type": "Bug",
    "power": 20,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "142": {
    "name": "Lovely Kiss",
    "type": "Normal",
    "power": " ",
    "accuracy": 75,
    "category": "Status",
    "target": "single"
  },
  "143": {
    "name": "Sky Attack",
    "type": "Flying",
    "power": 200,
    "accuracy": 90,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "144": {
    "name": "Transform",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "145": {
    "name": "Bubble",
    "type": "Water",
    "power": 20,
    "accuracy": 100,
    "category": "Special",
    "target": "multiple",
    "contact": false
  },
  "146": {
    "name": "Dizzy Punch",
    "type": "Normal",
    "power": 70,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "147": {
    "name": "Spore",
    "type": "Grass",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single"
  },
  "148": {
    "name": "Flash",
    "type": "Normal",
    "power": " ",
    "accuracy": 70,
    "category": "Status",
    "target": "single"
  },
  "149": {
    "name": "Psywave",
    "type": "Psychic",
    "power": " ",
    "accuracy": 80,
    "category": "Special",
    "target": "single",
    "contact": false,
    "varies": true,
  },
  "150": {
    "name": "Splash",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "151": {
    "name": "Acid Armor",
    "type": "Poison",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "152": {
    "name": "Crabhammer",
    "type": "Water",
    "power": 90,
    "accuracy": 85,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "153": {
    "name": "Explosion",
    "type": "Normal",
    "power": 250,
    "accuracy": 100,
    "category": "Physical",
    "target": "multiple",
    "contact": false
  },
  "154": {
    "name": "Fury Swipes",
    "type": "Normal",
    "power": 18,
    "accuracy": 80,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": true,
    "minHits": 2,
    "maxHits": 5
  },
  "155": {
    "name": "Bonemerang",
    "type": "Ground",
    "power": 50,
    "accuracy": 90,
    "category": "Physical",
    "target": "single",
    "contact": false,
    "multiple": true,
    "minHits": 2,
    "maxHits": 2
  },
  "156": {
    "name": "Rest",
    "type": "Psychic",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "157": {
    "name": "Rock Slide",
    "type": "Rock",
    "power": 75,
    "accuracy": 90,
    "category": "Physical",
    "target": "multiple",
    "contact": false
  },
  "158": {
    "name": "Hyper Fang",
    "type": "Normal",
    "power": 80,
    "accuracy": 90,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "159": {
    "name": "Sharpen",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "160": {
    "name": "Conversion",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "161": {
    "name": "Tri Attack",
    "type": "Normal",
    "power": 80,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "162": {
    "name": "Super Fang",
    "type": "Normal",
    "power": " ",
    "accuracy": 90,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "163": {
    "name": "Slash",
    "type": "Normal",
    "power": 70,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "164": {
    "name": "Substitute",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "165": {
    "name": "Struggle",
    "type": "Normal",
    "power": 50,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "166": {
    "name": "Sketch",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "167": {
    "name": "Triple Kick",
    "type": "Fighting",
    "power": 10,
    "accuracy": 90,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": true,
    "minHits": 1,
    "maxHits": 3
  },
  "168": {
    "name": "Thief",
    "type": "Dark",
    "power": 40,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "169": {
    "name": "Spider Web",
    "type": "Bug",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "170": {
    "name": "Mind Reader",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single"
  },
  "171": {
    "name": "Nightmare",
    "type": "Ghost",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "172": {
    "name": "Flame Wheel",
    "type": "Fire",
    "power": 60,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "173": {
    "name": "Snore",
    "type": "Normal",
    "power": 40,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "174": {
    "name": "Curse",
    "type": "Ghost",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "175": {
    "name": "Flail",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "176": {
    "name": "Conversion 2",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "177": {
    "name": "Aeroblast",
    "type": "Flying",
    "power": 100,
    "accuracy": 95,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "178": {
    "name": "Cotton Spore",
    "type": "Grass",
    "power": " ",
    "accuracy": 85,
    "category": "Status",
    "target": "multiple"
  },
  "179": {
    "name": "Reversal",
    "type": "Fighting",
    "power": " ",
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "180": {
    "name": "Spite",
    "type": "Ghost",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single"
  },
  "181": {
    "name": "Powder Snow",
    "type": "Ice",
    "power": 40,
    "accuracy": 100,
    "category": "Special",
    "target": "multiple",
    "contact": false
  },
  "182": {
    "name": "Protect",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "183": {
    "name": "Mach Punch",
    "type": "Fighting",
    "power": 40,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "184": {
    "name": "Scary Face",
    "type": "Normal",
    "power": " ",
    "accuracy": 90,
    "category": "Status",
    "target": "single"
  },
  "185": {
    "name": "Feint Attack",
    "type": "Dark",
    "power": 60,
    "accuracy": " ",
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "186": {
    "name": "Sweet Kiss",
    "type": "Fairy",
    "power": " ",
    "accuracy": 75,
    "category": "Status",
    "target": "single"
  },
  "187": {
    "name": "Belly Drum",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "188": {
    "name": "Sludge Bomb",
    "type": "Poison",
    "power": 90,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "189": {
    "name": "Mud Slap",
    "type": "Ground",
    "power": 20,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "190": {
    "name": "Octazooka",
    "type": "Water",
    "power": 65,
    "accuracy": 85,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "191": {
    "name": "Spikes",
    "type": "Ground",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "192": {
    "name": "Zap Cannon",
    "type": "Electric",
    "power": 100,
    "accuracy": 50,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "193": {
    "name": "Foresight",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single"
  },
  "194": {
    "name": "Destiny Bond",
    "type": "Ghost",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "195": {
    "name": "Perish Song",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "multiple"
  },
  "196": {
    "name": "Icy Wind",
    "type": "Ice",
    "power": 55,
    "accuracy": 95,
    "category": "Special",
    "target": "multiple",
    "contact": false
  },
  "197": {
    "name": "Detect",
    "type": "Fighting",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "198": {
    "name": "Bone Rush",
    "type": "Ground",
    "power": 25,
    "accuracy": 80,
    "category": "Physical",
    "target": "single",
    "contact": false,
    "multiple": true,
    "minHits": 2,
    "maxHits": 5
  },
  "199": {
    "name": "Lock On",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single"
  },
  "200": {
    "name": "Outrage",
    "type": "Dragon",
    "power": 90,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "201": {
    "name": "Sandstorm",
    "type": "Rock",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "202": {
    "name": "Giga Drain",
    "type": "Grass",
    "power": 60,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "203": {
    "name": "Endure",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "204": {
    "name": "Charm",
    "type": "Fairy",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single"
  },
  "205": {
    "name": "Rollout",
    "type": "Rock",
    "power": 30,
    "accuracy": 90,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": true,
    "minHits": 1,
    "maxHits": 5
  },
  "206": {
    "name": "False Swipe",
    "type": "Normal",
    "power": 40,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "207": {
    "name": "Swagger",
    "type": "Normal",
    "power": " ",
    "accuracy": 80,
    "category": "Status",
    "target": "single"
  },
  "208": {
    "name": "Milk Drink",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "209": {
    "name": "Spark",
    "type": "Electric",
    "power": 65,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "210": {
    "name": "Fury Cutter",
    "type": "Bug",
    "power": 10,
    "accuracy": 95,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": true,
    "minHits": 1,
    "maxHits": 5
  },
  "211": {
    "name": "Steel Wing",
    "type": "Steel",
    "power": 70,
    "accuracy": 90,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "212": {
    "name": "Mean Look",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "213": {
    "name": "Attract",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single"
  },
  "214": {
    "name": "Sleep Talk",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "215": {
    "name": "Heal Bell",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "216": {
    "name": "Return",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "217": {
    "name": "Present",
    "type": "Normal",
    "power": " ",
    "accuracy": 90,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "218": {
    "name": "Frustration",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "219": {
    "name": "Safeguard",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "220": {
    "name": "Pain Split",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "221": {
    "name": "Sacred Fire",
    "type": "Fire",
    "power": 100,
    "accuracy": 95,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "222": {
    "name": "Magnitude",
    "type": "Ground",
    "power": " ",
    "accuracy": 100,
    "category": "Physical",
    "target": "multiple",
    "contact": false
  },
  "223": {
    "name": "Dynamic Punch",
    "type": "Fighting",
    "power": 100,
    "accuracy": 50,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "224": {
    "name": "Megahorn",
    "type": "Bug",
    "power": 120,
    "accuracy": 85,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "225": {
    "name": "Dragon Breath",
    "type": "Dragon",
    "power": 60,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "226": {
    "name": "Baton Pass",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "227": {
    "name": "Encore",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single"
  },
  "228": {
    "name": "Pursuit",
    "type": "Dark",
    "power": 40,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "229": {
    "name": "Rapid Spin",
    "type": "Normal",
    "power": 20,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "230": {
    "name": "Sweet Scent",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "multiple"
  },
  "231": {
    "name": "Iron Tail",
    "type": "Steel",
    "power": 100,
    "accuracy": 75,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "232": {
    "name": "Metal Claw",
    "type": "Steel",
    "power": 50,
    "accuracy": 95,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "233": {
    "name": "Vital Throw",
    "type": "Fighting",
    "power": 70,
    "accuracy": " ",
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "234": {
    "name": "Morning Sun",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "235": {
    "name": "Synthesis",
    "type": "Grass",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "236": {
    "name": "Moonlight",
    "type": "Fairy",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "237": {
    "name": "Hidden Power",
    "type": "Normal",
    "power": 30,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "238": {
    "name": "Cross Chop",
    "type": "Fighting",
    "power": 100,
    "accuracy": 80,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "239": {
    "name": "Twister",
    "type": "Dragon",
    "power": 40,
    "accuracy": 100,
    "category": "Special",
    "target": "multiple",
    "contact": false
  },
  "240": {
    "name": "Rain Dance",
    "type": "Water",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "241": {
    "name": "Sunny Day",
    "type": "Fire",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "242": {
    "name": "Crunch",
    "type": "Dark",
    "power": 80,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "243": {
    "name": "Mirror Coat",
    "type": "Psychic",
    "power": " ",
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "244": {
    "name": "Psych Up",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "245": {
    "name": "Extreme Speed",
    "type": "Normal",
    "power": 80,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "246": {
    "name": "Ancient Power",
    "type": "Rock",
    "power": 60,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "247": {
    "name": "Shadow Ball",
    "type": "Ghost",
    "power": 80,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "248": {
    "name": "Future Sight",
    "type": "Psychic",
    "power": 80,
    "accuracy": 90,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "249": {
    "name": "Rock Smash",
    "type": "Fighting",
    "power": 20,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "250": {
    "name": "Whirlpool",
    "type": "Water",
    "power": 15,
    "accuracy": 70,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "251": {
    "name": "Beat Up",
    "type": "Dark",
    "power": 10,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "252": {
    "name": "Fake Out",
    "type": "Normal",
    "power": 40,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "253": {
    "name": "Uproar",
    "type": "Normal",
    "power": 50,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "254": {
    "name": "Stockpile",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "255": {
    "name": "Spit Up",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "256": {
    "name": "Swallow",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "257": {
    "name": "Heat Wave",
    "type": "Fire",
    "power": 100,
    "accuracy": 90,
    "category": "Special",
    "target": "multiple",
    "contact": false
  },
  "258": {
    "name": "Hail",
    "type": "Ice",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "259": {
    "name": "Torment",
    "type": "Dark",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single"
  },
  "260": {
    "name": "Flatter",
    "type": "Dark",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single"
  },
  "261": {
    "name": "Will O Wisp",
    "type": "Fire",
    "power": " ",
    "accuracy": 75,
    "category": "Status",
    "target": "single"
  },
  "262": {
    "name": "Memento",
    "type": "Dark",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "263": {
    "name": "Facade",
    "type": "Normal",
    "power": 70,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "264": {
    "name": "Focus Punch",
    "type": "Fighting",
    "power": 150,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "265": {
    "name": "Smelling Salts",
    "type": "Normal",
    "power": 60,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "266": {
    "name": "Follow Me",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "267": {
    "name": "Nature Power",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "268": {
    "name": "Charge",
    "type": "Electric",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "269": {
    "name": "Taunt",
    "type": "Dark",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single"
  },
  "270": {
    "name": "Helping Hand",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "271": {
    "name": "Trick",
    "type": "Psychic",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single"
  },
  "272": {
    "name": "Role Play",
    "type": "Psychic",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "273": {
    "name": "Wish",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "274": {
    "name": "Assist",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "275": {
    "name": "Ingrain",
    "type": "Grass",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "276": {
    "name": "Superpower",
    "type": "Fighting",
    "power": 120,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "277": {
    "name": "Magic Coat",
    "type": "Psychic",
    "power": " ",
    "accuracy": " ",
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "278": {
    "name": "Recycle",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "279": {
    "name": "Revenge",
    "type": "Fighting",
    "power": 60,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "280": {
    "name": "Brick Break",
    "type": "Fighting",
    "power": 75,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "281": {
    "name": "Yawn",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "282": {
    "name": "Knock Off",
    "type": "Dark",
    "power": 20,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "283": {
    "name": "Endeavor",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "284": {
    "name": "Eruption",
    "type": "Fire",
    "power": 150,
    "accuracy": 100,
    "category": "Special",
    "target": "multiple",
    "contact": false
  },
  "285": {
    "name": "Skill Swap",
    "type": "Psychic",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "286": {
    "name": "Imprison",
    "type": "Psychic",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "287": {
    "name": "Refresh",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "288": {
    "name": "Grudge",
    "type": "Ghost",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "289": {
    "name": "Snatch",
    "type": "Dark",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "290": {
    "name": "Secret Power",
    "type": "Normal",
    "power": 70,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "291": {
    "name": "Dive",
    "type": "Water",
    "power": 60,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "292": {
    "name": "Arm Thrust",
    "type": "Fighting",
    "power": 15,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true,
    "multiple": true,
    "minHits": 2,
    "maxHits": 5
  },
  "293": {
    "name": "Camouflage",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "294": {
    "name": "Tail Glow",
    "type": "Bug",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "295": {
    "name": "Luster Purge",
    "type": "Psychic",
    "power": 70,
    "accuracy": 100,
    "category": "Special",
    "target": "single"
  },
  "296": {
    "name": "Mist Ball",
    "type": "Psychic",
    "power": 70,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "297": {
    "name": "Feather Dance",
    "type": "Flying",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single"
  },
  "298": {
    "name": "Teeter Dance",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "multiple"
  },
  "299": {
    "name": "Blaze Kick",
    "type": "Fire",
    "power": 85,
    "accuracy": 90,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "300": {
    "name": "Mud Sport",
    "type": "Ground",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "301": {
    "name": "Ice Ball",
    "type": "Ice",
    "power": 30,
    "accuracy": 90,
    "category": "Special",
    "target": "single",
    "contact": true,
    "multiple": true,
    "minHits": 1,
    "maxHits": 5
  },
  "302": {
    "name": "Needle Arm",
    "type": "Grass",
    "power": 60,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "303": {
    "name": "Slack Off",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Physical",
    "target": "single"
  },
  "304": {
    "name": "Hyper Voice",
    "type": "Normal",
    "power": 90,
    "accuracy": 100,
    "category": "Physical",
    "target": "multiple",
    "contact": false
  },
  "305": {
    "name": "Poison Fang",
    "type": "Poison",
    "power": 50,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "306": {
    "name": "Crush Claw",
    "type": "Normal",
    "power": 75,
    "accuracy": 95,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "307": {
    "name": "Blast Burn",
    "type": "Fire",
    "power": 150,
    "accuracy": 90,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "308": {
    "name": "Hydro Cannon",
    "type": "Water",
    "power": 150,
    "accuracy": 90,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "309": {
    "name": "Meteor Mash",
    "type": "Steel",
    "power": 100,
    "accuracy": 85,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "310": {
    "name": "Astonish",
    "type": "Ghost",
    "power": 30,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "311": {
    "name": "Weather Ball",
    "type": "Normal",
    "power": 50,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "312": {
    "name": "Aromatherapy",
    "type": "Grass",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "313": {
    "name": "Fake Tears",
    "type": "Dark",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single"
  },
  "314": {
    "name": "Air Cutter",
    "type": "Flying",
    "power": 55,
    "accuracy": 95,
    "category": "Physical",
    "target": "multiple",
    "contact": false
  },
  "315": {
    "name": "Overheat",
    "type": "Fire",
    "power": 140,
    "accuracy": 90,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "316": {
    "name": "Odor Sleuth",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "317": {
    "name": "Rock Tomb",
    "type": "Rock",
    "power": 50,
    "accuracy": 80,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "318": {
    "name": "Silver Wind",
    "type": "Bug",
    "power": 60,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "319": {
    "name": "Metal Sound",
    "type": "Steel",
    "power": " ",
    "accuracy": 85,
    "category": "Status",
    "target": "single"
  },
  "320": {
    "name": "Grass Whistle",
    "type": "Grass",
    "power": " ",
    "accuracy": 55,
    "category": "Status",
    "target": "single"
  },
  "321": {
    "name": "Tickle",
    "type": "Normal",
    "power": " ",
    "accuracy": 100,
    "category": "Status",
    "target": "single"
  },
  "322": {
    "name": "Cosmic Power",
    "type": "Psychic",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "323": {
    "name": "Water Spout",
    "type": "Water",
    "power": 150,
    "accuracy": 100,
    "category": "Special",
    "target": "multiple",
    "contact": false
  },
  "324": {
    "name": "Signal Beam",
    "type": "Bug",
    "power": 75,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "325": {
    "name": "Shadow Punch",
    "type": "Ghost",
    "power": 60,
    "accuracy": " ",
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "326": {
    "name": "Extrasensory",
    "type": "Psychic",
    "power": 80,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "327": {
    "name": "Sky Uppercut",
    "type": "Fighting",
    "power": 85,
    "accuracy": 90,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "328": {
    "name": "Sand Tomb",
    "type": "Ground",
    "power": 15,
    "accuracy": 70,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "329": {
    "name": "Sheer Cold",
    "type": "Ice",
    "power": " ",
    "accuracy": 30,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "330": {
    "name": "Muddy Water",
    "type": "Water",
    "power": 95,
    "accuracy": 85,
    "category": "Special",
    "target": "multiple",
    "contact": false
  },
  "331": {
    "name": "Bullet Seed",
    "type": "Grass",
    "power": 10,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": true,
    "minHits": 2,
    "maxHits": 5
  },
  "332": {
    "name": "Aerial Ace",
    "type": "Flying",
    "power": 60,
    "accuracy": " ",
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "333": {
    "name": "Icicle Spear",
    "type": "Ice",
    "power": 10,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false,
    "multiple": true,
    "minHits": 2,
    "maxHits": 5
  },
  "334": {
    "name": "Iron Defense",
    "type": "Steel",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "335": {
    "name": "Block",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "336": {
    "name": "Howl",
    "type": "Normal",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "337": {
    "name": "Dragon Claw",
    "type": "Dragon",
    "power": 80,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "338": {
    "name": "Frenzy Plant",
    "type": "Grass",
    "power": 150,
    "accuracy": 90,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "339": {
    "name": "Bulk Up",
    "type": "Fighting",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "340": {
    "name": "Bounce",
    "type": "Flying",
    "power": 85,
    "accuracy": 85,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "341": {
    "name": "Mud Shot",
    "type": "Ground",
    "power": 55,
    "accuracy": 95,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "342": {
    "name": "Poison Tail",
    "type": "Poison",
    "power": 50,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": true
  },
  "343": {
    "name": "Covet",
    "type": "Normal",
    "power": 40,
    "accuracy": 100,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "344": {
    "name": "Volt Tackle",
    "type": "Electric",
    "power": 120,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "345": {
    "name": "Magical Leaf",
    "type": "Grass",
    "power": 60,
    "accuracy": " ",
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "346": {
    "name": "Water Sport",
    "type": "Water",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "347": {
    "name": "Calm Mind",
    "type": "Psychic",
    "power": " ",
    "accuracy": " ",
    "category": "Special",
    "target": "single"
  },
  "348": {
    "name": "Leaf Blade",
    "type": "Grass",
    "power": 70,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": true
  },
  "349": {
    "name": "Dragon Dance",
    "type": "Dragon",
    "power": " ",
    "accuracy": " ",
    "category": "Status",
    "target": "single"
  },
  "350": {
    "name": "Rock Blast",
    "type": "Rock",
    "power": 25,
    "accuracy": 80,
    "category": "Physical",
    "target": "single",
    "contact": false,
    "multiple": true,
    "minHits": 2,
    "maxHits": 5
  },
  "351": {
    "name": "Shock Wave",
    "type": "Electric",
    "power": 60,
    "accuracy": " ",
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "352": {
    "name": "Water Pulse",
    "type": "Water",
    "power": 60,
    "accuracy": 100,
    "category": "Special",
    "target": "single",
    "contact": false
  },
  "353": {
    "name": "Doom Desire",
    "type": "Steel",
    "power": 120,
    "accuracy": 85,
    "category": "Physical",
    "target": "single",
    "contact": false
  },
  "354": {
    "name": "Psycho Boost",
    "type": "Psychic",
    "power": 140,
    "accuracy": 90,
    "category": "Special",
    "target": "single",
    "contact": false
  }
}

In [ ]:

MALE = 136
FEMALE = 120
TEAM = 128
standard = {
    "Bug Catcher": MALE,
    "Lass": FEMALE,
    "Camper": MALE,
    "Leader Brock": MALE,
    "Youngster": MALE,
    "Super Nerd": MALE,
    "Team Rocket Grunt": MALE,
    "Hiker": MALE,
    "Rival": MALE,
    "Picknicker": FEMALE,
    "Swimmer": {"Luis": MALE, "Richard": MALE, "Reece": MALE, "Tony": MALE, "David": MALE, "Douglas": MALE, "Matthew": MALE, "Axle": MALE, "Barry": MALE, "Dean": MALE, "Darrin": MALE, "Spencer": MALE, "Jack": MALE, "Jerome": MALE, "Roland": MALE, "Tiffany": FEMALE, "Nora": FEMALE, "Melissa": FEMALE, "Anya": FEMALE, "Alice": FEMALE, "Connie": FEMALE, "Shirley": FEMALE, "Amara": FEMALE, "Maria": FEMALE, "Abigail": FEMALE, "Finn": MALE, "Garrett": MALE, "Tisha": FEMALE, "Toby": MALE, "Mymo": MALE, "Nicole": FEMALE, "Samir": MALE, "Denise": FEMALE},
    "Leader Misty": FEMALE,
    "Gamer": MALE,
    "Engineer": MALE,
    "Fisherman": MALE,
    "Sailor": MALE,
    "Gentleman": MALE,
    "Leader Surge": MALE,
    "Pokemaniac":MALE,
    "Twins": TEAM,
    "Biker": MALE,
    "Beauty": FEMALE,
    "Cooltrainer": {"Samuel": MALE, "George": MALE, "Colby": MALE, "Paul": MALE, "Rolando": MALE, "Gilbert": MALE, "Owen": MALE, "Berke": MALE, "Yuji": MALE, "Warren": MALE, "Mary": FEMALE, "Caroline": FEMALE, "Alexa": FEMALE, "Shannon": FEMALE, "Naomi": FEMALE, "Austina": FEMALE, "Julie": FEMALE, "Leroy": MALE, "Michelle": FEMALE, },
    "Leader Erika": FEMALE,
    "Giovanni": MALE,
    "Channeler": FEMALE,
    "Young Couple": TEAM,
    "Cue Ball": MALE,
    "Bird Keeper": MALE,
    "Juggler": MALE,
    "Tamer": MALE,
    "Leader Koga": MALE,
    "Black Belt": MALE,
    "Scientist": MALE,
    "Psychic": {"Johan": MALE, "Tyron": MALE, "Cameron": MALE, "Preston": MALE, "Jaclyn": FEMALE, "Dario": MALE, "Rodette": FEMALE, "Laura": FEMALE},
    "Leader Sabrina": FEMALE,
    "Rocker": MALE,
    "Crush Kin": TEAM,
    "Sis and Bro": TEAM,
    "Burglar": MALE,
    "Leader Blaine": MALE,
    "Crush Girl": FEMALE,
    "Pokemon Ranger":{"Nicolas": MALE, "Madeline": FEMALE, "Logan": MALE, "Jackson": MALE, "Beth": FEMALE, "Katelyn": FEMALE},
    "Aroma Lady": FEMALE,
    "Tuber": {"Amira": FEMALE, "Alexis": FEMALE},
    "Leader Giovanni": MALE,
    "Cool Couple": TEAM,
    "Lorelei": FEMALE,
    "Bruno": MALE,
    "Agatha": FEMALE,
    "Lance": MALE,
    "Champ": MALE,
    "Pokemon Breeder": FEMALE,
    "Painter": FEMALE,
    "Lady": FEMALE,
    "Ruin Maniac": MALE,
    "Team Rocket Admin 1": FEMALE,
    "Team Rocket Admin 2": MALE
    }

In [ ]:
import pandas as pd
trainers_data = pd.read_csv('FRLG Trainers.csv')

In [ ]:
def get_trainer_name(trainer_title):
    if 'Rival' in trainer_title or 'Champ' in trainer_title:
        return 'TERRY'
    elif 'Grunt' in trainer_title:
        return 'GRUNT'
    elif 'Admin' in trainer_title:
        return 'ADMIN'
    elif 'Goon' in trainer_title:
        return 'GOON'
    elif 'Leader' in trainer_title:
        return trainer_title[7:].upper()
    elif 'Giovanni' in trainer_title:
        return 'GIOVANNI'
    elif 'Lorelei' in trainer_title:
        return 'LORELEI'
    elif 'Bruno' in trainer_title:
        return 'BRUNO'
    elif 'Agatha' in trainer_title:
        return 'AGATHA'
    elif 'Lance' in trainer_title:
        return 'LANCE'
    for i in standard.keys():
        if i in trainer_title:
            return trainer_title[len(i)+1:].upper()

In [ ]:
import json

natures = [
  'Hardy', 'Lonely', 'Brave', 'Adamant', 'Naughty',
  'Bold', 'Docile', 'Relaxed', 'Impish', 'Lax',
  'Timid', 'Hasty', 'Serious', 'Jolly', 'Naive',
  'Modest', 'Mild', 'Quiet', 'Bashful', 'Rash',
  'Calm', 'Gentle', 'Sassy', 'Careful', 'Quirky'
]

# Open and read the JSON file
with open('char_encode.json', 'r') as file:
    char_data = json.load(file)

def get_natures(trainer_title, name, mon, counter = 0):
    for i in standard.keys():
        if i in trainer_title:
            if isinstance(standard[i], int):
                additive = standard[i]
            else:
                additive = standard[i][name.title()]

    for char in name:
        counter += char_data[char]
    for char in mon:
        counter += char_data[char]

    counter*=256
    counter += additive
    nature = natures[counter%25]
    ability_flag = counter%2
    counter = ((counter-additive)//256)%25
    return counter, nature, ability_flag


In [ ]:
import ast

pokemon_info = pd.read_csv('pokemon_data_gen3.csv')
pokemon_info['abilities'] = pokemon_info['abilities'].apply(ast.literal_eval)


In [ ]:
def get_trainer_teams(trainers_data):
    teams = {}
    n_counter = 0
    prev_name = ''
    outliers = []
    for index, rows in trainers_data.iterrows():
        name = get_trainer_name(rows['Opponent'])
        if prev_name!='' and prev_name!=rows['Opponent']:
            n_counter = 0
        n_counter, nature, ability_flag = get_natures(rows['Opponent'], name, rows['Pokémon'].upper(), n_counter)
        prev_name = rows['Opponent']
        mon = rows['Pokémon']
        if mon == 'Nidoran F':
            mon = 'Nidoran♀'
        elif mon == 'Nidoran M':
            mon = 'Nidoran♂'
        elif mon == 'Farfetchd':
            mon = "Farfetch'd"
        all_abilities = list(pokemon_info[pokemon_info['name']==mon].iloc[:]['abilities'])[0]

        for col in ['Attack 1', 'Attack 2', 'Attack 3', 'Attack 4']:
            misspelt = False
            for i in move_data.keys():
                if move_data[i]['name'] == rows[col]:
                    rows[col] = i
                    misspelt = True
                    break
            if not misspelt and rows[col] not in outliers:
                outliers.append(rows[col])


        #print(name, rows['Pokémon'].upper(), nature)
        poke_info = dict(rows.drop(['Opponent', 'Route']))
        poke_info['IVS'] = json.loads(poke_info['IVS'])
        poke_info['EVS'] = json.loads(poke_info['EVS'])
        poke_info['Nature'] = nature
        poke_info['Ability'] = all_abilities[ability_flag]

        if rows['Opponent'] not in teams.keys():
            teams[rows['Opponent']] = [poke_info]

        elif rows['Opponent'] in teams.keys():
            teams[rows['Opponent']].append(poke_info)

    print(outliers)
    return teams
output = get_trainer_teams(trainers_data)

[nan]


In [ ]:
output

In [ ]:
json.dump(output, open('frlg_trainers.json', 'w'))

In [ ]:
null = 0
frlg_trainer_data = {
        "Lab Rival(Bulbasaur)": [
            {
                "name": "Charmander",
                "Level": 5,
                "Attack_1": "10",
                "Attack_2": "45",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Blaze"
            }
        ],
        "Lab Rival(Squirtle)": [
            {
                "name": "Bulbasaur",
                "Level": 5,
                "Attack_1": "33",
                "Attack_2": "45",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Overgrow"
            }
        ],
        "Lab Rival(Charmander)": [
            {
                "name": "Squirtle",
                "Level": 5,
                "Attack_1": "33",
                "Attack_2": "45",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Torrent"
            }
        ],
        "22 Rival(Bulbasaur)": [
            {
                "name": "Pidgey",
                "Level": 9,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Keen Eye"
            },
            {
                "name": "Charmander",
                "Level": 9,
                "Attack_1": "10",
                "Attack_2": "45",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Blaze"
            }
        ],
        "22 Rival (Squirtle)": [
            {
                "name": "Pidgey",
                "Level": 9,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Keen Eye"
            },
            {
                "name": "Bulbasaur",
                "Level": 9,
                "Attack_1": "33",
                "Attack_2": "45",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Overgrow"
            }
        ],
        "22 Rival (Charmander)": [
            {
                "name": "Pidgey",
                "Level": 9,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Keen Eye"
            },
            {
                "name": "Squirtle",
                "Level": 9,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Torrent"
            }
        ],
        "Bug Catcher Doug": [
            {
                "name": "Weedle",
                "Level": 7,
                "Attack_1": "40",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Shield Dust"
            },
            {
                "name": "Kakuna",
                "Level": 7,
                "Attack_1": "106",
                "Attack_2": null,
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Shed Skin"
            },
            {
                "name": "Weedle",
                "Level": 7,
                "Attack_1": "40",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Shield Dust"
            }
        ],
        "Bug Catcher Rick": [
            {
                "name": "Weedle",
                "Level": 6,
                "Attack_1": "40",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Shield Dust"
            },
            {
                "name": "Caterpie",
                "Level": 6,
                "Attack_1": "33",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Shield Dust"
            }
        ],
        "Bug Catcher Anthony": [
            {
                "name": "Caterpie",
                "Level": 7,
                "Attack_1": "33",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Shield Dust"
            },
            {
                "name": "Caterpie",
                "Level": 8,
                "Attack_1": "33",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Shield Dust"
            }
        ],
        "Bug Catcher Charlie": [
            {
                "name": "Metapod",
                "Level": 7,
                "Attack_1": "106",
                "Attack_2": null,
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Shed Skin"
            },
            {
                "name": "Caterpie",
                "Level": 7,
                "Attack_1": "33",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Shield Dust"
            },
            {
                "name": "Metapod",
                "Level": 7,
                "Attack_1": "106",
                "Attack_2": null,
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Shed Skin"
            }
        ],
        "Bug Catcher Sammy": [
            {
                "name": "Weedle",
                "Level": 9,
                "Attack_1": "40",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Shield Dust"
            }
        ],
        "Camper Liam": [
            {
                "name": "Geodude",
                "Level": 10,
                "Attack_1": "33",
                "Attack_2": "111",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Rock Head"
            },
            {
                "name": "Sandshrew",
                "Level": 11,
                "Attack_1": "10",
                "Attack_2": "111",
                "Attack_3": "28",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Sand Veil"
            }
        ],
        "Leader Brock": [
            {
                "name": "Geodude",
                "Level": 12,
                "Attack_1": "33",
                "Attack_2": "111",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Rock Head"
            },
            {
                "name": "Onix",
                "Level": 14,
                "Attack_1": "33",
                "Attack_2": "20",
                "Attack_3": "317",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Rock Head"
            }
        ],
        "Lass Janice": [
            {
                "name": "Pidgey",
                "Level": 9,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": "16",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Keen Eye"
            },
            {
                "name": "Pidgey",
                "Level": 9,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": "16",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Keen Eye"
            }
        ],
        "Bug Catcher Colton": [
            {
                "name": "Caterpie",
                "Level": 10,
                "Attack_1": "33",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Shield Dust"
            },
            {
                "name": "Weedle",
                "Level": 10,
                "Attack_1": "40",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Shield Dust"
            },
            {
                "name": "Caterpie",
                "Level": 10,
                "Attack_1": "33",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Shield Dust"
            }
        ],
        "Youngster Ben": [
            {
                "name": "Rattata",
                "Level": 11,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Run Away"
            },
            {
                "name": "Ekans",
                "Level": 11,
                "Attack_1": "35",
                "Attack_2": "43",
                "Attack_3": "40",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Intimidate"
            }
        ],
        "Bug Catcher Greg": [
            {
                "name": "Weedle",
                "Level": 9,
                "Attack_1": "40",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Shield Dust"
            },
            {
                "name": "Kakuna",
                "Level": 9,
                "Attack_1": "106",
                "Attack_2": null,
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Shed Skin"
            },
            {
                "name": "Caterpie",
                "Level": 9,
                "Attack_1": "33",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Shield Dust"
            },
            {
                "name": "Metapod",
                "Level": 9,
                "Attack_1": "106",
                "Attack_2": null,
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Shed Skin"
            }
        ],
        "Lass Sally": [
            {
                "name": "Rattata",
                "Level": 10,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Run Away"
            },
            {
                "name": "Nidoran\u2640",
                "Level": 10,
                "Attack_1": "45",
                "Attack_2": "10",
                "Attack_3": "39",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Poison Point"
            }
        ],
        "Bug Catcher James": [
            {
                "name": "Caterpie",
                "Level": 11,
                "Attack_1": "33",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Shield Dust"
            },
            {
                "name": "Metapod",
                "Level": 11,
                "Attack_1": "106",
                "Attack_2": null,
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Shed Skin"
            }
        ],
        "Youngster Calvin": [
            {
                "name": "Spearow",
                "Level": 14,
                "Attack_1": "64",
                "Attack_2": "45",
                "Attack_3": "43",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Keen Eye"
            }
        ],
        "Lass Robin": [
            {
                "name": "Jigglypuff",
                "Level": 14,
                "Attack_1": "47",
                "Attack_2": "111",
                "Attack_3": "1",
                "Attack_4": "50",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Cute Charm"
            }
        ],
        "Lass Iris": [
            {
                "name": "Clefairy",
                "Level": 14,
                "Attack_1": "45",
                "Attack_2": "227",
                "Attack_3": "47",
                "Attack_4": "3",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Cute Charm"
            }
        ],
        "Bug Catcher Kent": [
            {
                "name": "Weedle",
                "Level": 11,
                "Attack_1": "40",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Shield Dust"
            },
            {
                "name": "Kakuna",
                "Level": 11,
                "Attack_1": "106",
                "Attack_2": null,
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Shed Skin"
            }
        ],
        "Super Nerd Jovan": [
            {
                "name": "Magnemite",
                "Level": 11,
                "Attack_1": "319",
                "Attack_2": "33",
                "Attack_3": "84",
                "Attack_4": "48",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Magnet Pull"
            },
            {
                "name": "Voltorb",
                "Level": 11,
                "Attack_1": "268",
                "Attack_2": "33",
                "Attack_3": "103",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Soundproof"
            }
        ],
        "Bug Catcher Robby": [
            {
                "name": "Caterpie",
                "Level": 10,
                "Attack_1": "33",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Shield Dust"
            },
            {
                "name": "Metapod",
                "Level": 10,
                "Attack_1": "106",
                "Attack_2": null,
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Shed Skin"
            },
            {
                "name": "Caterpie",
                "Level": 10,
                "Attack_1": "33",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Shield Dust"
            }
        ],
        "Lass Miriam": [
            {
                "name": "Oddish",
                "Level": 11,
                "Attack_1": "71",
                "Attack_2": "230",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Bellsprout",
                "Level": 11,
                "Attack_1": "22",
                "Attack_2": "74",
                "Attack_3": "35",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Chlorophyll"
            }
        ],
        "Team Rocket Grunt 1": [
            {
                "name": "Sandshrew",
                "Level": 11,
                "Attack_1": "10",
                "Attack_2": "111",
                "Attack_3": "28",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Sand Veil"
            },
            {
                "name": "Rattata",
                "Level": 11,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Run Away"
            },
            {
                "name": "Zubat",
                "Level": 11,
                "Attack_1": "141",
                "Attack_2": "310",
                "Attack_3": "48",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Inner Focus"
            }
        ],
        "Team Rocket Grunt 2": [
            {
                "name": "Zubat",
                "Level": 11,
                "Attack_1": "141",
                "Attack_2": "310",
                "Attack_3": "48",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Inner Focus"
            },
            {
                "name": "Ekans",
                "Level": 11,
                "Attack_1": "35",
                "Attack_2": "43",
                "Attack_3": "40",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Intimidate"
            }
        ],
        "Youngster Josh": [
            {
                "name": "Rattata",
                "Level": 10,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Run Away"
            },
            {
                "name": "Rattata",
                "Level": 10,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Run Away"
            },
            {
                "name": "Zubat",
                "Level": 10,
                "Attack_1": "141",
                "Attack_2": "310",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Inner Focus"
            }
        ],
        "Hiker Marcos": [
            {
                "name": "Geodude",
                "Level": 10,
                "Attack_1": "33",
                "Attack_2": "111",
                "Attack_3": "300",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Rock Head"
            },
            {
                "name": "Geodude",
                "Level": 10,
                "Attack_1": "33",
                "Attack_2": "111",
                "Attack_3": "300",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Rock Head"
            },
            {
                "name": "Onix",
                "Level": 10,
                "Attack_1": "33",
                "Attack_2": "103",
                "Attack_3": "20",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Rock Head"
            }
        ],
        "Team Rocket Grunt 3": [
            {
                "name": "Rattata",
                "Level": 13,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Run Away"
            },
            {
                "name": "Sandshrew",
                "Level": 13,
                "Attack_1": "10",
                "Attack_2": "111",
                "Attack_3": "28",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Sand Veil"
            }
        ],
        "Team Rocket Grunt 4": [
            {
                "name": "Rattata",
                "Level": 13,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Run Away"
            },
            {
                "name": "Zubat",
                "Level": 13,
                "Attack_1": "141",
                "Attack_2": "310",
                "Attack_3": "48",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Inner Focus"
            }
        ],
        "Super Nerd Miguel": [
            {
                "name": "Grimer",
                "Level": 12,
                "Attack_1": "139",
                "Attack_2": "1",
                "Attack_3": "106",
                "Attack_4": "50",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Stench"
            },
            {
                "name": "Voltorb",
                "Level": 12,
                "Attack_1": "268",
                "Attack_2": "33",
                "Attack_3": "103",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Soundproof"
            },
            {
                "name": "Koffing",
                "Level": 12,
                "Attack_1": "139",
                "Attack_2": "33",
                "Attack_3": "123",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Levitate"
            }
        ],
        "24 Rival(Bulbasaur)": [
            {
                "name": "Pidgeotto",
                "Level": 17,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": "16",
                "Attack_4": "98",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Keen Eye"
            },
            {
                "name": "Abra",
                "Level": 16,
                "Attack_1": "100",
                "Attack_2": null,
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Synchronize"
            },
            {
                "name": "Rattata",
                "Level": 15,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": null,
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Run Away"
            },
            {
                "name": "Charmander",
                "Level": 18,
                "Attack_1": "232",
                "Attack_2": "52",
                "Attack_3": "45",
                "Attack_4": "10",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Blaze"
            }
        ],
        "24 Rival(Squirtle)": [
            {
                "name": "Pidgeotto",
                "Level": 17,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": "16",
                "Attack_4": "98",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Keen Eye"
            },
            {
                "name": "Abra",
                "Level": 16,
                "Attack_1": "100",
                "Attack_2": null,
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Synchronize"
            },
            {
                "name": "Rattata",
                "Level": 15,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": null,
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Run Away"
            },
            {
                "name": "Bulbasaur",
                "Level": 18,
                "Attack_1": "79",
                "Attack_2": "77",
                "Attack_3": "22",
                "Attack_4": "73",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Overgrow"
            }
        ],
        "24 Rival(Charmander)": [
            {
                "name": "Pidgeotto",
                "Level": 17,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": "16",
                "Attack_4": "98",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Keen Eye"
            },
            {
                "name": "Abra",
                "Level": 16,
                "Attack_1": "100",
                "Attack_2": null,
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Synchronize"
            },
            {
                "name": "Rattata",
                "Level": 15,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": null,
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Run Away"
            },
            {
                "name": "Squirtle",
                "Level": 18,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "110",
                "Attack_4": "55",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Torrent"
            }
        ],
        "Bug Catcher Cale": [
            {
                "name": "Caterpie",
                "Level": 10,
                "Attack_1": "33",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Shield Dust"
            },
            {
                "name": "Weedle",
                "Level": 10,
                "Attack_1": "40",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Shield Dust"
            },
            {
                "name": "Metapod",
                "Level": 10,
                "Attack_1": "106",
                "Attack_2": null,
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Shed Skin"
            },
            {
                "name": "Kakuna",
                "Level": 10,
                "Attack_1": "106",
                "Attack_2": null,
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Shed Skin"
            }
        ],
        "Lass Ali": [
            {
                "name": "Pidgey",
                "Level": 12,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": "16",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Keen Eye"
            },
            {
                "name": "Oddish",
                "Level": 12,
                "Attack_1": "71",
                "Attack_2": "230",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Bellsprout",
                "Level": 12,
                "Attack_1": "22",
                "Attack_2": "74",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Chlorophyll"
            }
        ],
        "Youngster Timmy": [
            {
                "name": "Sandshrew",
                "Level": 14,
                "Attack_1": "10",
                "Attack_2": "111",
                "Attack_3": "28",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Sand Veil"
            },
            {
                "name": "Ekans",
                "Level": 14,
                "Attack_1": "35",
                "Attack_2": "43",
                "Attack_3": "40",
                "Attack_4": "44",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Intimidate"
            }
        ],
        "Lass Reli": [
            {
                "name": "Nidoran\u2642",
                "Level": 16,
                "Attack_1": "43",
                "Attack_2": "64",
                "Attack_3": "116",
                "Attack_4": "24",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Poison Point"
            },
            {
                "name": "Nidoran\u2640",
                "Level": 16,
                "Attack_1": "45",
                "Attack_2": "10",
                "Attack_3": "39",
                "Attack_4": "24",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Poison Point"
            }
        ],
        "Camper Ethan": [
            {
                "name": "Mankey",
                "Level": 18,
                "Attack_1": "43",
                "Attack_2": "67",
                "Attack_3": "2",
                "Attack_4": "154",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Vital Spirit"
            }
        ],
        "Team Rocket Grunt 5": [
            {
                "name": "Ekans",
                "Level": 15,
                "Attack_1": "35",
                "Attack_2": "43",
                "Attack_3": "40",
                "Attack_4": "44",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Intimidate"
            },
            {
                "name": "Zubat",
                "Level": 15,
                "Attack_1": "141",
                "Attack_2": "310",
                "Attack_3": "48",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Inner Focus"
            }
        ],
        "Camper Shane": [
            {
                "name": "Rattata",
                "Level": 14,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Run Away"
            },
            {
                "name": "Ekans",
                "Level": 14,
                "Attack_1": "35",
                "Attack_2": "43",
                "Attack_3": "40",
                "Attack_4": "44",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Intimidate"
            }
        ],
        "Hiker Franklin": [
            {
                "name": "Machop",
                "Level": 15,
                "Attack_1": "67",
                "Attack_2": "43",
                "Attack_3": "116",
                "Attack_4": "2",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Guts"
            },
            {
                "name": "Geodude",
                "Level": 15,
                "Attack_1": "33",
                "Attack_2": "111",
                "Attack_3": "300",
                "Attack_4": "88",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Rock Head"
            }
        ],
        "Hiker Wayne": [
            {
                "name": "Onix",
                "Level": 17,
                "Attack_1": "33",
                "Attack_2": "103",
                "Attack_3": "20",
                "Attack_4": "88",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Rock Head"
            }
        ],
        "Youngster Joey": [
            {
                "name": "Rattata",
                "Level": 15,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Run Away"
            },
            {
                "name": "Spearow",
                "Level": 15,
                "Attack_1": "64",
                "Attack_2": "45",
                "Attack_3": "43",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Keen Eye"
            }
        ],
        "Youngster Dan": [
            {
                "name": "Slowpoke",
                "Level": 17,
                "Attack_1": "33",
                "Attack_2": "45",
                "Attack_3": "55",
                "Attack_4": "93",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Oblivious"
            }
        ],
        "Picknicker Kelsey": [
            {
                "name": "Nidoran\u2642",
                "Level": 15,
                "Attack_1": "43",
                "Attack_2": "64",
                "Attack_3": "116",
                "Attack_4": "24",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Poison Point"
            },
            {
                "name": "Nidoran\u2640",
                "Level": 15,
                "Attack_1": "45",
                "Attack_2": "10",
                "Attack_3": "39",
                "Attack_4": "24",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Poison Point"
            }
        ],
        "Hiker Nob": [
            {
                "name": "Geodude",
                "Level": 13,
                "Attack_1": "33",
                "Attack_2": "111",
                "Attack_3": "300",
                "Attack_4": "88",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Rock Head"
            },
            {
                "name": "Geodude",
                "Level": 13,
                "Attack_1": "33",
                "Attack_2": "111",
                "Attack_3": "300",
                "Attack_4": "88",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Rock Head"
            },
            {
                "name": "Machop",
                "Level": 13,
                "Attack_1": "67",
                "Attack_2": "43",
                "Attack_3": "116",
                "Attack_4": "2",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Guts"
            },
            {
                "name": "Geodude",
                "Level": 13,
                "Attack_1": "33",
                "Attack_2": "111",
                "Attack_3": "300",
                "Attack_4": "88",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Rock Head"
            }
        ],
        "Camper Flint": [
            {
                "name": "Rattata",
                "Level": 14,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Run Away"
            },
            {
                "name": "Ekans",
                "Level": 14,
                "Attack_1": "35",
                "Attack_2": "43",
                "Attack_3": "40",
                "Attack_4": "44",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Intimidate"
            }
        ],
        "Youngster Chad": [
            {
                "name": "Ekans",
                "Level": 14,
                "Attack_1": "35",
                "Attack_2": "43",
                "Attack_3": "40",
                "Attack_4": "44",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Intimidate"
            },
            {
                "name": "Sandshrew",
                "Level": 14,
                "Attack_1": "10",
                "Attack_2": "111",
                "Attack_3": "28",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Sand Veil"
            }
        ],
        "Lass Haley": [
            {
                "name": "Oddish",
                "Level": 13,
                "Attack_1": "71",
                "Attack_2": "230",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Pidgey",
                "Level": 13,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": "16",
                "Attack_4": "98",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Keen Eye"
            },
            {
                "name": "Oddish",
                "Level": 13,
                "Attack_1": "71",
                "Attack_2": "230",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Chlorophyll"
            }
        ],
        "Swimmer Luis": [
            {
                "name": "Horsea",
                "Level": 16,
                "Attack_1": "145",
                "Attack_2": "108",
                "Attack_3": "43",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Swift Swim"
            },
            {
                "name": "Shellder",
                "Level": 16,
                "Attack_1": "33",
                "Attack_2": "110",
                "Attack_3": "333",
                "Attack_4": "48",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Shell Armor"
            }
        ],
        "Picknicker Diana": [
            {
                "name": "Goldeen",
                "Level": 19,
                "Attack_1": "39",
                "Attack_2": "346",
                "Attack_3": "48",
                "Attack_4": "30",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Swift Swim"
            }
        ],
        "Leader Misty": [
            {
                "name": "Staryu",
                "Level": 18,
                "Attack_1": "33",
                "Attack_2": "106",
                "Attack_3": "105",
                "Attack_4": "352",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Illuminate"
            },
            {
                "name": "Starmie",
                "Level": 21,
                "Attack_1": "129",
                "Attack_2": "105",
                "Attack_3": "229",
                "Attack_4": "352",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Illuminate"
            }
        ],
        "Team Rocket Grunt 6": [
            {
                "name": "Machop",
                "Level": 17,
                "Attack_1": "67",
                "Attack_2": "43",
                "Attack_3": "116",
                "Attack_4": "2",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Guts"
            },
            {
                "name": "Drowzee",
                "Level": 17,
                "Attack_1": "95",
                "Attack_2": "50",
                "Attack_3": "93",
                "Attack_4": "29",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Insomnia"
            }
        ],
        "Bug Catcher Keigo": [
            {
                "name": "Weedle",
                "Level": 16,
                "Attack_1": "40",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Shield Dust"
            },
            {
                "name": "Caterpie",
                "Level": 16,
                "Attack_1": "33",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Shield Dust"
            },
            {
                "name": "Weedle",
                "Level": 16,
                "Attack_1": "40",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Shield Dust"
            }
        ],
        "Camper Ricky": [
            {
                "name": "Squirtle",
                "Level": 20,
                "Attack_1": "145",
                "Attack_2": "110",
                "Attack_3": "55",
                "Attack_4": "44",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Torrent"
            }
        ],
        "Picknicker Nancy": [
            {
                "name": "Rattata",
                "Level": 16,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Run Away"
            },
            {
                "name": "Pikachu",
                "Level": 16,
                "Attack_1": "39",
                "Attack_2": "86",
                "Attack_3": "98",
                "Attack_4": "104",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Static"
            }
        ],
        "Bug Catcher Elijah": [
            {
                "name": "Butterfree",
                "Level": 20,
                "Attack_1": "77",
                "Attack_2": "78",
                "Attack_3": "79",
                "Attack_4": "48",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Compoundeyes"
            }
        ],
        "Picknicker Isabelle": [
            {
                "name": "Pidgey",
                "Level": 16,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": "16",
                "Attack_4": "98",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Keen Eye"
            },
            {
                "name": "Pidgey",
                "Level": 16,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": "16",
                "Attack_4": "98",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Keen Eye"
            },
            {
                "name": "Pidgey",
                "Level": 16,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": "16",
                "Attack_4": "98",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Keen Eye"
            }
        ],
        "Camper Jeff": [
            {
                "name": "Spearow",
                "Level": 16,
                "Attack_1": "64",
                "Attack_2": "45",
                "Attack_3": "43",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Keen Eye"
            },
            {
                "name": "Raticate",
                "Level": 16,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Run Away"
            }
        ],
        "Youngster Eddie": [
            {
                "name": "Ekans",
                "Level": 21,
                "Attack_1": "43",
                "Attack_2": "40",
                "Attack_3": "44",
                "Attack_4": "137",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Intimidate"
            }
        ],
        "Gamer Hugo": [
            {
                "name": "Poliwag",
                "Level": 18,
                "Attack_1": "145",
                "Attack_2": "95",
                "Attack_3": "55",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Water Absorb"
            },
            {
                "name": "Horsea",
                "Level": 18,
                "Attack_1": "145",
                "Attack_2": "108",
                "Attack_3": "43",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Swift Swim"
            }
        ],
        "Engineer Bernie": [
            {
                "name": "Magnemite",
                "Level": 18,
                "Attack_1": "33",
                "Attack_2": "84",
                "Attack_3": "48",
                "Attack_4": "49",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Magnet Pull"
            },
            {
                "name": "Magnemite",
                "Level": 18,
                "Attack_1": "33",
                "Attack_2": "84",
                "Attack_3": "48",
                "Attack_4": "49",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Magnet Pull"
            },
            {
                "name": "Magneton",
                "Level": 18,
                "Attack_1": "33",
                "Attack_2": "84",
                "Attack_3": "48",
                "Attack_4": "49",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Magnet Pull"
            }
        ],
        "Youngster Dave": [
            {
                "name": "Nidoran\u2642",
                "Level": 18,
                "Attack_1": "64",
                "Attack_2": "116",
                "Attack_3": "24",
                "Attack_4": "40",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Poison Point"
            },
            {
                "name": "Nidorino",
                "Level": 18,
                "Attack_1": "64",
                "Attack_2": "116",
                "Attack_3": "24",
                "Attack_4": "40",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Poison Point"
            }
        ],
        "Youngster Dillon": [
            {
                "name": "Sandshrew",
                "Level": 19,
                "Attack_1": "10",
                "Attack_2": "111",
                "Attack_3": "28",
                "Attack_4": "40",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Sand Veil"
            },
            {
                "name": "Zubat",
                "Level": 19,
                "Attack_1": "141",
                "Attack_2": "310",
                "Attack_3": "48",
                "Attack_4": "44",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Inner Focus"
            }
        ],
        "Gamer Jasper": [
            {
                "name": "Bellsprout",
                "Level": 18,
                "Attack_1": "74",
                "Attack_2": "35",
                "Attack_3": "79",
                "Attack_4": "77",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Oddish",
                "Level": 18,
                "Attack_1": "230",
                "Attack_2": "77",
                "Attack_3": "78",
                "Attack_4": "79",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Chlorophyll"
            }
        ],
        "Engineer Braxton": [
            {
                "name": "Magnemite",
                "Level": 21,
                "Attack_1": "84",
                "Attack_2": "48",
                "Attack_3": "49",
                "Attack_4": "86",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Magnet Pull"
            }
        ],
        "Youngster Yasu": [
            {
                "name": "Rattata",
                "Level": 17,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Run Away"
            },
            {
                "name": "Rattata",
                "Level": 17,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Run Away"
            },
            {
                "name": "Raticate",
                "Level": 17,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Run Away"
            }
        ],
        "Gamer Darian": [
            {
                "name": "Growlithe",
                "Level": 18,
                "Attack_1": "44",
                "Attack_2": "46",
                "Attack_3": "52",
                "Attack_4": "43",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Intimidate"
            },
            {
                "name": "Vulpix",
                "Level": 18,
                "Attack_1": "39",
                "Attack_2": "46",
                "Attack_3": "98",
                "Attack_4": "261",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Flash Fire"
            }
        ],
        "Gamer Dirk": [
            {
                "name": "Voltorb",
                "Level": 18,
                "Attack_1": "268",
                "Attack_2": "33",
                "Attack_3": "103",
                "Attack_4": "49",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Soundproof"
            },
            {
                "name": "Magnemite",
                "Level": 18,
                "Attack_1": "33",
                "Attack_2": "84",
                "Attack_3": "48",
                "Attack_4": "49",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Magnet Pull"
            }
        ],
        "Fisherman Barny": [
            {
                "name": "Tentacool",
                "Level": 17,
                "Attack_1": "40",
                "Attack_2": "48",
                "Attack_3": "132",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Clear Body"
            },
            {
                "name": "Staryu",
                "Level": 17,
                "Attack_1": "106",
                "Attack_2": "55",
                "Attack_3": "229",
                "Attack_4": "105",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Illuminate"
            },
            {
                "name": "Shellder",
                "Level": 17,
                "Attack_1": "33",
                "Attack_2": "110",
                "Attack_3": "333",
                "Attack_4": "48",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Shell Armor"
            }
        ],
        "Sailor Phillip": [
            {
                "name": "Machop",
                "Level": 20,
                "Attack_1": "43",
                "Attack_2": "116",
                "Attack_3": "2",
                "Attack_4": "69",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Guts"
            }
        ],
        "Sailor Huey": [
            {
                "name": "Tentacool",
                "Level": 18,
                "Attack_1": "40",
                "Attack_2": "48",
                "Attack_3": "132",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Clear Body"
            },
            {
                "name": "Staryu",
                "Level": 18,
                "Attack_1": "106",
                "Attack_2": "55",
                "Attack_3": "229",
                "Attack_4": "105",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Illuminate"
            }
        ],
        "Sailor Dylan": [
            {
                "name": "Horsea",
                "Level": 17,
                "Attack_1": "145",
                "Attack_2": "108",
                "Attack_3": "43",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Swift Swim"
            },
            {
                "name": "Horsea",
                "Level": 17,
                "Attack_1": "145",
                "Attack_2": "108",
                "Attack_3": "43",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Swift Swim"
            },
            {
                "name": "Horsea",
                "Level": 17,
                "Attack_1": "145",
                "Attack_2": "108",
                "Attack_3": "43",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Swift Swim"
            }
        ],
        "Sailor Duncan": [
            {
                "name": "Horsea",
                "Level": 17,
                "Attack_1": "145",
                "Attack_2": "108",
                "Attack_3": "43",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Swift Swim"
            },
            {
                "name": "Shellder",
                "Level": 17,
                "Attack_1": "33",
                "Attack_2": "110",
                "Attack_3": "333",
                "Attack_4": "48",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Shell Armor"
            },
            {
                "name": "Tentacool",
                "Level": 17,
                "Attack_1": "40",
                "Attack_2": "48",
                "Attack_3": "132",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Clear Body"
            }
        ],
        "Sailor Leonard": [
            {
                "name": "Shellder",
                "Level": 21,
                "Attack_1": "33",
                "Attack_2": "110",
                "Attack_3": "333",
                "Attack_4": "48",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Shell Armor"
            }
        ],
        "Gentleman Thomas": [
            {
                "name": "Growlithe",
                "Level": 18,
                "Attack_1": "44",
                "Attack_2": "46",
                "Attack_3": "52",
                "Attack_4": "43",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Intimidate"
            },
            {
                "name": "Growlithe",
                "Level": 18,
                "Attack_1": "44",
                "Attack_2": "46",
                "Attack_3": "52",
                "Attack_4": "43",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Intimidate"
            }
        ],
        "Gentleman Arthur": [
            {
                "name": "Nidoran\u2642",
                "Level": 19,
                "Attack_1": "64",
                "Attack_2": "116",
                "Attack_3": "24",
                "Attack_4": "40",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Poison Point"
            },
            {
                "name": "Nidoran\u2640",
                "Level": 19,
                "Attack_1": "10",
                "Attack_2": "39",
                "Attack_3": "24",
                "Attack_4": "40",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Poison Point"
            }
        ],
        "Lass Ann": [
            {
                "name": "Pidgey",
                "Level": 18,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": "16",
                "Attack_4": "98",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Keen Eye"
            },
            {
                "name": "Nidoran\u2640",
                "Level": 18,
                "Attack_1": "10",
                "Attack_2": "39",
                "Attack_3": "24",
                "Attack_4": "40",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Poison Point"
            }
        ],
        "Youngster Tyler": [
            {
                "name": "Nidoran\u2642",
                "Level": 21,
                "Attack_1": "116",
                "Attack_2": "24",
                "Attack_3": "40",
                "Attack_4": "30",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Poison Point"
            }
        ],
        "Sailor Trevor": [
            {
                "name": "Machop",
                "Level": 17,
                "Attack_1": "67",
                "Attack_2": "43",
                "Attack_3": "116",
                "Attack_4": "2",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Guts"
            },
            {
                "name": "Tentacool",
                "Level": 17,
                "Attack_1": "40",
                "Attack_2": "48",
                "Attack_3": "132",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Clear Body"
            }
        ],
        "Sailor Edmond": [
            {
                "name": "Machop",
                "Level": 18,
                "Attack_1": "67",
                "Attack_2": "43",
                "Attack_3": "116",
                "Attack_4": "2",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Guts"
            },
            {
                "name": "Shellder",
                "Level": 18,
                "Attack_1": "33",
                "Attack_2": "110",
                "Attack_3": "333",
                "Attack_4": "48",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Shell Armor"
            }
        ],
        "Fisherman Dale": [
            {
                "name": "Goldeen",
                "Level": 17,
                "Attack_1": "39",
                "Attack_2": "346",
                "Attack_3": "48",
                "Attack_4": "30",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Swift Swim"
            },
            {
                "name": "Goldeen",
                "Level": 17,
                "Attack_1": "39",
                "Attack_2": "346",
                "Attack_3": "48",
                "Attack_4": "30",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Swift Swim"
            },
            {
                "name": "Tentacool",
                "Level": 17,
                "Attack_1": "40",
                "Attack_2": "48",
                "Attack_3": "132",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Clear Body"
            }
        ],
        "Gentleman Brooks": [
            {
                "name": "Pikachu",
                "Level": 23,
                "Attack_1": "86",
                "Attack_2": "98",
                "Attack_3": "104",
                "Attack_4": "21",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Static"
            }
        ],
        "Lass Dawn": [
            {
                "name": "Rattata",
                "Level": 18,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Run Away"
            },
            {
                "name": "Pikachu",
                "Level": 18,
                "Attack_1": "39",
                "Attack_2": "86",
                "Attack_3": "98",
                "Attack_4": "104",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Static"
            }
        ],
        "Gentleman Lamar": [
            {
                "name": "Growlithe",
                "Level": 17,
                "Attack_1": "44",
                "Attack_2": "46",
                "Attack_3": "52",
                "Attack_4": "43",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Intimidate"
            },
            {
                "name": "Ponyta",
                "Level": 17,
                "Attack_1": "98",
                "Attack_2": "45",
                "Attack_3": "39",
                "Attack_4": "52",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Run Away"
            }
        ],
        "S.S Anne Rival(Bulbasaur)": [
            {
                "name": "Pidgeotto",
                "Level": 19,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": "16",
                "Attack_4": "98",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Keen Eye"
            },
            {
                "name": "Raticate",
                "Level": 16,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Run Away"
            },
            {
                "name": "Kadabra",
                "Level": 18,
                "Attack_1": "100",
                "Attack_2": "134",
                "Attack_3": "93",
                "Attack_4": "50",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Synchronize"
            },
            {
                "name": "Charmeleon",
                "Level": 20,
                "Attack_1": "45",
                "Attack_2": "52",
                "Attack_3": "232",
                "Attack_4": "108",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Blaze"
            }
        ],
        "S.S Anne Rival(Squirtle)": [
            {
                "name": "Pidgeotto",
                "Level": 19,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": "16",
                "Attack_4": "98",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Keen Eye"
            },
            {
                "name": "Raticate",
                "Level": 16,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Run Away"
            },
            {
                "name": "Kadabra",
                "Level": 18,
                "Attack_1": "100",
                "Attack_2": "134",
                "Attack_3": "93",
                "Attack_4": "50",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Synchronize"
            },
            {
                "name": "Ivysaur",
                "Level": 20,
                "Attack_1": "73",
                "Attack_2": "22",
                "Attack_3": "77",
                "Attack_4": "79",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Overgrow"
            }
        ],
        "S.S Anne Rival(Charmander)": [
            {
                "name": "Pidgeotto",
                "Level": 19,
                "Attack_1": "33",
                "Attack_2": "28",
                "Attack_3": "16",
                "Attack_4": "98",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Keen Eye"
            },
            {
                "name": "Raticate",
                "Level": 16,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Run Away"
            },
            {
                "name": "Kadabra",
                "Level": 18,
                "Attack_1": "100",
                "Attack_2": "134",
                "Attack_3": "93",
                "Attack_4": "50",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Synchronize"
            },
            {
                "name": "Wartortle",
                "Level": 20,
                "Attack_1": "145",
                "Attack_2": "110",
                "Attack_3": "55",
                "Attack_4": "44",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Torrent"
            }
        ],
        "Sailor Dwayne": [
            {
                "name": "Pikachu",
                "Level": 21,
                "Attack_1": "86",
                "Attack_2": "98",
                "Attack_3": "104",
                "Attack_4": "21",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Static"
            },
            {
                "name": "Pikachu",
                "Level": 21,
                "Attack_1": "86",
                "Attack_2": "98",
                "Attack_3": "104",
                "Attack_4": "21",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Static"
            }
        ],
        "Engineer Baily": [
            {
                "name": "Voltorb",
                "Level": 21,
                "Attack_1": "33",
                "Attack_2": "103",
                "Attack_3": "49",
                "Attack_4": "209",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Soundproof"
            },
            {
                "name": "Magnemite",
                "Level": 21,
                "Attack_1": "84",
                "Attack_2": "48",
                "Attack_3": "49",
                "Attack_4": "86",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Magnet Pull"
            }
        ],
        "Gentleman Tucker": [
            {
                "name": "Pikachu",
                "Level": 23,
                "Attack_1": "86",
                "Attack_2": "98",
                "Attack_3": "104",
                "Attack_4": "21",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Static"
            }
        ],
        "Leader Surge": [
            {
                "name": "Voltorb",
                "Level": 21,
                "Attack_1": "49",
                "Attack_2": "33",
                "Attack_3": "103",
                "Attack_4": "351",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Soundproof"
            },
            {
                "name": "Pikachu",
                "Level": 18,
                "Attack_1": "98",
                "Attack_2": "86",
                "Attack_3": "104",
                "Attack_4": "351",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Static"
            },
            {
                "name": "Raichu",
                "Level": 24,
                "Attack_1": "98",
                "Attack_2": "86",
                "Attack_3": "104",
                "Attack_4": "351",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Static"
            }
        ],
        "Picknicker Alicia": [
            {
                "name": "Oddish",
                "Level": 18,
                "Attack_1": "230",
                "Attack_2": "77",
                "Attack_3": "78",
                "Attack_4": "79",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Bellsprout",
                "Level": 18,
                "Attack_1": "74",
                "Attack_2": "35",
                "Attack_3": "79",
                "Attack_4": "77",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Oddish",
                "Level": 18,
                "Attack_1": "230",
                "Attack_2": "77",
                "Attack_3": "78",
                "Attack_4": "79",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Bellsprout",
                "Level": 18,
                "Attack_1": "74",
                "Attack_2": "35",
                "Attack_3": "79",
                "Attack_4": "77",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Chlorophyll"
            }
        ],
        "Hiker Jeremy": [
            {
                "name": "Machop",
                "Level": 20,
                "Attack_1": "43",
                "Attack_2": "116",
                "Attack_3": "2",
                "Attack_4": "69",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Guts"
            },
            {
                "name": "Onix",
                "Level": 20,
                "Attack_1": "103",
                "Attack_2": "20",
                "Attack_3": "88",
                "Attack_4": "106",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Rock Head"
            }
        ],
        "Camper Chris": [
            {
                "name": "Growlithe",
                "Level": 21,
                "Attack_1": "46",
                "Attack_2": "52",
                "Attack_3": "43",
                "Attack_4": "316",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Intimidate"
            },
            {
                "name": "Charmander",
                "Level": 21,
                "Attack_1": "45",
                "Attack_2": "52",
                "Attack_3": "232",
                "Attack_4": "108",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Blaze"
            }
        ],
        "Bug Catcher Brent": [
            {
                "name": "Beedrill",
                "Level": 19,
                "Attack_1": "31",
                "Attack_2": "116",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Swarm"
            },
            {
                "name": "Beedrill",
                "Level": 19,
                "Attack_1": "31",
                "Attack_2": "116",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Swarm"
            }
        ],
        "Bug Catcher Conner": [
            {
                "name": "Caterpie",
                "Level": 20,
                "Attack_1": "33",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Shield Dust"
            },
            {
                "name": "Weedle",
                "Level": 20,
                "Attack_1": "40",
                "Attack_2": "81",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Shield Dust"
            },
            {
                "name": "Venonat",
                "Level": 20,
                "Attack_1": "193",
                "Attack_2": "48",
                "Attack_3": "93",
                "Attack_4": "77",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Compoundeyes"
            }
        ],
        "Camper Drew": [
            {
                "name": "Rattata",
                "Level": 19,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Run Away"
            },
            {
                "name": "Sandshrew",
                "Level": 19,
                "Attack_1": "10",
                "Attack_2": "111",
                "Attack_3": "28",
                "Attack_4": "40",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Sand Veil"
            },
            {
                "name": "Ekans",
                "Level": 19,
                "Attack_1": "35",
                "Attack_2": "43",
                "Attack_3": "40",
                "Attack_4": "44",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Intimidate"
            },
            {
                "name": "Sandshrew",
                "Level": 19,
                "Attack_1": "10",
                "Attack_2": "111",
                "Attack_3": "28",
                "Attack_4": "40",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Sand Veil"
            }
        ],
        "Hiker Alan": [
            {
                "name": "Geodude",
                "Level": 21,
                "Attack_1": "222",
                "Attack_2": "88",
                "Attack_3": "300",
                "Attack_4": "111",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Rock Head"
            },
            {
                "name": "Onix",
                "Level": 21,
                "Attack_1": "106",
                "Attack_2": "88",
                "Attack_3": "20",
                "Attack_4": "103",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Rock Head"
            }
        ],
        "Hiker Brice": [
            {
                "name": "Geodude",
                "Level": 20,
                "Attack_1": "111",
                "Attack_2": "300",
                "Attack_3": "88",
                "Attack_4": "222",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Rock Head"
            },
            {
                "name": "Geodude",
                "Level": 20,
                "Attack_1": "111",
                "Attack_2": "300",
                "Attack_3": "88",
                "Attack_4": "222",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Rock Head"
            },
            {
                "name": "Machop",
                "Level": 20,
                "Attack_1": "43",
                "Attack_2": "116",
                "Attack_3": "2",
                "Attack_4": "69",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Guts"
            }
        ],
        "Picknicker Caitlin": [
            {
                "name": "Meowth",
                "Level": 23,
                "Attack_1": "10",
                "Attack_2": "45",
                "Attack_3": "44",
                "Attack_4": "6",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Pickup"
            }
        ],
        "Picknicker Heidi": [
            {
                "name": "Pikachu",
                "Level": 20,
                "Attack_1": "86",
                "Attack_2": "98",
                "Attack_3": "104",
                "Attack_4": "21",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Static"
            },
            {
                "name": "Clefairy",
                "Level": 20,
                "Attack_1": "227",
                "Attack_2": "47",
                "Attack_3": "3",
                "Attack_4": "266",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Cute Charm"
            }
        ],
        "Pokemaniac Mark": [
            {
                "name": "Rhyhorn",
                "Level": 29,
                "Attack_1": "23",
                "Attack_2": "31",
                "Attack_3": "184",
                "Attack_4": "350",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Lightningrod"
            },
            {
                "name": "Lickitung",
                "Level": 29,
                "Attack_1": "111",
                "Attack_2": "282",
                "Attack_3": "23",
                "Attack_4": "35",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Own Tempo"
            }
        ],
        "Picknicker Carol": [
            {
                "name": "Pidgey",
                "Level": 21,
                "Attack_1": "28",
                "Attack_2": "16",
                "Attack_3": "98",
                "Attack_4": "18",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Keen Eye"
            },
            {
                "name": "Pidgeotto",
                "Level": 21,
                "Attack_1": "28",
                "Attack_2": "16",
                "Attack_3": "98",
                "Attack_4": "18",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Keen Eye"
            }
        ],
        "Hiker Clark": [
            {
                "name": "Geodude",
                "Level": 21,
                "Attack_1": "222",
                "Attack_2": "88",
                "Attack_3": "300",
                "Attack_4": "111",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Rock Head"
            },
            {
                "name": "Onix",
                "Level": 21,
                "Attack_1": "106",
                "Attack_2": "88",
                "Attack_3": "20",
                "Attack_4": "103",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Rock Head"
            }
        ],
        "Hiker Trent": [
            {
                "name": "Onix",
                "Level": 19,
                "Attack_1": "103",
                "Attack_2": "20",
                "Attack_3": "88",
                "Attack_4": "106",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Rock Head"
            },
            {
                "name": "Graveler",
                "Level": 19,
                "Attack_1": "111",
                "Attack_2": "300",
                "Attack_3": "88",
                "Attack_4": "222",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Rock Head"
            }
        ],
        "Pokemaniac Herman": [
            {
                "name": "Cubone",
                "Level": 20,
                "Attack_1": "39",
                "Attack_2": "125",
                "Attack_3": "29",
                "Attack_4": "43",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Rock Head"
            },
            {
                "name": "Slowpoke",
                "Level": 20,
                "Attack_1": "33",
                "Attack_2": "45",
                "Attack_3": "55",
                "Attack_4": "93",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Oblivious"
            }
        ],
        "Pokemaniac Ashton": [
            {
                "name": "Cubone",
                "Level": 23,
                "Attack_1": "125",
                "Attack_2": "29",
                "Attack_3": "43",
                "Attack_4": "116",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Rock Head"
            },
            {
                "name": "Slowpoke",
                "Level": 23,
                "Attack_1": "33",
                "Attack_2": "45",
                "Attack_3": "55",
                "Attack_4": "93",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Oblivious"
            }
        ],
        "Pokemaniac Winston": [
            {
                "name": "Slowpoke",
                "Level": 25,
                "Attack_1": "45",
                "Attack_2": "55",
                "Attack_3": "93",
                "Attack_4": "50",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Oblivious"
            }
        ],
        "Picknicker Martha": [
            {
                "name": "Oddish",
                "Level": 22,
                "Attack_1": "230",
                "Attack_2": "77",
                "Attack_3": "78",
                "Attack_4": "79",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Bulbasaur",
                "Level": 22,
                "Attack_1": "22",
                "Attack_2": "77",
                "Attack_3": "79",
                "Attack_4": "75",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Overgrow"
            }
        ],
        "Pokemaniac Steve": [
            {
                "name": "Charmander",
                "Level": 22,
                "Attack_1": "45",
                "Attack_2": "52",
                "Attack_3": "232",
                "Attack_4": "108",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Blaze"
            },
            {
                "name": "Cubone",
                "Level": 22,
                "Attack_1": "125",
                "Attack_2": "29",
                "Attack_3": "43",
                "Attack_4": "116",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Rock Head"
            }
        ],
        "Hiker Allen": [
            {
                "name": "Geodude",
                "Level": 25,
                "Attack_1": "222",
                "Attack_2": "88",
                "Attack_3": "300",
                "Attack_4": "111",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Rock Head"
            }
        ],
        "Hiker Eric": [
            {
                "name": "Machop",
                "Level": 20,
                "Attack_1": "43",
                "Attack_2": "116",
                "Attack_3": "2",
                "Attack_4": "69",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Guts"
            },
            {
                "name": "Onix",
                "Level": 20,
                "Attack_1": "103",
                "Attack_2": "20",
                "Attack_3": "88",
                "Attack_4": "106",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Rock Head"
            }
        ],
        "Hiker Lenny": [
            {
                "name": "Geodude",
                "Level": 19,
                "Attack_1": "111",
                "Attack_2": "300",
                "Attack_3": "88",
                "Attack_4": "222",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Rock Head"
            },
            {
                "name": "Machop",
                "Level": 19,
                "Attack_1": "43",
                "Attack_2": "116",
                "Attack_3": "2",
                "Attack_4": "69",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Guts"
            },
            {
                "name": "Geodude",
                "Level": 19,
                "Attack_1": "111",
                "Attack_2": "300",
                "Attack_3": "88",
                "Attack_4": "222",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Rock Head"
            },
            {
                "name": "Geodude",
                "Level": 19,
                "Attack_1": "111",
                "Attack_2": "300",
                "Attack_3": "88",
                "Attack_4": "222",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Rock Head"
            }
        ],
        "Hiker Oliver": [
            {
                "name": "Onix",
                "Level": 20,
                "Attack_1": "103",
                "Attack_2": "20",
                "Attack_3": "88",
                "Attack_4": "106",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Rock Head"
            },
            {
                "name": "Onix",
                "Level": 20,
                "Attack_1": "103",
                "Attack_2": "20",
                "Attack_3": "88",
                "Attack_4": "106",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Rock Head"
            },
            {
                "name": "Geodude",
                "Level": 20,
                "Attack_1": "111",
                "Attack_2": "300",
                "Attack_3": "88",
                "Attack_4": "222",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Rock Head"
            }
        ],
        "Hiker Lucas": [
            {
                "name": "Geodude",
                "Level": 21,
                "Attack_1": "222",
                "Attack_2": "88",
                "Attack_3": "300",
                "Attack_4": "111",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Rock Head"
            },
            {
                "name": "Graveler",
                "Level": 21,
                "Attack_1": "222",
                "Attack_2": "88",
                "Attack_3": "300",
                "Attack_4": "111",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Rock Head"
            }
        ],
        "Picknicker Sofia": [
            {
                "name": "Jigglypuff",
                "Level": 21,
                "Attack_1": "111",
                "Attack_2": "1",
                "Attack_3": "50",
                "Attack_4": "205",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Cute Charm"
            },
            {
                "name": "Pidgey",
                "Level": 21,
                "Attack_1": "28",
                "Attack_2": "16",
                "Attack_3": "98",
                "Attack_4": "18",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Keen Eye"
            },
            {
                "name": "Meowth",
                "Level": 21,
                "Attack_1": "10",
                "Attack_2": "45",
                "Attack_3": "44",
                "Attack_4": "6",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Pickup"
            }
        ],
        "Hiker Dudley": [
            {
                "name": "Geodude",
                "Level": 21,
                "Attack_1": "222",
                "Attack_2": "88",
                "Attack_3": "300",
                "Attack_4": "111",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Rock Head"
            },
            {
                "name": "Geodude",
                "Level": 21,
                "Attack_1": "222",
                "Attack_2": "88",
                "Attack_3": "300",
                "Attack_4": "111",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Rock Head"
            },
            {
                "name": "Graveler",
                "Level": 21,
                "Attack_1": "222",
                "Attack_2": "88",
                "Attack_3": "300",
                "Attack_4": "111",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Rock Head"
            }
        ],
        "Pokemaniac Cooper": [
            {
                "name": "Slowpoke",
                "Level": 20,
                "Attack_1": "33",
                "Attack_2": "45",
                "Attack_3": "55",
                "Attack_4": "93",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Oblivious"
            },
            {
                "name": "Slowpoke",
                "Level": 20,
                "Attack_1": "33",
                "Attack_2": "45",
                "Attack_3": "55",
                "Attack_4": "93",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Oblivious"
            },
            {
                "name": "Slowpoke",
                "Level": 20,
                "Attack_1": "33",
                "Attack_2": "45",
                "Attack_3": "55",
                "Attack_4": "93",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Oblivious"
            }
        ],
        "Picknicker Leah": [
            {
                "name": "Bellsprout",
                "Level": 22,
                "Attack_1": "35",
                "Attack_2": "79",
                "Attack_3": "77",
                "Attack_4": "78",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Clefairy",
                "Level": 22,
                "Attack_1": "47",
                "Attack_2": "3",
                "Attack_3": "266",
                "Attack_4": "107",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Cute Charm"
            }
        ],
        "Picknicker Ariana": [
            {
                "name": "Pidgey",
                "Level": 19,
                "Attack_1": "28",
                "Attack_2": "16",
                "Attack_3": "98",
                "Attack_4": "18",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Keen Eye"
            },
            {
                "name": "Rattata",
                "Level": 19,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Run Away"
            },
            {
                "name": "Rattata",
                "Level": 19,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Run Away"
            },
            {
                "name": "Bellsprout",
                "Level": 19,
                "Attack_1": "35",
                "Attack_2": "79",
                "Attack_3": "77",
                "Attack_4": "78",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Chlorophyll"
            }
        ],
        "Picknicker Dana": [
            {
                "name": "Meowth",
                "Level": 20,
                "Attack_1": "10",
                "Attack_2": "45",
                "Attack_3": "44",
                "Attack_4": "6",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Pickup"
            },
            {
                "name": "Oddish",
                "Level": 20,
                "Attack_1": "230",
                "Attack_2": "77",
                "Attack_3": "78",
                "Attack_4": "79",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Pidgey",
                "Level": 20,
                "Attack_1": "28",
                "Attack_2": "16",
                "Attack_3": "98",
                "Attack_4": "18",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Keen Eye"
            }
        ],
        "Lass Julia": [
            {
                "name": "Clefairy",
                "Level": 22,
                "Attack_1": "47",
                "Attack_2": "3",
                "Attack_3": "266",
                "Attack_4": "107",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Cute Charm"
            },
            {
                "name": "Clefairy",
                "Level": 22,
                "Attack_1": "47",
                "Attack_2": "3",
                "Attack_3": "266",
                "Attack_4": "107",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Cute Charm"
            }
        ],
        "Gamer Rich": [
            {
                "name": "Growlithe",
                "Level": 24,
                "Attack_1": "46",
                "Attack_2": "52",
                "Attack_3": "43",
                "Attack_4": "316",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Intimidate"
            },
            {
                "name": "Vulpix",
                "Level": 24,
                "Attack_1": "46",
                "Attack_2": "98",
                "Attack_3": "261",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Flash Fire"
            }
        ],
        "Super Nerd Glen": [
            {
                "name": "Grimer",
                "Level": 22,
                "Attack_1": "106",
                "Attack_2": "50",
                "Attack_3": "124",
                "Attack_4": "107",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Stench"
            },
            {
                "name": "Muk",
                "Level": 22,
                "Attack_1": "106",
                "Attack_2": "50",
                "Attack_3": "124",
                "Attack_4": "107",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Stench"
            },
            {
                "name": "Grimer",
                "Level": 22,
                "Attack_1": "106",
                "Attack_2": "50",
                "Attack_3": "124",
                "Attack_4": "107",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Stench"
            }
        ],
        "Twins Eli & Anne": [
            {
                "name": "Clefairy",
                "Level": 22,
                "Attack_1": "47",
                "Attack_2": "3",
                "Attack_3": "266",
                "Attack_4": "107",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Cute Charm"
            },
            {
                "name": "Jigglypuff",
                "Level": 22,
                "Attack_1": "111",
                "Attack_2": "1",
                "Attack_3": "50",
                "Attack_4": "205",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Cute Charm"
            }
        ],
        "Lass Paige": [
            {
                "name": "Nidoran\u2642",
                "Level": 23,
                "Attack_1": "24",
                "Attack_2": "40",
                "Attack_3": "44",
                "Attack_4": "270",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Poison Point"
            },
            {
                "name": "Nidorina",
                "Level": 23,
                "Attack_1": "39",
                "Attack_2": "24",
                "Attack_3": "40",
                "Attack_4": "44",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Poison Point"
            }
        ],
        "Super Nerd Leslie": [
            {
                "name": "Koffing",
                "Level": 26,
                "Attack_1": "124",
                "Attack_2": "108",
                "Attack_3": "123",
                "Attack_4": "120",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Levitate"
            }
        ],
        "Lass Andrea": [
            {
                "name": "Meowth",
                "Level": 24,
                "Attack_1": "10",
                "Attack_2": "45",
                "Attack_3": "44",
                "Attack_4": "6",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Pickup"
            },
            {
                "name": "Meowth",
                "Level": 24,
                "Attack_1": "10",
                "Attack_2": "45",
                "Attack_3": "44",
                "Attack_4": "6",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Pickup"
            },
            {
                "name": "Meowth",
                "Level": 24,
                "Attack_1": "10",
                "Attack_2": "45",
                "Attack_3": "44",
                "Attack_4": "6",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Pickup"
            }
        ],
        "Lass Megan": [
            {
                "name": "Pidgey",
                "Level": 19,
                "Attack_1": "28",
                "Attack_2": "16",
                "Attack_3": "98",
                "Attack_4": "18",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Keen Eye"
            },
            {
                "name": "Rattata",
                "Level": 19,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Run Away"
            },
            {
                "name": "Nidoran\u2642",
                "Level": 19,
                "Attack_1": "64",
                "Attack_2": "116",
                "Attack_3": "24",
                "Attack_4": "40",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Poison Point"
            },
            {
                "name": "Meowth",
                "Level": 19,
                "Attack_1": "10",
                "Attack_2": "45",
                "Attack_3": "44",
                "Attack_4": "6",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Pickup"
            },
            {
                "name": "Pikachu",
                "Level": 19,
                "Attack_1": "39",
                "Attack_2": "86",
                "Attack_3": "98",
                "Attack_4": "104",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Static"
            }
        ],
        "Biker Jaren": [
            {
                "name": "Grimer",
                "Level": 24,
                "Attack_1": "106",
                "Attack_2": "50",
                "Attack_3": "124",
                "Attack_4": "107",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Stench"
            },
            {
                "name": "Grimer",
                "Level": 24,
                "Attack_1": "106",
                "Attack_2": "50",
                "Attack_3": "124",
                "Attack_4": "107",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Stench"
            }
        ],
        "Biker Ricardo": [
            {
                "name": "Koffing",
                "Level": 22,
                "Attack_1": "124",
                "Attack_2": "123",
                "Attack_3": "33",
                "Attack_4": "139",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Levitate"
            },
            {
                "name": "Koffing",
                "Level": 22,
                "Attack_1": "124",
                "Attack_2": "123",
                "Attack_3": "33",
                "Attack_4": "139",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Levitate"
            },
            {
                "name": "Grimer",
                "Level": 22,
                "Attack_1": "107",
                "Attack_2": "124",
                "Attack_3": "50",
                "Attack_4": "1",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Stench"
            }
        ],
        "Gamer Stan": [
            {
                "name": "Poliwag",
                "Level": 22,
                "Attack_1": "145",
                "Attack_2": "95",
                "Attack_3": "55",
                "Attack_4": "3",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Water Absorb"
            },
            {
                "name": "Poliwag",
                "Level": 22,
                "Attack_1": "145",
                "Attack_2": "95",
                "Attack_3": "55",
                "Attack_4": "3",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Water Absorb"
            },
            {
                "name": "Poliwag",
                "Level": 22,
                "Attack_1": "145",
                "Attack_2": "95",
                "Attack_3": "55",
                "Attack_4": "3",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Water Absorb"
            }
        ],
        "Super Nerd Aidan": [
            {
                "name": "Voltorb",
                "Level": 20,
                "Attack_1": "49",
                "Attack_2": "103",
                "Attack_3": "33",
                "Attack_4": "268",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Soundproof"
            },
            {
                "name": "Koffing",
                "Level": 20,
                "Attack_1": "124",
                "Attack_2": "123",
                "Attack_3": "33",
                "Attack_4": "139",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Levitate"
            },
            {
                "name": "Voltorb",
                "Level": 20,
                "Attack_1": "49",
                "Attack_2": "103",
                "Attack_3": "33",
                "Attack_4": "268",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Soundproof"
            },
            {
                "name": "Magnemite",
                "Level": 20,
                "Attack_1": "49",
                "Attack_2": "48",
                "Attack_3": "84",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Magnet Pull"
            }
        ],
        "Lass Kay": [
            {
                "name": "Bellsprout",
                "Level": 23,
                "Attack_1": "79",
                "Attack_2": "77",
                "Attack_3": "78",
                "Attack_4": "51",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Weepinbell",
                "Level": 23,
                "Attack_1": "35",
                "Attack_2": "79",
                "Attack_3": "77",
                "Attack_4": "78",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Chlorophyll"
            }
        ],
        "Beauty Bridget": [
            {
                "name": "Oddish",
                "Level": 21,
                "Attack_1": "230",
                "Attack_2": "77",
                "Attack_3": "78",
                "Attack_4": "79",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Bellsprout",
                "Level": 21,
                "Attack_1": "35",
                "Attack_2": "79",
                "Attack_3": "77",
                "Attack_4": "78",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Oddish",
                "Level": 21,
                "Attack_1": "230",
                "Attack_2": "77",
                "Attack_3": "78",
                "Attack_4": "79",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Bellsprout",
                "Level": 21,
                "Attack_1": "35",
                "Attack_2": "79",
                "Attack_3": "77",
                "Attack_4": "78",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Chlorophyll"
            }
        ],
        "Cooltrainer Mary": [
            {
                "name": "Bellsprout",
                "Level": 22,
                "Attack_1": "35",
                "Attack_2": "78",
                "Attack_3": "77",
                "Attack_4": "74",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Oddish",
                "Level": 22,
                "Attack_1": "77",
                "Attack_2": "78",
                "Attack_3": "71",
                "Attack_4": "230",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Weepinbell",
                "Level": 22,
                "Attack_1": "22",
                "Attack_2": "78",
                "Attack_3": "77",
                "Attack_4": "74",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Gloom",
                "Level": 22,
                "Attack_1": "71",
                "Attack_2": "78",
                "Attack_3": "77",
                "Attack_4": "230",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Ivysaur",
                "Level": 22,
                "Attack_1": "75",
                "Attack_2": "79",
                "Attack_3": "22",
                "Attack_4": "73",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Overgrow"
            }
        ],
        "Lass Lisa": [
            {
                "name": "Oddish",
                "Level": 23,
                "Attack_1": "77",
                "Attack_2": "78",
                "Attack_3": "79",
                "Attack_4": "51",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Gloom",
                "Level": 23,
                "Attack_1": "230",
                "Attack_2": "77",
                "Attack_3": "78",
                "Attack_4": "79",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Chlorophyll"
            }
        ],
        "Picknicker Tina": [
            {
                "name": "Bulbasaur",
                "Level": 24,
                "Attack_1": "22",
                "Attack_2": "77",
                "Attack_3": "79",
                "Attack_4": "75",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Overgrow"
            },
            {
                "name": "Ivysaur",
                "Level": 24,
                "Attack_1": "22",
                "Attack_2": "77",
                "Attack_3": "79",
                "Attack_4": "75",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Overgrow"
            }
        ],
        "Beauty Lori": [
            {
                "name": "Exeggcute",
                "Level": 24,
                "Attack_1": "95",
                "Attack_2": "115",
                "Attack_3": "73",
                "Attack_4": "93",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Chlorophyll"
            }
        ],
        "Beauty Tamia": [
            {
                "name": "Bellsprout",
                "Level": 24,
                "Attack_1": "79",
                "Attack_2": "77",
                "Attack_3": "78",
                "Attack_4": "51",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Bellsprout",
                "Level": 24,
                "Attack_1": "79",
                "Attack_2": "77",
                "Attack_3": "78",
                "Attack_4": "51",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Chlorophyll"
            }
        ],
        "Leader Erika": [
            {
                "name": "Victreebel",
                "Level": 29,
                "Attack_1": "78",
                "Attack_2": "51",
                "Attack_3": "77",
                "Attack_4": "202",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Tangela",
                "Level": 24,
                "Attack_1": "77",
                "Attack_2": "132",
                "Attack_3": "275",
                "Attack_4": "202",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Vileplume",
                "Level": 29,
                "Attack_1": "79",
                "Attack_2": "51",
                "Attack_3": "78",
                "Attack_4": "202",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Chlorophyll"
            }
        ],
        "Team Rocket Grunt 7": [
            {
                "name": "Raticate",
                "Level": 20,
                "Attack_1": "39",
                "Attack_2": "98",
                "Attack_3": "158",
                "Attack_4": "184",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Run Away"
            },
            {
                "name": "Zubat",
                "Level": 20,
                "Attack_1": "141",
                "Attack_2": "310",
                "Attack_3": "48",
                "Attack_4": "44",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Inner Focus"
            }
        ],
        "Team Rocket Grunt 8": [
            {
                "name": "Raticate",
                "Level": 21,
                "Attack_1": "39",
                "Attack_2": "98",
                "Attack_3": "158",
                "Attack_4": "184",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Run Away"
            },
            {
                "name": "Raticate",
                "Level": 21,
                "Attack_1": "39",
                "Attack_2": "98",
                "Attack_3": "158",
                "Attack_4": "184",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Run Away"
            }
        ],
        "Team Rocket Grunt 9": [
            {
                "name": "Drowzee",
                "Level": 21,
                "Attack_1": "50",
                "Attack_2": "93",
                "Attack_3": "29",
                "Attack_4": "139",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Insomnia"
            },
            {
                "name": "Machop",
                "Level": 21,
                "Attack_1": "43",
                "Attack_2": "116",
                "Attack_3": "2",
                "Attack_4": "69",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Guts"
            }
        ],
        "Team Rocket Grunt 10": [
            {
                "name": "Rattata",
                "Level": 19,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Run Away"
            },
            {
                "name": "Raticate",
                "Level": 19,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Run Away"
            },
            {
                "name": "Raticate",
                "Level": 19,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Run Away"
            },
            {
                "name": "Rattata",
                "Level": 19,
                "Attack_1": "33",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "158",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Run Away"
            }
        ],
        "Team Rocket Grunt 11": [
            {
                "name": "Grimer",
                "Level": 20,
                "Attack_1": "107",
                "Attack_2": "124",
                "Attack_3": "50",
                "Attack_4": "106",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Stench"
            },
            {
                "name": "Koffing",
                "Level": 20,
                "Attack_1": "123",
                "Attack_2": "33",
                "Attack_3": "139",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Levitate"
            },
            {
                "name": "Koffing",
                "Level": 20,
                "Attack_1": "123",
                "Attack_2": "33",
                "Attack_3": "139",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Levitate"
            }
        ],
        "Team Rocket Grunt 12": [
            {
                "name": "Grimer",
                "Level": 22,
                "Attack_1": "107",
                "Attack_2": "124",
                "Attack_3": "50",
                "Attack_4": "106",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Stench"
            },
            {
                "name": "Koffing",
                "Level": 22,
                "Attack_1": "124",
                "Attack_2": "123",
                "Attack_3": "33",
                "Attack_4": "139",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Levitate"
            }
        ],
        "Team Rocket Grunt 13": [
            {
                "name": "Zubat",
                "Level": 17,
                "Attack_1": "44",
                "Attack_2": "310",
                "Attack_3": "48",
                "Attack_4": "141",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Inner Focus"
            },
            {
                "name": "Koffing",
                "Level": 17,
                "Attack_1": "123",
                "Attack_2": "33",
                "Attack_3": "139",
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Levitate"
            },
            {
                "name": "Grimer",
                "Level": 17,
                "Attack_1": "124",
                "Attack_2": "50",
                "Attack_3": "106",
                "Attack_4": "1",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Stench"
            },
            {
                "name": "Zubat",
                "Level": 17,
                "Attack_1": "44",
                "Attack_2": "310",
                "Attack_3": "48",
                "Attack_4": "141",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Inner Focus"
            },
            {
                "name": "Raticate",
                "Level": 17,
                "Attack_1": "158",
                "Attack_2": "98",
                "Attack_3": "39",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Run Away"
            }
        ],
        "Team Rocket Grunt 14": [
            {
                "name": "Machop",
                "Level": 21,
                "Attack_1": "43",
                "Attack_2": "116",
                "Attack_3": "2",
                "Attack_4": "69",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Guts"
            },
            {
                "name": "Machop",
                "Level": 21,
                "Attack_1": "43",
                "Attack_2": "116",
                "Attack_3": "2",
                "Attack_4": "69",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Guts"
            }
        ],
        "Team Rocket Grunt 15": [
            {
                "name": "Rattata",
                "Level": 20,
                "Attack_1": "39",
                "Attack_2": "98",
                "Attack_3": "158",
                "Attack_4": "116",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Run Away"
            },
            {
                "name": "Raticate",
                "Level": 20,
                "Attack_1": "39",
                "Attack_2": "98",
                "Attack_3": "158",
                "Attack_4": "116",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Run Away"
            },
            {
                "name": "Drowzee",
                "Level": 20,
                "Attack_1": "95",
                "Attack_2": "50",
                "Attack_3": "93",
                "Attack_4": "29",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Insomnia"
            }
        ],
        "Team Rocket Grunt 16": [
            {
                "name": "Koffing",
                "Level": 21,
                "Attack_1": "33",
                "Attack_2": "123",
                "Attack_3": "120",
                "Attack_4": "124",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Levitate"
            },
            {
                "name": "Zubat",
                "Level": 21,
                "Attack_1": "310",
                "Attack_2": "48",
                "Attack_3": "44",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Inner Focus"
            }
        ],
        "Team Rocket Grunt 17": [
            {
                "name": "Sandshrew",
                "Level": 23,
                "Attack_1": "111",
                "Attack_2": "28",
                "Attack_3": "40",
                "Attack_4": "163",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Sand Veil"
            },
            {
                "name": "Ekans",
                "Level": 23,
                "Attack_1": "43",
                "Attack_2": "40",
                "Attack_3": "44",
                "Attack_4": "137",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Intimidate"
            },
            {
                "name": "Sandslash",
                "Level": 23,
                "Attack_1": "10",
                "Attack_2": "111",
                "Attack_3": "28",
                "Attack_4": "40",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Sand Veil"
            }
        ],
        "Team Rocket Grunt 18": [
            {
                "name": "Ekans",
                "Level": 23,
                "Attack_1": "43",
                "Attack_2": "40",
                "Attack_3": "44",
                "Attack_4": "137",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Intimidate"
            },
            {
                "name": "Sandshrew",
                "Level": 23,
                "Attack_1": "111",
                "Attack_2": "28",
                "Attack_3": "40",
                "Attack_4": "163",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Sand Veil"
            },
            {
                "name": "Arbok",
                "Level": 23,
                "Attack_1": "43",
                "Attack_2": "40",
                "Attack_3": "44",
                "Attack_4": "137",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Intimidate"
            }
        ],
        "Giovanni 1": [
            {
                "name": "Onix",
                "Level": 25,
                "Attack_1": "20",
                "Attack_2": "88",
                "Attack_3": "106",
                "Attack_4": "99",
                "IVS": {
                    "hp": 30,
                    "atk": 30,
                    "def": 30,
                    "spa": 30,
                    "spd": 30,
                    "spe": 30
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Rock Head"
            },
            {
                "name": "Rhyhorn",
                "Level": 24,
                "Attack_1": "39",
                "Attack_2": "23",
                "Attack_3": "31",
                "Attack_4": "184",
                "IVS": {
                    "hp": 30,
                    "atk": 30,
                    "def": 30,
                    "spa": 30,
                    "spd": 30,
                    "spe": 30
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Lightningrod"
            },
            {
                "name": "Kangaskhan",
                "Level": 29,
                "Attack_1": "44",
                "Attack_2": "39",
                "Attack_3": "252",
                "Attack_4": "5",
                "IVS": {
                    "hp": 30,
                    "atk": 30,
                    "def": 30,
                    "spa": 30,
                    "spd": 30,
                    "spe": 30
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Early Bird"
            }
        ],
        "Tower Rival(Bulbasaur)": [
            {
                "name": "Pidgeotto",
                "Level": 25,
                "Attack_1": "28",
                "Attack_2": "16",
                "Attack_3": "98",
                "Attack_4": "18",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Keen Eye"
            },
            {
                "name": "Kadabra",
                "Level": 20,
                "Attack_1": "100",
                "Attack_2": "134",
                "Attack_3": "93",
                "Attack_4": "50",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Synchronize"
            },
            {
                "name": "Exeggcute",
                "Level": 23,
                "Attack_1": "95",
                "Attack_2": "115",
                "Attack_3": "73",
                "Attack_4": "93",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Gyarados",
                "Level": 22,
                "Attack_1": "37",
                "Attack_2": "44",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Intimidate"
            },
            {
                "name": "Charmeleon",
                "Level": 25,
                "Attack_1": "45",
                "Attack_2": "52",
                "Attack_3": "232",
                "Attack_4": "45",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Blaze"
            }
        ],
        "Tower Rival(Squirtle)": [
            {
                "name": "Pidgeotto",
                "Level": 25,
                "Attack_1": "28",
                "Attack_2": "16",
                "Attack_3": "98",
                "Attack_4": "18",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Keen Eye"
            },
            {
                "name": "Kadabra",
                "Level": 20,
                "Attack_1": "100",
                "Attack_2": "134",
                "Attack_3": "93",
                "Attack_4": "50",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Synchronize"
            },
            {
                "name": "Growlithe",
                "Level": 22,
                "Attack_1": "46",
                "Attack_2": "52",
                "Attack_3": "43",
                "Attack_4": "316",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Intimidate"
            },
            {
                "name": "Gyarados",
                "Level": 23,
                "Attack_1": "37",
                "Attack_2": "44",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Intimidate"
            },
            {
                "name": "Ivysaur",
                "Level": 25,
                "Attack_1": "22",
                "Attack_2": "77",
                "Attack_3": "79",
                "Attack_4": "75",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Overgrow"
            }
        ],
        "Tower Rival(Charmander)": [
            {
                "name": "Pidgeotto",
                "Level": 25,
                "Attack_1": "28",
                "Attack_2": "16",
                "Attack_3": "98",
                "Attack_4": "18",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Keen Eye"
            },
            {
                "name": "Kadabra",
                "Level": 20,
                "Attack_1": "100",
                "Attack_2": "134",
                "Attack_3": "93",
                "Attack_4": "50",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Synchronize"
            },
            {
                "name": "Growlithe",
                "Level": 23,
                "Attack_1": "46",
                "Attack_2": "52",
                "Attack_3": "43",
                "Attack_4": "316",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Intimidate"
            },
            {
                "name": "Exeggcute",
                "Level": 22,
                "Attack_1": "95",
                "Attack_2": "115",
                "Attack_3": "73",
                "Attack_4": "93",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Wartortle",
                "Level": 25,
                "Attack_1": "110",
                "Attack_2": "55",
                "Attack_3": "44",
                "Attack_4": "229",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Torrent"
            }
        ],
        "Channeler Hope": [
            {
                "name": "Gastly",
                "Level": 23,
                "Attack_1": "180",
                "Attack_2": "174",
                "Attack_3": "101",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Levitate"
            }
        ],
        "Channeler Patricia": [
            {
                "name": "Gastly",
                "Level": 22,
                "Attack_1": "180",
                "Attack_2": "174",
                "Attack_3": "101",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Levitate"
            }
        ],
        "Channeler Carly": [
            {
                "name": "Gastly",
                "Level": 24,
                "Attack_1": "180",
                "Attack_2": "174",
                "Attack_3": "101",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Levitate"
            }
        ],
        "Channeler Laurel": [
            {
                "name": "Gastly",
                "Level": 23,
                "Attack_1": "180",
                "Attack_2": "174",
                "Attack_3": "101",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Levitate"
            },
            {
                "name": "Gastly",
                "Level": 23,
                "Attack_1": "180",
                "Attack_2": "174",
                "Attack_3": "101",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Levitate"
            }
        ],
        "Channeler Jody": [
            {
                "name": "Gastly",
                "Level": 22,
                "Attack_1": "180",
                "Attack_2": "174",
                "Attack_3": "101",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Levitate"
            }
        ],
        "Channeler Paula": [
            {
                "name": "Gastly",
                "Level": 24,
                "Attack_1": "180",
                "Attack_2": "174",
                "Attack_3": "101",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Levitate"
            }
        ],
        "Channeler Ruth": [
            {
                "name": "Gastly",
                "Level": 22,
                "Attack_1": "180",
                "Attack_2": "174",
                "Attack_3": "101",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Levitate"
            }
        ],
        "Channeler Tammy": [
            {
                "name": "Haunter",
                "Level": 23,
                "Attack_1": "180",
                "Attack_2": "174",
                "Attack_3": "101",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Levitate"
            }
        ],
        "Channeler Karina": [
            {
                "name": "Gastly",
                "Level": 24,
                "Attack_1": "180",
                "Attack_2": "174",
                "Attack_3": "101",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Levitate"
            }
        ],
        "Channeler Janae": [
            {
                "name": "Gastly",
                "Level": 22,
                "Attack_1": "180",
                "Attack_2": "174",
                "Attack_3": "101",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Levitate"
            }
        ],
        "Channeler Angelica": [
            {
                "name": "Gastly",
                "Level": 22,
                "Attack_1": "180",
                "Attack_2": "174",
                "Attack_3": "101",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Levitate"
            },
            {
                "name": "Gastly",
                "Level": 22,
                "Attack_1": "180",
                "Attack_2": "174",
                "Attack_3": "101",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Levitate"
            },
            {
                "name": "Gastly",
                "Level": 22,
                "Attack_1": "180",
                "Attack_2": "174",
                "Attack_3": "101",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Levitate"
            }
        ],
        "Channeler Jennifer": [
            {
                "name": "Gastly",
                "Level": 24,
                "Attack_1": "180",
                "Attack_2": "174",
                "Attack_3": "101",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Levitate"
            }
        ],
        "Channeler Emilia": [
            {
                "name": "Gastly",
                "Level": 24,
                "Attack_1": "180",
                "Attack_2": "174",
                "Attack_3": "101",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Levitate"
            }
        ],
        "Team Rocket Grunt 19": [
            {
                "name": "Zubat",
                "Level": 25,
                "Attack_1": "310",
                "Attack_2": "48",
                "Attack_3": "44",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Inner Focus"
            },
            {
                "name": "Zubat",
                "Level": 25,
                "Attack_1": "310",
                "Attack_2": "48",
                "Attack_3": "44",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Inner Focus"
            },
            {
                "name": "Golbat",
                "Level": 25,
                "Attack_1": "310",
                "Attack_2": "48",
                "Attack_3": "44",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Inner Focus"
            }
        ],
        "Team Rocket Grunt 20": [
            {
                "name": "Koffing",
                "Level": 26,
                "Attack_1": "123",
                "Attack_2": "120",
                "Attack_3": "124",
                "Attack_4": "108",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Levitate"
            },
            {
                "name": "Drowzee",
                "Level": 26,
                "Attack_1": "50",
                "Attack_2": "93",
                "Attack_3": "29",
                "Attack_4": "139",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Insomnia"
            }
        ],
        "Team Rocket Grunt 21": [
            {
                "name": "Zubat",
                "Level": 23,
                "Attack_1": "310",
                "Attack_2": "48",
                "Attack_3": "44",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Inner Focus"
            },
            {
                "name": "Rattata",
                "Level": 23,
                "Attack_1": "39",
                "Attack_2": "98",
                "Attack_3": "158",
                "Attack_4": "116",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Run Away"
            },
            {
                "name": "Raticate",
                "Level": 23,
                "Attack_1": "39",
                "Attack_2": "98",
                "Attack_3": "158",
                "Attack_4": "184",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Run Away"
            },
            {
                "name": "Zubat",
                "Level": 23,
                "Attack_1": "310",
                "Attack_2": "48",
                "Attack_3": "44",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Inner Focus"
            }
        ],
        "Young Couple Lea & Jed": [
            {
                "name": "Rapidash",
                "Level": 29,
                "Attack_1": "39",
                "Attack_2": "52",
                "Attack_3": "23",
                "Attack_4": "83",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Run Away"
            },
            {
                "name": "Ninetales",
                "Level": 29,
                "Attack_1": "52",
                "Attack_2": "98",
                "Attack_3": "109",
                "Attack_4": "219",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Flash Fire"
            }
        ],
        "Biker Lao": [
            {
                "name": "Grimer",
                "Level": 29,
                "Attack_1": "103",
                "Attack_2": "107",
                "Attack_3": "124",
                "Attack_4": "50",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Stench"
            },
            {
                "name": "Koffing",
                "Level": 29,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Levitate"
            }
        ],
        "Cue Ball Koji": [
            {
                "name": "Machop",
                "Level": 28,
                "Attack_1": "2",
                "Attack_2": "69",
                "Attack_3": "193",
                "Attack_4": "279",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Guts"
            },
            {
                "name": "Mankey",
                "Level": 28,
                "Attack_1": "2",
                "Attack_2": "154",
                "Attack_3": "116",
                "Attack_4": "69",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Vital Spirit"
            },
            {
                "name": "Machop",
                "Level": 28,
                "Attack_1": "2",
                "Attack_2": "69",
                "Attack_3": "193",
                "Attack_4": "279",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Guts"
            }
        ],
        "Cue Ball Luke": [
            {
                "name": "Mankey",
                "Level": 29,
                "Attack_1": "2",
                "Attack_2": "154",
                "Attack_3": "116",
                "Attack_4": "69",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Vital Spirit"
            },
            {
                "name": "Machop",
                "Level": 29,
                "Attack_1": "2",
                "Attack_2": "69",
                "Attack_3": "193",
                "Attack_4": "279",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Guts"
            }
        ],
        "Biker Hideo": [
            {
                "name": "Weezing",
                "Level": 33,
                "Attack_1": "120",
                "Attack_2": "124",
                "Attack_3": "108",
                "Attack_4": "114",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Levitate"
            }
        ],
        "Biker Ruben": [
            {
                "name": "Weezing",
                "Level": 28,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Levitate"
            },
            {
                "name": "Koffing",
                "Level": 28,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Levitate"
            },
            {
                "name": "Weezing",
                "Level": 28,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Levitate"
            }
        ],
        "Cue Ball Camron": [
            {
                "name": "Mankey",
                "Level": 29,
                "Attack_1": "2",
                "Attack_2": "154",
                "Attack_3": "116",
                "Attack_4": "69",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Vital Spirit"
            },
            {
                "name": "Machop",
                "Level": 29,
                "Attack_1": "2",
                "Attack_2": "69",
                "Attack_3": "193",
                "Attack_4": "279",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Guts"
            }
        ],
        "Cue Ball Isaiah": [
            {
                "name": "Machop",
                "Level": 29,
                "Attack_1": "2",
                "Attack_2": "69",
                "Attack_3": "193",
                "Attack_4": "279",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Guts"
            },
            {
                "name": "Machamp",
                "Level": 29,
                "Attack_1": "2",
                "Attack_2": "69",
                "Attack_3": "193",
                "Attack_4": "279",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Guts"
            }
        ],
        "Biker Virgil": [
            {
                "name": "Weezing",
                "Level": 28,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Levitate"
            },
            {
                "name": "Koffing",
                "Level": 28,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Levitate"
            },
            {
                "name": "Weezing",
                "Level": 28,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Levitate"
            }
        ],
        "Cue Ball Raul": [
            {
                "name": "Mankey",
                "Level": 29,
                "Attack_1": "2",
                "Attack_2": "154",
                "Attack_3": "116",
                "Attack_4": "69",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Vital Spirit"
            },
            {
                "name": "Primeape",
                "Level": 29,
                "Attack_1": "154",
                "Attack_2": "116",
                "Attack_3": "69",
                "Attack_4": "99",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Vital Spirit"
            }
        ],
        "Biker Billy": [
            {
                "name": "Muk",
                "Level": 33,
                "Attack_1": "50",
                "Attack_2": "124",
                "Attack_3": "107",
                "Attack_4": "103",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Stench"
            }
        ],
        "Cue Ball Jamal": [
            {
                "name": "Mankey",
                "Level": 26,
                "Attack_1": "2",
                "Attack_2": "154",
                "Attack_3": "116",
                "Attack_4": "69",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Vital Spirit"
            },
            {
                "name": "Mankey",
                "Level": 26,
                "Attack_1": "2",
                "Attack_2": "154",
                "Attack_3": "116",
                "Attack_4": "69",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Vital Spirit"
            },
            {
                "name": "Machamp",
                "Level": 26,
                "Attack_1": "2",
                "Attack_2": "69",
                "Attack_3": "193",
                "Attack_4": "279",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Guts"
            },
            {
                "name": "Machop",
                "Level": 26,
                "Attack_1": "2",
                "Attack_2": "69",
                "Attack_3": "193",
                "Attack_4": "279",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Guts"
            }
        ],
        "Biker Nikolas": [
            {
                "name": "Voltorb",
                "Level": 29,
                "Attack_1": "209",
                "Attack_2": "49",
                "Attack_3": "103",
                "Attack_4": "268",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Soundproof"
            },
            {
                "name": "Voltorb",
                "Level": 29,
                "Attack_1": "209",
                "Attack_2": "49",
                "Attack_3": "103",
                "Attack_4": "268",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Soundproof"
            }
        ],
        "Cue Ball Zeek": [
            {
                "name": "Machoke",
                "Level": 33,
                "Attack_1": "69",
                "Attack_2": "193",
                "Attack_3": "279",
                "Attack_4": "233",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Guts"
            }
        ],
        "Cue Ball Corey": [
            {
                "name": "Primeape",
                "Level": 29,
                "Attack_1": "154",
                "Attack_2": "116",
                "Attack_3": "69",
                "Attack_4": "99",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Vital Spirit"
            },
            {
                "name": "Machoke",
                "Level": 29,
                "Attack_1": "2",
                "Attack_2": "69",
                "Attack_3": "193",
                "Attack_4": "279",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Guts"
            }
        ],
        "Biker Jaxon": [
            {
                "name": "Weezing",
                "Level": 29,
                "Attack_1": "124",
                "Attack_2": "108",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Levitate"
            },
            {
                "name": "Muk",
                "Level": 29,
                "Attack_1": "103",
                "Attack_2": "107",
                "Attack_3": "124",
                "Attack_4": "50",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Stench"
            }
        ],
        "Biker William": [
            {
                "name": "Koffing",
                "Level": 25,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Levitate"
            },
            {
                "name": "Weezing",
                "Level": 25,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Levitate"
            },
            {
                "name": "Koffing",
                "Level": 25,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Levitate"
            },
            {
                "name": "Koffing",
                "Level": 25,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Levitate"
            },
            {
                "name": "Weezing",
                "Level": 25,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "120",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Levitate"
            }
        ],
        "Bird Keeper Jacob": [
            {
                "name": "Spearow",
                "Level": 26,
                "Attack_1": "43",
                "Attack_2": "31",
                "Attack_3": "228",
                "Attack_4": "332",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Keen Eye"
            },
            {
                "name": "Spearow",
                "Level": 26,
                "Attack_1": "43",
                "Attack_2": "31",
                "Attack_3": "228",
                "Attack_4": "332",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Keen Eye"
            },
            {
                "name": "Fearow",
                "Level": 26,
                "Attack_1": "45",
                "Attack_2": "43",
                "Attack_3": "31",
                "Attack_4": "228",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Keen Eye"
            },
            {
                "name": "Spearow",
                "Level": 26,
                "Attack_1": "43",
                "Attack_2": "31",
                "Attack_3": "228",
                "Attack_4": "332",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Keen Eye"
            }
        ],
        "Bird Keeper Wilton": [
            {
                "name": "Spearow",
                "Level": 29,
                "Attack_1": "43",
                "Attack_2": "31",
                "Attack_3": "228",
                "Attack_4": "332",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Keen Eye"
            },
            {
                "name": "Fearow",
                "Level": 29,
                "Attack_1": "45",
                "Attack_2": "43",
                "Attack_3": "31",
                "Attack_4": "228",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Keen Eye"
            }
        ],
        "Bird Keeper Ramiro": [
            {
                "name": "Dodrio",
                "Level": 34,
                "Attack_1": "228",
                "Attack_2": "31",
                "Attack_3": "161",
                "Attack_4": "99",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Run Away"
            }
        ],
        "Juggler Nate": [
            {
                "name": "Drowzee",
                "Level": 34,
                "Attack_1": "29",
                "Attack_2": "139",
                "Attack_3": "96",
                "Attack_4": "94",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Insomnia"
            },
            {
                "name": "Kadabra",
                "Level": 34,
                "Attack_1": "115",
                "Attack_2": "105",
                "Attack_3": "248",
                "Attack_4": "272",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Synchronize"
            }
        ],
        "Juggler Kayden": [
            {
                "name": "Hypno",
                "Level": 38,
                "Attack_1": "29",
                "Attack_2": "139",
                "Attack_3": "96",
                "Attack_4": "94",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Insomnia"
            }
        ],
        "Juggler Kirk": [
            {
                "name": "Drowzee",
                "Level": 31,
                "Attack_1": "29",
                "Attack_2": "139",
                "Attack_3": "96",
                "Attack_4": "94",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Insomnia"
            },
            {
                "name": "Drowzee",
                "Level": 31,
                "Attack_1": "29",
                "Attack_2": "139",
                "Attack_3": "96",
                "Attack_4": "94",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Insomnia"
            },
            {
                "name": "Kadabra",
                "Level": 31,
                "Attack_1": "60",
                "Attack_2": "115",
                "Attack_3": "105",
                "Attack_4": "248",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Synchronize"
            },
            {
                "name": "Drowzee",
                "Level": 31,
                "Attack_1": "29",
                "Attack_2": "139",
                "Attack_3": "96",
                "Attack_4": "94",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Insomnia"
            }
        ],
        "Tamer Edgar": [
            {
                "name": "Arbok",
                "Level": 33,
                "Attack_1": "40",
                "Attack_2": "44",
                "Attack_3": "137",
                "Attack_4": "103",
                "IVS": {
                    "hp": 4,
                    "atk": 4,
                    "def": 4,
                    "spa": 4,
                    "spd": 4,
                    "spe": 4
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Intimidate"
            },
            {
                "name": "Arbok",
                "Level": 33,
                "Attack_1": "40",
                "Attack_2": "44",
                "Attack_3": "137",
                "Attack_4": "103",
                "IVS": {
                    "hp": 4,
                    "atk": 4,
                    "def": 4,
                    "spa": 4,
                    "spd": 4,
                    "spe": 4
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Intimidate"
            },
            {
                "name": "Sandslash",
                "Level": 33,
                "Attack_1": "28",
                "Attack_2": "40",
                "Attack_3": "163",
                "Attack_4": "129",
                "IVS": {
                    "hp": 4,
                    "atk": 4,
                    "def": 4,
                    "spa": 4,
                    "spd": 4,
                    "spe": 4
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Sand Veil"
            }
        ],
        "Tamer Phil": [
            {
                "name": "Sandslash",
                "Level": 34,
                "Attack_1": "28",
                "Attack_2": "40",
                "Attack_3": "163",
                "Attack_4": "129",
                "IVS": {
                    "hp": 4,
                    "atk": 4,
                    "def": 4,
                    "spa": 4,
                    "spd": 4,
                    "spe": 4
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Sand Veil"
            },
            {
                "name": "Arbok",
                "Level": 34,
                "Attack_1": "40",
                "Attack_2": "44",
                "Attack_3": "137",
                "Attack_4": "103",
                "IVS": {
                    "hp": 4,
                    "atk": 4,
                    "def": 4,
                    "spa": 4,
                    "spd": 4,
                    "spe": 4
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Intimidate"
            }
        ],
        "Juggler Shawn": [
            {
                "name": "Drowzee",
                "Level": 34,
                "Attack_1": "29",
                "Attack_2": "139",
                "Attack_3": "96",
                "Attack_4": "94",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Insomnia"
            },
            {
                "name": "Hypno",
                "Level": 34,
                "Attack_1": "93",
                "Attack_2": "29",
                "Attack_3": "139",
                "Attack_4": "96",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Insomnia"
            }
        ],
        "Leader Koga": [
            {
                "name": "Koffing",
                "Level": 37,
                "Attack_1": "120",
                "Attack_2": "124",
                "Attack_3": "108",
                "Attack_4": "92",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Levitate"
            },
            {
                "name": "Muk",
                "Level": 39,
                "Attack_1": "107",
                "Attack_2": "124",
                "Attack_3": "151",
                "Attack_4": "92",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Stench"
            },
            {
                "name": "Koffing",
                "Level": 37,
                "Attack_1": "120",
                "Attack_2": "124",
                "Attack_3": "108",
                "Attack_4": "92",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Levitate"
            },
            {
                "name": "Weezing",
                "Level": 43,
                "Attack_1": "33",
                "Attack_2": "124",
                "Attack_3": "108",
                "Attack_4": "92",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Levitate"
            }
        ],
        "Black Belt Hideki": [
            {
                "name": "Machop",
                "Level": 32,
                "Attack_1": "69",
                "Attack_2": "193",
                "Attack_3": "279",
                "Attack_4": "233",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Guts"
            },
            {
                "name": "Machoke",
                "Level": 32,
                "Attack_1": "2",
                "Attack_2": "69",
                "Attack_3": "193",
                "Attack_4": "279",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Guts"
            }
        ],
        "Black Belt Hitoshi": [
            {
                "name": "Machop",
                "Level": 31,
                "Attack_1": "69",
                "Attack_2": "193",
                "Attack_3": "279",
                "Attack_4": "233",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Guts"
            },
            {
                "name": "Mankey",
                "Level": 31,
                "Attack_1": "154",
                "Attack_2": "116",
                "Attack_3": "69",
                "Attack_4": "238",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Vital Spirit"
            },
            {
                "name": "Primeape",
                "Level": 31,
                "Attack_1": "154",
                "Attack_2": "116",
                "Attack_3": "69",
                "Attack_4": "99",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Vital Spirit"
            }
        ],
        "Black Belt Mike": [
            {
                "name": "Mankey",
                "Level": 31,
                "Attack_1": "154",
                "Attack_2": "116",
                "Attack_3": "69",
                "Attack_4": "238",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Vital Spirit"
            },
            {
                "name": "Mankey",
                "Level": 31,
                "Attack_1": "154",
                "Attack_2": "116",
                "Attack_3": "69",
                "Attack_4": "238",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Vital Spirit"
            },
            {
                "name": "Primeape",
                "Level": 31,
                "Attack_1": "154",
                "Attack_2": "116",
                "Attack_3": "69",
                "Attack_4": "99",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Vital Spirit"
            }
        ],
        "Black Belt Aaron": [
            {
                "name": "Primeape",
                "Level": 36,
                "Attack_1": "116",
                "Attack_2": "69",
                "Attack_3": "99",
                "Attack_4": "238",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Vital Spirit"
            }
        ],
        "Black Belt Koichi": [
            {
                "name": "Hitmonchan",
                "Level": 37,
                "Attack_1": "9",
                "Attack_2": "8",
                "Attack_3": "7",
                "Attack_4": "327",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Keen Eye"
            },
            {
                "name": "Hitmonlee",
                "Level": 37,
                "Attack_1": "116",
                "Attack_2": "136",
                "Attack_3": "170",
                "Attack_4": "193",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Limber"
            }
        ],
        "Team Rocket Grunt 22": [
            {
                "name": "Golbat",
                "Level": 25,
                "Attack_1": "310",
                "Attack_2": "48",
                "Attack_3": "44",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Inner Focus"
            },
            {
                "name": "Zubat",
                "Level": 25,
                "Attack_1": "310",
                "Attack_2": "48",
                "Attack_3": "44",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Inner Focus"
            },
            {
                "name": "Zubat",
                "Level": 25,
                "Attack_1": "310",
                "Attack_2": "48",
                "Attack_3": "44",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Inner Focus"
            },
            {
                "name": "Raticate",
                "Level": 25,
                "Attack_1": "39",
                "Attack_2": "98",
                "Attack_3": "158",
                "Attack_4": "184",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Run Away"
            },
            {
                "name": "Zubat",
                "Level": 25,
                "Attack_1": "310",
                "Attack_2": "48",
                "Attack_3": "44",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Inner Focus"
            }
        ],
        "Scientist Jerry": [
            {
                "name": "Magnemite",
                "Level": 28,
                "Attack_1": "48",
                "Attack_2": "49",
                "Attack_3": "86",
                "Attack_4": "209",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Magnet Pull"
            },
            {
                "name": "Voltorb",
                "Level": 28,
                "Attack_1": "103",
                "Attack_2": "49",
                "Attack_3": "209",
                "Attack_4": "120",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Soundproof"
            },
            {
                "name": "Magneton",
                "Level": 28,
                "Attack_1": "48",
                "Attack_2": "49",
                "Attack_3": "86",
                "Attack_4": "209",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Magnet Pull"
            }
        ],
        "Team Rocket Grunt 23": [
            {
                "name": "Cubone",
                "Level": 29,
                "Attack_1": "43",
                "Attack_2": "116",
                "Attack_3": "155",
                "Attack_4": "99",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Rock Head"
            },
            {
                "name": "Zubat",
                "Level": 29,
                "Attack_1": "48",
                "Attack_2": "44",
                "Attack_3": "17",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Inner Focus"
            }
        ],
        "Scientist Connor": [
            {
                "name": "Grimer",
                "Level": 26,
                "Attack_1": "103",
                "Attack_2": "107",
                "Attack_3": "124",
                "Attack_4": "50",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Stench"
            },
            {
                "name": "Weezing",
                "Level": 26,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Levitate"
            },
            {
                "name": "Koffing",
                "Level": 26,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "120",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Levitate"
            },
            {
                "name": "Weezing",
                "Level": 26,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Levitate"
            }
        ],
        "Team Rocket Grunt 24": [
            {
                "name": "Raticate",
                "Level": 28,
                "Attack_1": "39",
                "Attack_2": "98",
                "Attack_3": "158",
                "Attack_4": "184",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Run Away"
            },
            {
                "name": "Hypno",
                "Level": 28,
                "Attack_1": "50",
                "Attack_2": "93",
                "Attack_3": "29",
                "Attack_4": "139",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Insomnia"
            },
            {
                "name": "Raticate",
                "Level": 28,
                "Attack_1": "39",
                "Attack_2": "98",
                "Attack_3": "158",
                "Attack_4": "184",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Run Away"
            }
        ],
        "Scientist Jose": [
            {
                "name": "Electrode",
                "Level": 29,
                "Attack_1": "209",
                "Attack_2": "49",
                "Attack_3": "103",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Soundproof"
            },
            {
                "name": "Weezing",
                "Level": 29,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Levitate"
            }
        ],
        "Team Rocket Grunt 25": [
            {
                "name": "Ekans",
                "Level": 28,
                "Attack_1": "40",
                "Attack_2": "44",
                "Attack_3": "137",
                "Attack_4": "103",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Intimidate"
            },
            {
                "name": "Zubat",
                "Level": 28,
                "Attack_1": "48",
                "Attack_2": "44",
                "Attack_3": "17",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Inner Focus"
            },
            {
                "name": "Cubone",
                "Level": 28,
                "Attack_1": "29",
                "Attack_2": "43",
                "Attack_3": "116",
                "Attack_4": "155",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Rock Head"
            }
        ],
        "Team Rocket Grunt 26": [
            {
                "name": "Machop",
                "Level": 29,
                "Attack_1": "2",
                "Attack_2": "69",
                "Attack_3": "193",
                "Attack_4": "279",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Guts"
            },
            {
                "name": "Drowzee",
                "Level": 29,
                "Attack_1": "93",
                "Attack_2": "29",
                "Attack_3": "139",
                "Attack_4": "96",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Insomnia"
            }
        ],
        "Scientist Rodney": [
            {
                "name": "Electrode",
                "Level": 33,
                "Attack_1": "209",
                "Attack_2": "49",
                "Attack_3": "103",
                "Attack_4": "120",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Soundproof"
            }
        ],
        "Team Rocket Grunt 27": [
            {
                "name": "Hypno",
                "Level": 33,
                "Attack_1": "93",
                "Attack_2": "29",
                "Attack_3": "139",
                "Attack_4": "96",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Insomnia"
            }
        ],
        "Juggler Dalton": [
            {
                "name": "Kadabra",
                "Level": 29,
                "Attack_1": "50",
                "Attack_2": "60",
                "Attack_3": "115",
                "Attack_4": "105",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Synchronize"
            },
            {
                "name": "Mr. Mime",
                "Level": 29,
                "Attack_1": "115",
                "Attack_2": "345",
                "Attack_3": "227",
                "Attack_4": "60",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Soundproof"
            }
        ],
        "Scientist Beau": [
            {
                "name": "Magneton",
                "Level": 26,
                "Attack_1": "209",
                "Attack_2": "86",
                "Attack_3": "49",
                "Attack_4": "48",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Magnet Pull"
            },
            {
                "name": "Koffing",
                "Level": 26,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Levitate"
            },
            {
                "name": "Weezing",
                "Level": 26,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Levitate"
            },
            {
                "name": "Magnemite",
                "Level": 26,
                "Attack_1": "209",
                "Attack_2": "86",
                "Attack_3": "49",
                "Attack_4": "48",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Magnet Pull"
            }
        ],
        "Team Rocket Grunt 28": [
            {
                "name": "Arbok",
                "Level": 33,
                "Attack_1": "40",
                "Attack_2": "44",
                "Attack_3": "137",
                "Attack_4": "103",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Intimidate"
            }
        ],
        "Team Rocket Grunt 29": [
            {
                "name": "Machop",
                "Level": 29,
                "Attack_1": "2",
                "Attack_2": "69",
                "Attack_3": "193",
                "Attack_4": "279",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Guts"
            },
            {
                "name": "Machoke",
                "Level": 29,
                "Attack_1": "2",
                "Attack_2": "69",
                "Attack_3": "193",
                "Attack_4": "279",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Guts"
            }
        ],
        "Team Rocket Grunt 30": [
            {
                "name": "Zubat",
                "Level": 28,
                "Attack_1": "48",
                "Attack_2": "44",
                "Attack_3": "17",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Inner Focus"
            },
            {
                "name": "Zubat",
                "Level": 28,
                "Attack_1": "48",
                "Attack_2": "44",
                "Attack_3": "17",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Inner Focus"
            },
            {
                "name": "Golbat",
                "Level": 28,
                "Attack_1": "48",
                "Attack_2": "44",
                "Attack_3": "17",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Inner Focus"
            }
        ],
        "Scientist Taylor": [
            {
                "name": "Voltorb",
                "Level": 25,
                "Attack_1": "209",
                "Attack_2": "49",
                "Attack_3": "103",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Soundproof"
            },
            {
                "name": "Koffing",
                "Level": 25,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Levitate"
            },
            {
                "name": "Magneton",
                "Level": 25,
                "Attack_1": "86",
                "Attack_2": "49",
                "Attack_3": "48",
                "Attack_4": "84",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Magnet Pull"
            },
            {
                "name": "Magnemite",
                "Level": 25,
                "Attack_1": "86",
                "Attack_2": "49",
                "Attack_3": "48",
                "Attack_4": "84",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Magnet Pull"
            },
            {
                "name": "Koffing",
                "Level": 25,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "120",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Levitate"
            }
        ],
        "Team Rocket Grunt 31": [
            {
                "name": "Cubone",
                "Level": 29,
                "Attack_1": "43",
                "Attack_2": "116",
                "Attack_3": "155",
                "Attack_4": "99",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Rock Head"
            },
            {
                "name": "Cubone",
                "Level": 29,
                "Attack_1": "43",
                "Attack_2": "116",
                "Attack_3": "155",
                "Attack_4": "99",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Rock Head"
            }
        ],
        "Team Rocket Grunt 32": [
            {
                "name": "Raticate",
                "Level": 26,
                "Attack_1": "39",
                "Attack_2": "98",
                "Attack_3": "158",
                "Attack_4": "184",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Run Away"
            },
            {
                "name": "Zubat",
                "Level": 26,
                "Attack_1": "48",
                "Attack_2": "44",
                "Attack_3": "17",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Inner Focus"
            },
            {
                "name": "Golbat",
                "Level": 26,
                "Attack_1": "310",
                "Attack_2": "48",
                "Attack_3": "44",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Inner Focus"
            },
            {
                "name": "Rattata",
                "Level": 26,
                "Attack_1": "39",
                "Attack_2": "98",
                "Attack_3": "158",
                "Attack_4": "184",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Run Away"
            }
        ],
        "Scientist Joshua": [
            {
                "name": "Electrode",
                "Level": 29,
                "Attack_1": "103",
                "Attack_2": "49",
                "Attack_3": "209",
                "Attack_4": "120",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Soundproof"
            },
            {
                "name": "Muk",
                "Level": 29,
                "Attack_1": "50",
                "Attack_2": "124",
                "Attack_3": "107",
                "Attack_4": "103",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Stench"
            }
        ],
        "Team Rocket Grunt 33": [
            {
                "name": "Sandshrew",
                "Level": 29,
                "Attack_1": "111",
                "Attack_2": "28",
                "Attack_3": "40",
                "Attack_4": "163",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Sand Veil"
            },
            {
                "name": "Sandslash",
                "Level": 29,
                "Attack_1": "111",
                "Attack_2": "28",
                "Attack_3": "40",
                "Attack_4": "163",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Sand Veil"
            }
        ],
        "Silph Rival(Bulbasaur)": [
            {
                "name": "Pidgeot",
                "Level": 37,
                "Attack_1": "98",
                "Attack_2": "18",
                "Attack_3": "17",
                "Attack_4": "297",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Keen Eye"
            },
            {
                "name": "Alakazam",
                "Level": 35,
                "Attack_1": "115",
                "Attack_2": "105",
                "Attack_3": "248",
                "Attack_4": "347",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Synchronize"
            },
            {
                "name": "Exeggcute",
                "Level": 38,
                "Attack_1": "93",
                "Attack_2": "78",
                "Attack_3": "77",
                "Attack_4": "79",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Gyarados",
                "Level": 35,
                "Attack_1": "44",
                "Attack_2": "82",
                "Attack_3": "43",
                "Attack_4": "239",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Intimidate"
            },
            {
                "name": "Charizard",
                "Level": 40,
                "Attack_1": "108",
                "Attack_2": "184",
                "Attack_3": "53",
                "Attack_4": "17",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Blaze"
            }
        ],
        "Silph Rival(Squirtle)": [
            {
                "name": "Pidgeot",
                "Level": 37,
                "Attack_1": "98",
                "Attack_2": "18",
                "Attack_3": "17",
                "Attack_4": "297",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Keen Eye"
            },
            {
                "name": "Alakazam",
                "Level": 35,
                "Attack_1": "115",
                "Attack_2": "105",
                "Attack_3": "248",
                "Attack_4": "347",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Synchronize"
            },
            {
                "name": "Growlithe",
                "Level": 35,
                "Attack_1": "316",
                "Attack_2": "36",
                "Attack_3": "172",
                "Attack_4": "43",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Intimidate"
            },
            {
                "name": "Gyarados",
                "Level": 38,
                "Attack_1": "44",
                "Attack_2": "43",
                "Attack_3": "239",
                "Attack_4": "82",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Intimidate"
            },
            {
                "name": "Venusaur",
                "Level": 40,
                "Attack_1": "77",
                "Attack_2": "79",
                "Attack_3": "75",
                "Attack_4": "230",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Overgrow"
            }
        ],
        "Silph Rival(Charmander)": [
            {
                "name": "Pidgeot",
                "Level": 37,
                "Attack_1": "98",
                "Attack_2": "18",
                "Attack_3": "17",
                "Attack_4": "297",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Keen Eye"
            },
            {
                "name": "Alakazam",
                "Level": 35,
                "Attack_1": "115",
                "Attack_2": "105",
                "Attack_3": "248",
                "Attack_4": "347",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Synchronize"
            },
            {
                "name": "Growlithe",
                "Level": 38,
                "Attack_1": "316",
                "Attack_2": "36",
                "Attack_3": "172",
                "Attack_4": "270",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Intimidate"
            },
            {
                "name": "Exeggcute",
                "Level": 35,
                "Attack_1": "73",
                "Attack_2": "93",
                "Attack_3": "78",
                "Attack_4": "77",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Blastoise",
                "Level": 40,
                "Attack_1": "55",
                "Attack_2": "44",
                "Attack_3": "229",
                "Attack_4": "182",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Torrent"
            }
        ],
        "Team Rocket Grunt 34": [
            {
                "name": "Raticate",
                "Level": 26,
                "Attack_1": "184",
                "Attack_2": "158",
                "Attack_3": "98",
                "Attack_4": "39",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Run Away"
            },
            {
                "name": "Arbok",
                "Level": 26,
                "Attack_1": "137",
                "Attack_2": "44",
                "Attack_3": "40",
                "Attack_4": "43",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Intimidate"
            },
            {
                "name": "Koffing",
                "Level": 26,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Levitate"
            },
            {
                "name": "Golbat",
                "Level": 26,
                "Attack_1": "17",
                "Attack_2": "44",
                "Attack_3": "310",
                "Attack_4": "48",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Inner Focus"
            }
        ],
        "Team Rocket Grunt 35": [
            {
                "name": "Weezing",
                "Level": 28,
                "Attack_1": "123",
                "Attack_2": "120",
                "Attack_3": "124",
                "Attack_4": "108",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Levitate"
            },
            {
                "name": "Golbat",
                "Level": 28,
                "Attack_1": "48",
                "Attack_2": "44",
                "Attack_3": "17",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Inner Focus"
            },
            {
                "name": "Koffing",
                "Level": 28,
                "Attack_1": "123",
                "Attack_2": "120",
                "Attack_3": "124",
                "Attack_4": "108",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Levitate"
            }
        ],
        "Scientist Parker": [
            {
                "name": "Grimer",
                "Level": 29,
                "Attack_1": "50",
                "Attack_2": "124",
                "Attack_3": "107",
                "Attack_4": "103",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Stench"
            },
            {
                "name": "Electrode",
                "Level": 29,
                "Attack_1": "103",
                "Attack_2": "49",
                "Attack_3": "209",
                "Attack_4": "120",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Soundproof"
            }
        ],
        "Team Rocket Grunt 36": [
            {
                "name": "Golbat",
                "Level": 28,
                "Attack_1": "48",
                "Attack_2": "44",
                "Attack_3": "17",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Inner Focus"
            },
            {
                "name": "Drowzee",
                "Level": 28,
                "Attack_1": "93",
                "Attack_2": "29",
                "Attack_3": "139",
                "Attack_4": "96",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Insomnia"
            },
            {
                "name": "Hypno",
                "Level": 28,
                "Attack_1": "50",
                "Attack_2": "93",
                "Attack_3": "29",
                "Attack_4": "139",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Insomnia"
            }
        ],
        "Team Rocket Grunt 37": [
            {
                "name": "Drowzee",
                "Level": 28,
                "Attack_1": "93",
                "Attack_2": "29",
                "Attack_3": "139",
                "Attack_4": "96",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Insomnia"
            },
            {
                "name": "Grimer",
                "Level": 28,
                "Attack_1": "50",
                "Attack_2": "124",
                "Attack_3": "107",
                "Attack_4": "103",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Stench"
            },
            {
                "name": "Machop",
                "Level": 28,
                "Attack_1": "2",
                "Attack_2": "69",
                "Attack_3": "193",
                "Attack_4": "279",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Guts"
            }
        ],
        "Scientist Ed": [
            {
                "name": "Voltorb",
                "Level": 28,
                "Attack_1": "209",
                "Attack_2": "49",
                "Attack_3": "103",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Soundproof"
            },
            {
                "name": "Koffing",
                "Level": 28,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "33",
                "Attack_4": "123",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Levitate"
            },
            {
                "name": "Magneton",
                "Level": 28,
                "Attack_1": "209",
                "Attack_2": "86",
                "Attack_3": "49",
                "Attack_4": "48",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Magnet Pull"
            }
        ],
        "Team Rocket Grunt 38": [
            {
                "name": "Machoke",
                "Level": 33,
                "Attack_1": "69",
                "Attack_2": "193",
                "Attack_3": "279",
                "Attack_4": "233",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Guts"
            }
        ],
        "Scientist Travis": [
            {
                "name": "Magnemite",
                "Level": 29,
                "Attack_1": "48",
                "Attack_2": "49",
                "Attack_3": "86",
                "Attack_4": "209",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Magnet Pull"
            },
            {
                "name": "Koffing",
                "Level": 29,
                "Attack_1": "123",
                "Attack_2": "120",
                "Attack_3": "124",
                "Attack_4": "108",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Levitate"
            }
        ],
        "Team Rocket Grunt 39": [
            {
                "name": "Rattata",
                "Level": 25,
                "Attack_1": "39",
                "Attack_2": "98",
                "Attack_3": "158",
                "Attack_4": "116",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Run Away"
            },
            {
                "name": "Rattata",
                "Level": 25,
                "Attack_1": "39",
                "Attack_2": "98",
                "Attack_3": "158",
                "Attack_4": "116",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Run Away"
            },
            {
                "name": "Zubat",
                "Level": 25,
                "Attack_1": "310",
                "Attack_2": "48",
                "Attack_3": "44",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Inner Focus"
            },
            {
                "name": "Rattata",
                "Level": 25,
                "Attack_1": "39",
                "Attack_2": "98",
                "Attack_3": "158",
                "Attack_4": "116",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Run Away"
            },
            {
                "name": "Ekans",
                "Level": 25,
                "Attack_1": "40",
                "Attack_2": "44",
                "Attack_3": "137",
                "Attack_4": "103",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Intimidate"
            }
        ],
        "Team Rocket Grunt 40": [
            {
                "name": "Cubone",
                "Level": 32,
                "Attack_1": "43",
                "Attack_2": "116",
                "Attack_3": "155",
                "Attack_4": "99",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Rock Head"
            },
            {
                "name": "Drowzee",
                "Level": 32,
                "Attack_1": "29",
                "Attack_2": "139",
                "Attack_3": "96",
                "Attack_4": "94",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Insomnia"
            },
            {
                "name": "Marowak",
                "Level": 32,
                "Attack_1": "43",
                "Attack_2": "116",
                "Attack_3": "155",
                "Attack_4": "99",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Rock Head"
            }
        ],
        "Giovanni 2": [
            {
                "name": "Nidorino",
                "Level": 37,
                "Attack_1": "40",
                "Attack_2": "30",
                "Attack_3": "270",
                "Attack_4": "31",
                "IVS": {
                    "hp": 30,
                    "atk": 30,
                    "def": 30,
                    "spa": 30,
                    "spd": 30,
                    "spe": 30
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Poison Point"
            },
            {
                "name": "Rhyhorn",
                "Level": 37,
                "Attack_1": "23",
                "Attack_2": "31",
                "Attack_3": "184",
                "Attack_4": "350",
                "IVS": {
                    "hp": 30,
                    "atk": 30,
                    "def": 30,
                    "spa": 30,
                    "spd": 30,
                    "spe": 30
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Lightningrod"
            },
            {
                "name": "Kangaskhan",
                "Level": 35,
                "Attack_1": "39",
                "Attack_2": "252",
                "Attack_3": "5",
                "Attack_4": "99",
                "IVS": {
                    "hp": 30,
                    "atk": 30,
                    "def": 30,
                    "spa": 30,
                    "spd": 30,
                    "spe": 30
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Early Bird"
            },
            {
                "name": "Nidoqueen",
                "Level": 41,
                "Attack_1": "39",
                "Attack_2": "24",
                "Attack_3": "40",
                "Attack_4": "34",
                "IVS": {
                    "hp": 30,
                    "atk": 30,
                    "def": 30,
                    "spa": 30,
                    "spd": 30,
                    "spe": 30
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Poison Point"
            }
        ],
        "Psychic Cameron": [
            {
                "name": "Slowpoke",
                "Level": 33,
                "Attack_1": "55",
                "Attack_2": "93",
                "Attack_3": "50",
                "Attack_4": "29",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Oblivious"
            },
            {
                "name": "Slowpoke",
                "Level": 33,
                "Attack_1": "55",
                "Attack_2": "93",
                "Attack_3": "50",
                "Attack_4": "29",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Oblivious"
            },
            {
                "name": "Slowbro",
                "Level": 33,
                "Attack_1": "55",
                "Attack_2": "93",
                "Attack_3": "50",
                "Attack_4": "29",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Oblivious"
            }
        ],
        "Psychic Tyron": [
            {
                "name": "Mr. Mime",
                "Level": 34,
                "Attack_1": "345",
                "Attack_2": "227",
                "Attack_3": "60",
                "Attack_4": "278",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Soundproof"
            },
            {
                "name": "Kadabra",
                "Level": 34,
                "Attack_1": "115",
                "Attack_2": "105",
                "Attack_3": "248",
                "Attack_4": "272",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Synchronize"
            }
        ],
        "Channeler Stacy": [
            {
                "name": "Haunter",
                "Level": 38,
                "Attack_1": "101",
                "Attack_2": "109",
                "Attack_3": "325",
                "Attack_4": "138",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Levitate"
            }
        ],
        "Psychic Preston": [
            {
                "name": "Slowbro",
                "Level": 38,
                "Attack_1": "50",
                "Attack_2": "29",
                "Attack_3": "133",
                "Attack_4": "110",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Oblivious"
            }
        ],
        "Channeler Amanda": [
            {
                "name": "Gastly",
                "Level": 34,
                "Attack_1": "101",
                "Attack_2": "109",
                "Attack_3": "138",
                "Attack_4": "194",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Levitate"
            },
            {
                "name": "Haunter",
                "Level": 34,
                "Attack_1": "101",
                "Attack_2": "109",
                "Attack_3": "325",
                "Attack_4": "138",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Levitate"
            }
        ],
        "Channeler Tasha": [
            {
                "name": "Gastly",
                "Level": 33,
                "Attack_1": "101",
                "Attack_2": "109",
                "Attack_3": "138",
                "Attack_4": "194",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Levitate"
            },
            {
                "name": "Gastly",
                "Level": 33,
                "Attack_1": "101",
                "Attack_2": "109",
                "Attack_3": "138",
                "Attack_4": "194",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Levitate"
            },
            {
                "name": "Haunter",
                "Level": 33,
                "Attack_1": "101",
                "Attack_2": "109",
                "Attack_3": "325",
                "Attack_4": "138",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Levitate"
            }
        ],
        "Psychic Johan": [
            {
                "name": "Kadabra",
                "Level": 31,
                "Attack_1": "60",
                "Attack_2": "115",
                "Attack_3": "105",
                "Attack_4": "248",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Synchronize"
            },
            {
                "name": "Mr. Mime",
                "Level": 31,
                "Attack_1": "115",
                "Attack_2": "345",
                "Attack_3": "227",
                "Attack_4": "60",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Soundproof"
            },
            {
                "name": "Slowpoke",
                "Level": 31,
                "Attack_1": "55",
                "Attack_2": "93",
                "Attack_3": "50",
                "Attack_4": "29",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Oblivious"
            },
            {
                "name": "Kadabra",
                "Level": 31,
                "Attack_1": "60",
                "Attack_2": "115",
                "Attack_3": "105",
                "Attack_4": "248",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Synchronize"
            }
        ],
        "Leader Sabrina": [
            {
                "name": "Kadabra",
                "Level": 38,
                "Attack_1": "60",
                "Attack_2": "115",
                "Attack_3": "248",
                "Attack_4": "347",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Synchronize"
            },
            {
                "name": "Mr. Mime",
                "Level": 37,
                "Attack_1": "112",
                "Attack_2": "60",
                "Attack_3": "226",
                "Attack_4": "347",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Soundproof"
            },
            {
                "name": "Venomoth",
                "Level": 38,
                "Attack_1": "60",
                "Attack_2": "16",
                "Attack_3": "141",
                "Attack_4": "48",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Shield Dust"
            },
            {
                "name": "Alakazam",
                "Level": 43,
                "Attack_1": "94",
                "Attack_2": "105",
                "Attack_3": "248",
                "Attack_4": "347",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Synchronize"
            }
        ],
        "Fisherman Ned": [
            {
                "name": "Goldeen",
                "Level": 22,
                "Attack_1": "39",
                "Attack_2": "346",
                "Attack_3": "48",
                "Attack_4": "30",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Swift Swim"
            },
            {
                "name": "Poliwag",
                "Level": 22,
                "Attack_1": "145",
                "Attack_2": "95",
                "Attack_3": "55",
                "Attack_4": "3",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Water Absorb"
            },
            {
                "name": "Goldeen",
                "Level": 22,
                "Attack_1": "39",
                "Attack_2": "346",
                "Attack_3": "48",
                "Attack_4": "30",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Swift Swim"
            }
        ],
        "Fisherman Chip": [
            {
                "name": "Tentacool",
                "Level": 24,
                "Attack_1": "40",
                "Attack_2": "48",
                "Attack_3": "132",
                "Attack_4": "51",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Clear Body"
            },
            {
                "name": "Goldeen",
                "Level": 24,
                "Attack_1": "346",
                "Attack_2": "48",
                "Attack_3": "30",
                "Attack_4": "175",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Swift Swim"
            }
        ],
        "Fisherman Hank": [
            {
                "name": "Goldeen",
                "Level": 27,
                "Attack_1": "346",
                "Attack_2": "48",
                "Attack_3": "30",
                "Attack_4": "175",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Swift Swim"
            }
        ],
        "Fisherman Elliot": [
            {
                "name": "Poliwag",
                "Level": 21,
                "Attack_1": "145",
                "Attack_2": "95",
                "Attack_3": "55",
                "Attack_4": "3",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Water Absorb"
            },
            {
                "name": "Shellder",
                "Level": 21,
                "Attack_1": "33",
                "Attack_2": "110",
                "Attack_3": "333",
                "Attack_4": "48",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Shell Armor"
            },
            {
                "name": "Goldeen",
                "Level": 21,
                "Attack_1": "39",
                "Attack_2": "346",
                "Attack_3": "48",
                "Attack_4": "30",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Swift Swim"
            },
            {
                "name": "Horsea",
                "Level": 21,
                "Attack_1": "145",
                "Attack_2": "108",
                "Attack_3": "43",
                "Attack_4": "55",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Swift Swim"
            }
        ],
        "Young Couple Gia & Ges": [
            {
                "name": "Nidoran\u2642",
                "Level": 24,
                "Attack_1": "24",
                "Attack_2": "40",
                "Attack_3": "30",
                "Attack_4": "270",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Poison Point"
            },
            {
                "name": "Nidoran\u2640",
                "Level": 24,
                "Attack_1": "24",
                "Attack_2": "40",
                "Attack_3": "44",
                "Attack_4": "270",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Poison Point"
            }
        ],
        "Rocker Luca": [
            {
                "name": "Voltorb",
                "Level": 29,
                "Attack_1": "103",
                "Attack_2": "49",
                "Attack_3": "209",
                "Attack_4": "120",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Soundproof"
            },
            {
                "name": "Electrode",
                "Level": 29,
                "Attack_1": "103",
                "Attack_2": "49",
                "Attack_3": "209",
                "Attack_4": "120",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Soundproof"
            }
        ],
        "Camper Justin": [
            {
                "name": "Nidoran\u2642",
                "Level": 29,
                "Attack_1": "24",
                "Attack_2": "40",
                "Attack_3": "30",
                "Attack_4": "270",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Poison Point"
            },
            {
                "name": "Nidorino",
                "Level": 29,
                "Attack_1": "24",
                "Attack_2": "40",
                "Attack_3": "30",
                "Attack_4": "270",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Poison Point"
            }
        ],
        "Fisherman Andrew": [
            {
                "name": "Magikarp",
                "Level": 24,
                "Attack_1": "150",
                "Attack_2": "33",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Swift Swim"
            },
            {
                "name": "Magikarp",
                "Level": 24,
                "Attack_1": "150",
                "Attack_2": "33",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Swift Swim"
            }
        ],
        "Picknicker Alma": [
            {
                "name": "Goldeen",
                "Level": 28,
                "Attack_1": "346",
                "Attack_2": "48",
                "Attack_3": "30",
                "Attack_4": "175",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Swift Swim"
            },
            {
                "name": "Poliwag",
                "Level": 28,
                "Attack_1": "95",
                "Attack_2": "55",
                "Attack_3": "3",
                "Attack_4": "240",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Water Absorb"
            },
            {
                "name": "Horsea",
                "Level": 28,
                "Attack_1": "145",
                "Attack_2": "108",
                "Attack_3": "43",
                "Attack_4": "55",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Swift Swim"
            }
        ],
        "Bird Keeper Sebastian": [
            {
                "name": "Pidgey",
                "Level": 29,
                "Attack_1": "16",
                "Attack_2": "98",
                "Attack_3": "18",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Keen Eye"
            },
            {
                "name": "Pidgeotto",
                "Level": 29,
                "Attack_1": "16",
                "Attack_2": "98",
                "Attack_3": "18",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Keen Eye"
            }
        ],
        "Picknicker Susie": [
            {
                "name": "Pidgey",
                "Level": 24,
                "Attack_1": "28",
                "Attack_2": "16",
                "Attack_3": "98",
                "Attack_4": "18",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Keen Eye"
            },
            {
                "name": "Meowth",
                "Level": 24,
                "Attack_1": "10",
                "Attack_2": "45",
                "Attack_3": "44",
                "Attack_4": "6",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Pickup"
            },
            {
                "name": "Rattata",
                "Level": 24,
                "Attack_1": "39",
                "Attack_2": "98",
                "Attack_3": "158",
                "Attack_4": "116",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Run Away"
            },
            {
                "name": "Pikachu",
                "Level": 24,
                "Attack_1": "86",
                "Attack_2": "98",
                "Attack_3": "104",
                "Attack_4": "21",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Static"
            },
            {
                "name": "Meowth",
                "Level": 24,
                "Attack_1": "10",
                "Attack_2": "45",
                "Attack_3": "44",
                "Attack_4": "6",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Pickup"
            }
        ],
        "Beauty Lola": [
            {
                "name": "Rattata",
                "Level": 27,
                "Attack_1": "98",
                "Attack_2": "158",
                "Attack_3": "116",
                "Attack_4": "228",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Run Away"
            },
            {
                "name": "Pikachu",
                "Level": 27,
                "Attack_1": "98",
                "Attack_2": "104",
                "Attack_3": "21",
                "Attack_4": "85",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Static"
            },
            {
                "name": "Rattata",
                "Level": 27,
                "Attack_1": "98",
                "Attack_2": "158",
                "Attack_3": "116",
                "Attack_4": "228",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Run Away"
            }
        ],
        "Beauty Sheila": [
            {
                "name": "Clefairy",
                "Level": 29,
                "Attack_1": "266",
                "Attack_2": "107",
                "Attack_3": "111",
                "Attack_4": "118",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Cute Charm"
            },
            {
                "name": "Meowth",
                "Level": 29,
                "Attack_1": "45",
                "Attack_2": "44",
                "Attack_3": "6",
                "Attack_4": "185",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Pickup"
            }
        ],
        "Picknicker Valeri": [
            {
                "name": "Poliwag",
                "Level": 30,
                "Attack_1": "95",
                "Attack_2": "55",
                "Attack_3": "3",
                "Attack_4": "240",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Water Absorb"
            },
            {
                "name": "Poliwag",
                "Level": 30,
                "Attack_1": "95",
                "Attack_2": "55",
                "Attack_3": "3",
                "Attack_4": "240",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Water Absorb"
            }
        ],
        "Picknicker Gwen": [
            {
                "name": "Pidgey",
                "Level": 27,
                "Attack_1": "16",
                "Attack_2": "98",
                "Attack_3": "18",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Keen Eye"
            },
            {
                "name": "Meowth",
                "Level": 27,
                "Attack_1": "45",
                "Attack_2": "44",
                "Attack_3": "6",
                "Attack_4": "185",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Pickup"
            },
            {
                "name": "Pidgey",
                "Level": 27,
                "Attack_1": "16",
                "Attack_2": "98",
                "Attack_3": "18",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Keen Eye"
            },
            {
                "name": "Pidgeotto",
                "Level": 27,
                "Attack_1": "16",
                "Attack_2": "98",
                "Attack_3": "18",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Keen Eye"
            }
        ],
        "Bird Keeper Robert": [
            {
                "name": "Pidgey",
                "Level": 26,
                "Attack_1": "16",
                "Attack_2": "98",
                "Attack_3": "18",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Keen Eye"
            },
            {
                "name": "Pidgeotto",
                "Level": 26,
                "Attack_1": "28",
                "Attack_2": "16",
                "Attack_3": "98",
                "Attack_4": "18",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Keen Eye"
            },
            {
                "name": "Spearow",
                "Level": 26,
                "Attack_1": "43",
                "Attack_2": "31",
                "Attack_3": "228",
                "Attack_4": "332",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Keen Eye"
            },
            {
                "name": "Fearow",
                "Level": 26,
                "Attack_1": "45",
                "Attack_2": "43",
                "Attack_3": "31",
                "Attack_4": "228",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Keen Eye"
            }
        ],
        "Bird Keeper Perry": [
            {
                "name": "Spearow",
                "Level": 25,
                "Attack_1": "43",
                "Attack_2": "31",
                "Attack_3": "228",
                "Attack_4": "332",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Keen Eye"
            },
            {
                "name": "Pidgey",
                "Level": 25,
                "Attack_1": "16",
                "Attack_2": "98",
                "Attack_3": "18",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Keen Eye"
            },
            {
                "name": "Pidgey",
                "Level": 25,
                "Attack_1": "16",
                "Attack_2": "98",
                "Attack_3": "18",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Keen Eye"
            },
            {
                "name": "Spearow",
                "Level": 25,
                "Attack_1": "43",
                "Attack_2": "31",
                "Attack_3": "228",
                "Attack_4": "332",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Keen Eye"
            },
            {
                "name": "Spearow",
                "Level": 25,
                "Attack_1": "43",
                "Attack_2": "31",
                "Attack_3": "228",
                "Attack_4": "332",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Keen Eye"
            }
        ],
        "Biker Jared": [
            {
                "name": "Koffing",
                "Level": 28,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "139",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Levitate"
            },
            {
                "name": "Koffing",
                "Level": 28,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Levitate"
            },
            {
                "name": "Koffing",
                "Level": 28,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Levitate"
            }
        ],
        "Bird Keeper Carter": [
            {
                "name": "Pidgey",
                "Level": 28,
                "Attack_1": "16",
                "Attack_2": "98",
                "Attack_3": "18",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Keen Eye"
            },
            {
                "name": "Doduo",
                "Level": 28,
                "Attack_1": "228",
                "Attack_2": "31",
                "Attack_3": "161",
                "Attack_4": "99",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Run Away"
            },
            {
                "name": "Pidgeotto",
                "Level": 28,
                "Attack_1": "16",
                "Attack_2": "98",
                "Attack_3": "18",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Keen Eye"
            }
        ],
        "Bird Keeper Mitch": [
            {
                "name": "Pidgey",
                "Level": 26,
                "Attack_1": "16",
                "Attack_2": "98",
                "Attack_3": "18",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Keen Eye"
            },
            {
                "name": "Spearow",
                "Level": 26,
                "Attack_1": "43",
                "Attack_2": "31",
                "Attack_3": "228",
                "Attack_4": "332",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Keen Eye"
            },
            {
                "name": "Pidgey",
                "Level": 26,
                "Attack_1": "16",
                "Attack_2": "98",
                "Attack_3": "18",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Keen Eye"
            },
            {
                "name": "Fearow",
                "Level": 26,
                "Attack_1": "45",
                "Attack_2": "43",
                "Attack_3": "31",
                "Attack_4": "228",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Keen Eye"
            }
        ],
        "Bird Keeper Marlon": [
            {
                "name": "Spearow",
                "Level": 28,
                "Attack_1": "43",
                "Attack_2": "31",
                "Attack_3": "228",
                "Attack_4": "332",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Keen Eye"
            },
            {
                "name": "Doduo",
                "Level": 28,
                "Attack_1": "228",
                "Attack_2": "31",
                "Attack_3": "161",
                "Attack_4": "99",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Run Away"
            },
            {
                "name": "Fearow",
                "Level": 28,
                "Attack_1": "45",
                "Attack_2": "43",
                "Attack_3": "31",
                "Attack_4": "228",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Keen Eye"
            }
        ],
        "Bird Keeper Beck": [
            {
                "name": "Pidgeotto",
                "Level": 29,
                "Attack_1": "16",
                "Attack_2": "98",
                "Attack_3": "18",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Keen Eye"
            },
            {
                "name": "Fearow",
                "Level": 29,
                "Attack_1": "45",
                "Attack_2": "43",
                "Attack_3": "31",
                "Attack_4": "228",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Keen Eye"
            }
        ],
        "Bird Keeper Donald": [
            {
                "name": "Farfetch'd",
                "Level": 33,
                "Attack_1": "31",
                "Attack_2": "282",
                "Attack_3": "210",
                "Attack_4": "14",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Keen Eye"
            }
        ],
        "Bird Keeper Benny": [
            {
                "name": "Spearow",
                "Level": 29,
                "Attack_1": "43",
                "Attack_2": "31",
                "Attack_3": "228",
                "Attack_4": "332",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Keen Eye"
            },
            {
                "name": "Fearow",
                "Level": 29,
                "Attack_1": "45",
                "Attack_2": "43",
                "Attack_3": "31",
                "Attack_4": "228",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Keen Eye"
            }
        ],
        "Twins Kiri & Jan": [
            {
                "name": "Charmander",
                "Level": 29,
                "Attack_1": "52",
                "Attack_2": "232",
                "Attack_3": "108",
                "Attack_4": "184",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Blaze"
            },
            {
                "name": "Squirtle",
                "Level": 29,
                "Attack_1": "55",
                "Attack_2": "44",
                "Attack_3": "229",
                "Attack_4": "182",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Torrent"
            }
        ],
        "Biker Gerald": [
            {
                "name": "Koffing",
                "Level": 29,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Levitate"
            },
            {
                "name": "Muk",
                "Level": 29,
                "Attack_1": "103",
                "Attack_2": "107",
                "Attack_3": "124",
                "Attack_4": "50",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Stench"
            }
        ],
        "Biker Malik": [
            {
                "name": "Koffing",
                "Level": 29,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Levitate"
            },
            {
                "name": "Grimer",
                "Level": 29,
                "Attack_1": "103",
                "Attack_2": "107",
                "Attack_3": "124",
                "Attack_4": "50",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Stench"
            }
        ],
        "Biker Isaac": [
            {
                "name": "Grimer",
                "Level": 28,
                "Attack_1": "103",
                "Attack_2": "107",
                "Attack_3": "124",
                "Attack_4": "50",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Stench"
            },
            {
                "name": "Grimer",
                "Level": 28,
                "Attack_1": "103",
                "Attack_2": "107",
                "Attack_3": "124",
                "Attack_4": "50",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Stench"
            },
            {
                "name": "Koffing",
                "Level": 28,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Levitate"
            }
        ],
        "Biker Lukas": [
            {
                "name": "Koffing",
                "Level": 26,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Levitate"
            },
            {
                "name": "Koffing",
                "Level": 26,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Levitate"
            },
            {
                "name": "Grimer",
                "Level": 26,
                "Attack_1": "103",
                "Attack_2": "107",
                "Attack_3": "124",
                "Attack_4": "50",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Stench"
            },
            {
                "name": "Koffing",
                "Level": 26,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Levitate"
            }
        ],
        "Picknicker Becky": [
            {
                "name": "Pikachu",
                "Level": 29,
                "Attack_1": "98",
                "Attack_2": "104",
                "Attack_3": "21",
                "Attack_4": "85",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Static"
            },
            {
                "name": "Raichu",
                "Level": 29,
                "Attack_1": "84",
                "Attack_2": "39",
                "Attack_3": "98",
                "Attack_4": "85",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Static"
            }
        ],
        "Crush Kin Ron & Mya": [
            {
                "name": "Hitmonlee",
                "Level": 29,
                "Attack_1": "26",
                "Attack_2": "280",
                "Attack_3": "116",
                "Attack_4": "136",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Limber"
            },
            {
                "name": "Hitmonchan",
                "Level": 29,
                "Attack_1": "183",
                "Attack_2": "9",
                "Attack_3": "8",
                "Attack_4": "7",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Keen Eye"
            }
        ],
        "Picknicker Celia": [
            {
                "name": "Clefairy",
                "Level": 33,
                "Attack_1": "107",
                "Attack_2": "111",
                "Attack_3": "118",
                "Attack_4": "322",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Cute Charm"
            }
        ],
        "Biker Ernest": [
            {
                "name": "Koffing",
                "Level": 25,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Levitate"
            },
            {
                "name": "Koffing",
                "Level": 25,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Levitate"
            },
            {
                "name": "Weezing",
                "Level": 25,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Levitate"
            },
            {
                "name": "Koffing",
                "Level": 25,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Levitate"
            },
            {
                "name": "Grimer",
                "Level": 25,
                "Attack_1": "107",
                "Attack_2": "124",
                "Attack_3": "50",
                "Attack_4": "1",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Stench"
            }
        ],
        "Biker Alex": [
            {
                "name": "Koffing",
                "Level": 28,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Levitate"
            },
            {
                "name": "Grimer",
                "Level": 28,
                "Attack_1": "103",
                "Attack_2": "107",
                "Attack_3": "124",
                "Attack_4": "50",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Stench"
            },
            {
                "name": "Weezing",
                "Level": 28,
                "Attack_1": "108",
                "Attack_2": "124",
                "Attack_3": "123",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Levitate"
            }
        ],
        "Beauty Grace": [
            {
                "name": "Pidgeotto",
                "Level": 29,
                "Attack_1": "16",
                "Attack_2": "98",
                "Attack_3": "18",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Keen Eye"
            },
            {
                "name": "Wigglytuff",
                "Level": 29,
                "Attack_1": "47",
                "Attack_2": "50",
                "Attack_3": "111",
                "Attack_4": "3",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Cute Charm"
            }
        ],
        "Beauty Olivia": [
            {
                "name": "Bulbasaur",
                "Level": 29,
                "Attack_1": "77",
                "Attack_2": "79",
                "Attack_3": "75",
                "Attack_4": "230",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Overgrow"
            },
            {
                "name": "Ivysaur",
                "Level": 29,
                "Attack_1": "77",
                "Attack_2": "79",
                "Attack_3": "75",
                "Attack_4": "230",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Overgrow"
            }
        ],
        "Picknicker Kindra": [
            {
                "name": "Gloom",
                "Level": 28,
                "Attack_1": "77",
                "Attack_2": "78",
                "Attack_3": "79",
                "Attack_4": "51",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Oddish",
                "Level": 28,
                "Attack_1": "77",
                "Attack_2": "78",
                "Attack_3": "79",
                "Attack_4": "51",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Oddish",
                "Level": 28,
                "Attack_1": "77",
                "Attack_2": "78",
                "Attack_3": "79",
                "Attack_4": "51",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Chlorophyll"
            }
        ],
        "Bird Keeper Chester": [
            {
                "name": "Dodrio",
                "Level": 28,
                "Attack_1": "228",
                "Attack_2": "31",
                "Attack_3": "161",
                "Attack_4": "99",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Run Away"
            },
            {
                "name": "Doduo",
                "Level": 28,
                "Attack_1": "228",
                "Attack_2": "31",
                "Attack_3": "161",
                "Attack_4": "99",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Run Away"
            },
            {
                "name": "Doduo",
                "Level": 28,
                "Attack_1": "228",
                "Attack_2": "31",
                "Attack_3": "161",
                "Attack_4": "99",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Run Away"
            }
        ],
        "Bird Keeper Edwin": [
            {
                "name": "Pidgeotto",
                "Level": 26,
                "Attack_1": "28",
                "Attack_2": "16",
                "Attack_3": "98",
                "Attack_4": "18",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Keen Eye"
            },
            {
                "name": "Farfetch'd",
                "Level": 26,
                "Attack_1": "43",
                "Attack_2": "31",
                "Attack_3": "282",
                "Attack_4": "210",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Keen Eye"
            },
            {
                "name": "Doduo",
                "Level": 26,
                "Attack_1": "228",
                "Attack_2": "31",
                "Attack_3": "161",
                "Attack_4": "99",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Run Away"
            },
            {
                "name": "Pidgey",
                "Level": 26,
                "Attack_1": "16",
                "Attack_2": "98",
                "Attack_3": "18",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Keen Eye"
            }
        ],
        "Picknicker Yazmin": [
            {
                "name": "Bellsprout",
                "Level": 29,
                "Attack_1": "79",
                "Attack_2": "77",
                "Attack_3": "78",
                "Attack_4": "51",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Oddish",
                "Level": 29,
                "Attack_1": "77",
                "Attack_2": "78",
                "Attack_3": "79",
                "Attack_4": "51",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Tangela",
                "Level": 29,
                "Attack_1": "74",
                "Attack_2": "77",
                "Attack_3": "22",
                "Attack_4": "20",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Chlorophyll"
            }
        ],
        "Swimmer Richard": [
            {
                "name": "Tentacool",
                "Level": 30,
                "Attack_1": "132",
                "Attack_2": "51",
                "Attack_3": "61",
                "Attack_4": "35",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Clear Body"
            },
            {
                "name": "Shellder",
                "Level": 30,
                "Attack_1": "333",
                "Attack_2": "48",
                "Attack_3": "62",
                "Attack_4": "182",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Shell Armor"
            }
        ],
        "Swimmer Reece": [
            {
                "name": "Goldeen",
                "Level": 29,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Swift Swim"
            },
            {
                "name": "Horsea",
                "Level": 29,
                "Attack_1": "108",
                "Attack_2": "43",
                "Attack_3": "55",
                "Attack_4": "239",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Swift Swim"
            },
            {
                "name": "Staryu",
                "Level": 29,
                "Attack_1": "105",
                "Attack_2": "293",
                "Attack_3": "129",
                "Attack_4": "61",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Illuminate"
            }
        ],
        "Swimmer Tony": [
            {
                "name": "Horsea",
                "Level": 30,
                "Attack_1": "108",
                "Attack_2": "43",
                "Attack_3": "55",
                "Attack_4": "239",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Swift Swim"
            },
            {
                "name": "Horsea",
                "Level": 30,
                "Attack_1": "108",
                "Attack_2": "43",
                "Attack_3": "55",
                "Attack_4": "239",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Swift Swim"
            }
        ],
        "Swimmer David": [
            {
                "name": "Goldeen",
                "Level": 29,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Swift Swim"
            },
            {
                "name": "Shellder",
                "Level": 29,
                "Attack_1": "333",
                "Attack_2": "48",
                "Attack_3": "62",
                "Attack_4": "182",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Shell Armor"
            },
            {
                "name": "Seaking",
                "Level": 29,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Swift Swim"
            }
        ],
        "Swimmer Douglas": [
            {
                "name": "Horsea",
                "Level": 27,
                "Attack_1": "145",
                "Attack_2": "108",
                "Attack_3": "43",
                "Attack_4": "55",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Swift Swim"
            },
            {
                "name": "Tentacool",
                "Level": 27,
                "Attack_1": "48",
                "Attack_2": "132",
                "Attack_3": "51",
                "Attack_4": "61",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Clear Body"
            },
            {
                "name": "Tentacool",
                "Level": 27,
                "Attack_1": "48",
                "Attack_2": "132",
                "Attack_3": "51",
                "Attack_4": "61",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Clear Body"
            },
            {
                "name": "Goldeen",
                "Level": 27,
                "Attack_1": "346",
                "Attack_2": "48",
                "Attack_3": "30",
                "Attack_4": "175",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Swift Swim"
            }
        ],
        "Swimmer Matthew": [
            {
                "name": "Poliwag",
                "Level": 30,
                "Attack_1": "95",
                "Attack_2": "55",
                "Attack_3": "3",
                "Attack_4": "240",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Water Absorb"
            },
            {
                "name": "Poliwhirl",
                "Level": 30,
                "Attack_1": "95",
                "Attack_2": "55",
                "Attack_3": "3",
                "Attack_4": "240",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Water Absorb"
            }
        ],
        "Sis and Bro Lia & Luc": [
            {
                "name": "Goldeen",
                "Level": 30,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Swift Swim"
            },
            {
                "name": "Seaking",
                "Level": 30,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Swift Swim"
            }
        ],
        "Swimmer Axle": [
            {
                "name": "Tentacool",
                "Level": 27,
                "Attack_1": "48",
                "Attack_2": "132",
                "Attack_3": "51",
                "Attack_4": "61",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Clear Body"
            },
            {
                "name": "Tentacool",
                "Level": 27,
                "Attack_1": "48",
                "Attack_2": "132",
                "Attack_3": "51",
                "Attack_4": "61",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Clear Body"
            },
            {
                "name": "Staryu",
                "Level": 27,
                "Attack_1": "229",
                "Attack_2": "105",
                "Attack_3": "293",
                "Attack_4": "129",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Illuminate"
            },
            {
                "name": "Horsea",
                "Level": 27,
                "Attack_1": "145",
                "Attack_2": "108",
                "Attack_3": "43",
                "Attack_4": "55",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Swift Swim"
            },
            {
                "name": "Tentacruel",
                "Level": 27,
                "Attack_1": "48",
                "Attack_2": "132",
                "Attack_3": "51",
                "Attack_4": "61",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Clear Body"
            }
        ],
        "Swimmer Alice": [
            {
                "name": "Goldeen",
                "Level": 30,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Swift Swim"
            },
            {
                "name": "Seaking",
                "Level": 30,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Swift Swim"
            }
        ],
        "Swimmer Anya": [
            {
                "name": "Poliwag",
                "Level": 27,
                "Attack_1": "95",
                "Attack_2": "55",
                "Attack_3": "3",
                "Attack_4": "240",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Water Absorb"
            },
            {
                "name": "Goldeen",
                "Level": 27,
                "Attack_1": "346",
                "Attack_2": "48",
                "Attack_3": "30",
                "Attack_4": "175",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Swift Swim"
            },
            {
                "name": "Seaking",
                "Level": 27,
                "Attack_1": "346",
                "Attack_2": "48",
                "Attack_3": "30",
                "Attack_4": "175",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Swift Swim"
            },
            {
                "name": "Goldeen",
                "Level": 27,
                "Attack_1": "346",
                "Attack_2": "48",
                "Attack_3": "30",
                "Attack_4": "175",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Swift Swim"
            },
            {
                "name": "Poliwag",
                "Level": 27,
                "Attack_1": "95",
                "Attack_2": "55",
                "Attack_3": "3",
                "Attack_4": "240",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Water Absorb"
            }
        ],
        "Swimmer Connie": [
            {
                "name": "Staryu",
                "Level": 29,
                "Attack_1": "105",
                "Attack_2": "293",
                "Attack_3": "129",
                "Attack_4": "61",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Illuminate"
            },
            {
                "name": "Staryu",
                "Level": 29,
                "Attack_1": "105",
                "Attack_2": "293",
                "Attack_3": "129",
                "Attack_4": "61",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Illuminate"
            },
            {
                "name": "Staryu",
                "Level": 29,
                "Attack_1": "105",
                "Attack_2": "293",
                "Attack_3": "129",
                "Attack_4": "61",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Illuminate"
            }
        ],
        "Swimmer Barry": [
            {
                "name": "Shellder",
                "Level": 31,
                "Attack_1": "333",
                "Attack_2": "48",
                "Attack_3": "62",
                "Attack_4": "182",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Shell Armor"
            },
            {
                "name": "Cloyster",
                "Level": 31,
                "Attack_1": "110",
                "Attack_2": "48",
                "Attack_3": "62",
                "Attack_4": "182",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Shell Armor"
            }
        ],
        "Swimmer Darrin": [
            {
                "name": "Horsea",
                "Level": 28,
                "Attack_1": "145",
                "Attack_2": "108",
                "Attack_3": "43",
                "Attack_4": "55",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Swift Swim"
            },
            {
                "name": "Horsea",
                "Level": 28,
                "Attack_1": "145",
                "Attack_2": "108",
                "Attack_3": "43",
                "Attack_4": "55",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Swift Swim"
            },
            {
                "name": "Horsea",
                "Level": 28,
                "Attack_1": "145",
                "Attack_2": "108",
                "Attack_3": "43",
                "Attack_4": "55",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Swift Swim"
            },
            {
                "name": "Seadra",
                "Level": 28,
                "Attack_1": "145",
                "Attack_2": "108",
                "Attack_3": "43",
                "Attack_4": "55",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Poison Point"
            }
        ],
        "Swimmer Shirley": [
            {
                "name": "Seadra",
                "Level": 30,
                "Attack_1": "108",
                "Attack_2": "43",
                "Attack_3": "55",
                "Attack_4": "239",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Poison Point"
            },
            {
                "name": "Horsea",
                "Level": 30,
                "Attack_1": "108",
                "Attack_2": "43",
                "Attack_3": "55",
                "Attack_4": "239",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Swift Swim"
            },
            {
                "name": "Seadra",
                "Level": 30,
                "Attack_1": "108",
                "Attack_2": "43",
                "Attack_3": "55",
                "Attack_4": "239",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Poison Point"
            }
        ],
        "Swimmer Tiffany": [
            {
                "name": "Seaking",
                "Level": 35,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Swift Swim"
            }
        ],
        "Picknicker Irene": [
            {
                "name": "Tentacool",
                "Level": 30,
                "Attack_1": "132",
                "Attack_2": "51",
                "Attack_3": "61",
                "Attack_4": "35",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Clear Body"
            },
            {
                "name": "Horsea",
                "Level": 30,
                "Attack_1": "108",
                "Attack_2": "43",
                "Attack_3": "55",
                "Attack_4": "239",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Swift Swim"
            },
            {
                "name": "Seel",
                "Level": 30,
                "Attack_1": "45",
                "Attack_2": "196",
                "Attack_3": "62",
                "Attack_4": "156",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Thick Fat"
            }
        ],
        "Bird Keeper Roger": [
            {
                "name": "Fearow",
                "Level": 30,
                "Attack_1": "45",
                "Attack_2": "43",
                "Attack_3": "31",
                "Attack_4": "228",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Keen Eye"
            },
            {
                "name": "Fearow",
                "Level": 30,
                "Attack_1": "45",
                "Attack_2": "43",
                "Attack_3": "31",
                "Attack_4": "228",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Keen Eye"
            },
            {
                "name": "Pidgeotto",
                "Level": 30,
                "Attack_1": "16",
                "Attack_2": "98",
                "Attack_3": "18",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Keen Eye"
            }
        ],
        "Swimmer Nora": [
            {
                "name": "Shellder",
                "Level": 30,
                "Attack_1": "333",
                "Attack_2": "48",
                "Attack_3": "62",
                "Attack_4": "182",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Shell Armor"
            },
            {
                "name": "Shellder",
                "Level": 30,
                "Attack_1": "333",
                "Attack_2": "48",
                "Attack_3": "62",
                "Attack_4": "182",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Shell Armor"
            },
            {
                "name": "Cloyster",
                "Level": 30,
                "Attack_1": "110",
                "Attack_2": "48",
                "Attack_3": "62",
                "Attack_4": "182",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Shell Armor"
            }
        ],
        "Swimmer Dean": [
            {
                "name": "Staryu",
                "Level": 35,
                "Attack_1": "293",
                "Attack_2": "129",
                "Attack_3": "61",
                "Attack_4": "107",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Illuminate"
            }
        ],
        "Picknicker Missy": [
            {
                "name": "Goldeen",
                "Level": 31,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Swift Swim"
            },
            {
                "name": "Seaking",
                "Level": 31,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Swift Swim"
            }
        ],
        "Swimmer Melissa": [
            {
                "name": "Poliwag",
                "Level": 31,
                "Attack_1": "55",
                "Attack_2": "3",
                "Attack_3": "240",
                "Attack_4": "34",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Water Absorb"
            },
            {
                "name": "Seaking",
                "Level": 31,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Swift Swim"
            }
        ],
        "Youngster Johnson": [
            {
                "name": "Ekans",
                "Level": 33,
                "Attack_1": "44",
                "Attack_2": "137",
                "Attack_3": "103",
                "Attack_4": "51",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Intimidate"
            },
            {
                "name": "Ekans",
                "Level": 33,
                "Attack_1": "44",
                "Attack_2": "137",
                "Attack_3": "103",
                "Attack_4": "51",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Intimidate"
            },
            {
                "name": "Raticate",
                "Level": 34,
                "Attack_1": "98",
                "Attack_2": "158",
                "Attack_3": "184",
                "Attack_4": "228",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Run Away"
            }
        ],
        "Burglar Arnie": [
            {
                "name": "Charmander",
                "Level": 34,
                "Attack_1": "232",
                "Attack_2": "108",
                "Attack_3": "184",
                "Attack_4": "53",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Blaze"
            },
            {
                "name": "Charmeleon",
                "Level": 34,
                "Attack_1": "232",
                "Attack_2": "108",
                "Attack_3": "184",
                "Attack_4": "53",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Blaze"
            }
        ],
        "Burglar Simon": [
            {
                "name": "Ninetales",
                "Level": 38,
                "Attack_1": "52",
                "Attack_2": "98",
                "Attack_3": "109",
                "Attack_4": "219",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Flash Fire"
            }
        ],
        "Scientist Braydon": [
            {
                "name": "Magnemite",
                "Level": 33,
                "Attack_1": "49",
                "Attack_2": "86",
                "Attack_3": "209",
                "Attack_4": "199",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Magnet Pull"
            },
            {
                "name": "Magneton",
                "Level": 33,
                "Attack_1": "48",
                "Attack_2": "49",
                "Attack_3": "86",
                "Attack_4": "209",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Magnet Pull"
            },
            {
                "name": "Voltorb",
                "Level": 33,
                "Attack_1": "49",
                "Attack_2": "209",
                "Attack_3": "120",
                "Attack_4": "205",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Soundproof"
            }
        ],
        "Scientist Ted": [
            {
                "name": "Electrode",
                "Level": 29,
                "Attack_1": "103",
                "Attack_2": "49",
                "Attack_3": "209",
                "Attack_4": "120",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Soundproof"
            },
            {
                "name": "Weezing",
                "Level": 29,
                "Attack_1": "123",
                "Attack_2": "120",
                "Attack_3": "124",
                "Attack_4": "108",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Levitate"
            }
        ],
        "Burglar Lewis": [
            {
                "name": "Growlithe",
                "Level": 34,
                "Attack_1": "43",
                "Attack_2": "316",
                "Attack_3": "36",
                "Attack_4": "172",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Intimidate"
            },
            {
                "name": "Ponyta",
                "Level": 34,
                "Attack_1": "52",
                "Attack_2": "23",
                "Attack_3": "83",
                "Attack_4": "36",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Run Away"
            }
        ],
        "Scientist Ivan": [
            {
                "name": "Magnemite",
                "Level": 34,
                "Attack_1": "49",
                "Attack_2": "86",
                "Attack_3": "209",
                "Attack_4": "199",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Magnet Pull"
            },
            {
                "name": "Electrode",
                "Level": 34,
                "Attack_1": "49",
                "Attack_2": "209",
                "Attack_3": "120",
                "Attack_4": "205",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Soundproof"
            }
        ],
        "Lass Crissy": [
            {
                "name": "Paras",
                "Level": 31,
                "Attack_1": "77",
                "Attack_2": "141",
                "Attack_3": "147",
                "Attack_4": "163",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Effect Spore"
            },
            {
                "name": "Paras",
                "Level": 31,
                "Attack_1": "77",
                "Attack_2": "141",
                "Attack_3": "147",
                "Attack_4": "163",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Effect Spore"
            },
            {
                "name": "Paras",
                "Level": 31,
                "Attack_1": "78",
                "Attack_2": "77",
                "Attack_3": "141",
                "Attack_4": "147",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Effect Spore"
            }
        ],
        "Burglar Quinn": [
            {
                "name": "Growlithe",
                "Level": 36,
                "Attack_1": "43",
                "Attack_2": "316",
                "Attack_3": "36",
                "Attack_4": "172",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Intimidate"
            },
            {
                "name": "Vulpix",
                "Level": 36,
                "Attack_1": "109",
                "Attack_2": "286",
                "Attack_3": "53",
                "Attack_4": "219",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Flash Fire"
            },
            {
                "name": "Ninetales",
                "Level": 36,
                "Attack_1": "52",
                "Attack_2": "98",
                "Attack_3": "109",
                "Attack_4": "219",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Flash Fire"
            }
        ],
        "Super Nerd Erik": [
            {
                "name": "Vulpix",
                "Level": 36,
                "Attack_1": "109",
                "Attack_2": "286",
                "Attack_3": "53",
                "Attack_4": "219",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Flash Fire"
            },
            {
                "name": "Vulpix",
                "Level": 36,
                "Attack_1": "109",
                "Attack_2": "286",
                "Attack_3": "53",
                "Attack_4": "219",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Flash Fire"
            },
            {
                "name": "Ninetales",
                "Level": 36,
                "Attack_1": "52",
                "Attack_2": "98",
                "Attack_3": "109",
                "Attack_4": "219",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Flash Fire"
            }
        ],
        "Super Nerd Avery": [
            {
                "name": "Ponyta",
                "Level": 34,
                "Attack_1": "52",
                "Attack_2": "23",
                "Attack_3": "83",
                "Attack_4": "36",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Run Away"
            },
            {
                "name": "Charmander",
                "Level": 34,
                "Attack_1": "232",
                "Attack_2": "108",
                "Attack_3": "184",
                "Attack_4": "53",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Blaze"
            },
            {
                "name": "Vulpix",
                "Level": 34,
                "Attack_1": "109",
                "Attack_2": "286",
                "Attack_3": "53",
                "Attack_4": "219",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Flash Fire"
            },
            {
                "name": "Growlithe",
                "Level": 34,
                "Attack_1": "43",
                "Attack_2": "316",
                "Attack_3": "36",
                "Attack_4": "172",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Intimidate"
            }
        ],
        "Burglar Ramon": [
            {
                "name": "Ponyta",
                "Level": 41,
                "Attack_1": "23",
                "Attack_2": "83",
                "Attack_3": "36",
                "Attack_4": "97",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Run Away"
            }
        ],
        "Super Nerd Derek": [
            {
                "name": "Rapidash",
                "Level": 41,
                "Attack_1": "83",
                "Attack_2": "36",
                "Attack_3": "97",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Run Away"
            }
        ],
        "Burglar Dusty": [
            {
                "name": "Vulpix",
                "Level": 37,
                "Attack_1": "286",
                "Attack_2": "53",
                "Attack_3": "219",
                "Attack_4": "288",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Flash Fire"
            },
            {
                "name": "Growlithe",
                "Level": 37,
                "Attack_1": "316",
                "Attack_2": "36",
                "Attack_3": "172",
                "Attack_4": "270",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Intimidate"
            }
        ],
        "Super Nerd Zac": [
            {
                "name": "Growlithe",
                "Level": 37,
                "Attack_1": "316",
                "Attack_2": "36",
                "Attack_3": "172",
                "Attack_4": "270",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Intimidate"
            },
            {
                "name": "Vulpix",
                "Level": 37,
                "Attack_1": "286",
                "Attack_2": "53",
                "Attack_3": "219",
                "Attack_4": "288",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Flash Fire"
            }
        ],
        "Leader Blaine": [
            {
                "name": "Growlithe",
                "Level": 42,
                "Attack_1": "44",
                "Attack_2": "46",
                "Attack_3": "36",
                "Attack_4": "126",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Intimidate"
            },
            {
                "name": "Ponyta",
                "Level": 40,
                "Attack_1": "23",
                "Attack_2": "340",
                "Attack_3": "83",
                "Attack_4": "126",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Run Away"
            },
            {
                "name": "Rapidash",
                "Level": 42,
                "Attack_1": "23",
                "Attack_2": "340",
                "Attack_3": "83",
                "Attack_4": "126",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Run Away"
            },
            {
                "name": "Arcanine",
                "Level": 47,
                "Attack_1": "44",
                "Attack_2": "46",
                "Attack_3": "36",
                "Attack_4": "126",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Intimidate"
            }
        ],
        "Fisherman Wade": [
            {
                "name": "Magikarp",
                "Level": 27,
                "Attack_1": "150",
                "Attack_2": "33",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Swift Swim"
            },
            {
                "name": "Magikarp",
                "Level": 27,
                "Attack_1": "150",
                "Attack_2": "33",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Swift Swim"
            },
            {
                "name": "Magikarp",
                "Level": 27,
                "Attack_1": "150",
                "Attack_2": "33",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Swift Swim"
            },
            {
                "name": "Magikarp",
                "Level": 27,
                "Attack_1": "150",
                "Attack_2": "33",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Swift Swim"
            },
            {
                "name": "Magikarp",
                "Level": 27,
                "Attack_1": "150",
                "Attack_2": "33",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Swift Swim"
            },
            {
                "name": "Magikarp",
                "Level": 27,
                "Attack_1": "150",
                "Attack_2": "33",
                "Attack_3": null,
                "Attack_4": null,
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Swift Swim"
            }
        ],
        "Fisherman Ronald": [
            {
                "name": "Seaking",
                "Level": 28,
                "Attack_1": "346",
                "Attack_2": "48",
                "Attack_3": "30",
                "Attack_4": "175",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Swift Swim"
            },
            {
                "name": "Goldeen",
                "Level": 28,
                "Attack_1": "346",
                "Attack_2": "48",
                "Attack_3": "30",
                "Attack_4": "175",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Swift Swim"
            },
            {
                "name": "Seaking",
                "Level": 28,
                "Attack_1": "346",
                "Attack_2": "48",
                "Attack_3": "30",
                "Attack_4": "175",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Swift Swim"
            },
            {
                "name": "Seaking",
                "Level": 28,
                "Attack_1": "346",
                "Attack_2": "48",
                "Attack_3": "30",
                "Attack_4": "175",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Swift Swim"
            }
        ],
        "Sis and Bro Lil & Ian": [
            {
                "name": "Seadra",
                "Level": 33,
                "Attack_1": "108",
                "Attack_2": "43",
                "Attack_3": "55",
                "Attack_4": "239",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Poison Point"
            },
            {
                "name": "Starmie",
                "Level": 33,
                "Attack_1": "229",
                "Attack_2": "105",
                "Attack_3": "129",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Illuminate"
            }
        ],
        "Swimmer Spencer": [
            {
                "name": "Seadra",
                "Level": 33,
                "Attack_1": "108",
                "Attack_2": "43",
                "Attack_3": "55",
                "Attack_4": "239",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Poison Point"
            },
            {
                "name": "Tentacruel",
                "Level": 33,
                "Attack_1": "132",
                "Attack_2": "51",
                "Attack_3": "61",
                "Attack_4": "35",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Clear Body"
            }
        ],
        "Fisherman Claude": [
            {
                "name": "Shellder",
                "Level": 31,
                "Attack_1": "333",
                "Attack_2": "48",
                "Attack_3": "62",
                "Attack_4": "182",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Shell Armor"
            },
            {
                "name": "Cloyster",
                "Level": 31,
                "Attack_1": "110",
                "Attack_2": "48",
                "Attack_3": "62",
                "Attack_4": "182",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Shell Armor"
            }
        ],
        "Fisherman Nolan": [
            {
                "name": "Seaking",
                "Level": 33,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Swift Swim"
            },
            {
                "name": "Goldeen",
                "Level": 33,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Swift Swim"
            }
        ],
        "Swimmer Jack": [
            {
                "name": "Starmie",
                "Level": 37,
                "Attack_1": "229",
                "Attack_2": "105",
                "Attack_3": "129",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Illuminate"
            }
        ],
        "Swimmer Roland": [
            {
                "name": "Poliwhirl",
                "Level": 32,
                "Attack_1": "95",
                "Attack_2": "55",
                "Attack_3": "3",
                "Attack_4": "240",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Water Absorb"
            },
            {
                "name": "Tentacool",
                "Level": 32,
                "Attack_1": "132",
                "Attack_2": "51",
                "Attack_3": "61",
                "Attack_4": "35",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Clear Body"
            },
            {
                "name": "Seadra",
                "Level": 32,
                "Attack_1": "108",
                "Attack_2": "43",
                "Attack_3": "55",
                "Attack_4": "239",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Poison Point"
            }
        ],
        "Swimmer Jerome": [
            {
                "name": "Staryu",
                "Level": 33,
                "Attack_1": "293",
                "Attack_2": "129",
                "Attack_3": "61",
                "Attack_4": "107",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Illuminate"
            },
            {
                "name": "Wartortle",
                "Level": 33,
                "Attack_1": "55",
                "Attack_2": "44",
                "Attack_3": "229",
                "Attack_4": "182",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Torrent"
            }
        ],
        "Swimmer Amara": [
            {
                "name": "Seel",
                "Level": 36,
                "Attack_1": "45",
                "Attack_2": "196",
                "Attack_3": "62",
                "Attack_4": "156",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Thick Fat"
            },
            {
                "name": "Seel",
                "Level": 36,
                "Attack_1": "45",
                "Attack_2": "196",
                "Attack_3": "62",
                "Attack_4": "156",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Thick Fat"
            },
            {
                "name": "Dewgong",
                "Level": 36,
                "Attack_1": "196",
                "Attack_2": "62",
                "Attack_3": "156",
                "Attack_4": "329",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Thick Fat"
            }
        ],
        "Swimmer Abigail": [
            {
                "name": "Psyduck",
                "Level": 35,
                "Attack_1": "50",
                "Attack_2": "93",
                "Attack_3": "103",
                "Attack_4": "244",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Damp"
            },
            {
                "name": "Psyduck",
                "Level": 36,
                "Attack_1": "50",
                "Attack_2": "93",
                "Attack_3": "103",
                "Attack_4": "244",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Damp"
            },
            {
                "name": "Golduck",
                "Level": 37,
                "Attack_1": "50",
                "Attack_2": "93",
                "Attack_3": "103",
                "Attack_4": "244",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Damp"
            }
        ],
        "Picknicker Claire": [
            {
                "name": "Meowth",
                "Level": 35,
                "Attack_1": "44",
                "Attack_2": "6",
                "Attack_3": "185",
                "Attack_4": "103",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Pickup"
            },
            {
                "name": "Meowth",
                "Level": 35,
                "Attack_1": "44",
                "Attack_2": "6",
                "Attack_3": "185",
                "Attack_4": "103",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Pickup"
            },
            {
                "name": "Pikachu",
                "Level": 35,
                "Attack_1": "104",
                "Attack_2": "21",
                "Attack_3": "85",
                "Attack_4": "97",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Static"
            },
            {
                "name": "Clefairy",
                "Level": 35,
                "Attack_1": "107",
                "Attack_2": "111",
                "Attack_3": "118",
                "Attack_4": "322",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Cute Charm"
            }
        ],
        "Crush Girl Tanya": [
            {
                "name": "Hitmonlee",
                "Level": 38,
                "Attack_1": "116",
                "Attack_2": "136",
                "Attack_3": "170",
                "Attack_4": "193",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Limber"
            },
            {
                "name": "Hitmonchan",
                "Level": 38,
                "Attack_1": "8",
                "Attack_2": "7",
                "Attack_3": "327",
                "Attack_4": "5",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Keen Eye"
            }
        ],
        "Camper Bryce": [
            {
                "name": "Nidorino",
                "Level": 36,
                "Attack_1": "40",
                "Attack_2": "30",
                "Attack_3": "270",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Poison Point"
            },
            {
                "name": "Raticate",
                "Level": 36,
                "Attack_1": "98",
                "Attack_2": "158",
                "Attack_3": "184",
                "Attack_4": "228",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Run Away"
            },
            {
                "name": "Sandslash",
                "Level": 36,
                "Attack_1": "28",
                "Attack_2": "40",
                "Attack_3": "163",
                "Attack_4": "129",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Sand Veil"
            }
        ],
        "Swimmer Garrett": [
            {
                "name": "Shellder",
                "Level": 35,
                "Attack_1": "333",
                "Attack_2": "48",
                "Attack_3": "62",
                "Attack_4": "182",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Shell Armor"
            },
            {
                "name": "Cloyster",
                "Level": 35,
                "Attack_1": "110",
                "Attack_2": "48",
                "Attack_3": "62",
                "Attack_4": "182",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Shell Armor"
            },
            {
                "name": "Wartortle",
                "Level": 38,
                "Attack_1": "44",
                "Attack_2": "229",
                "Attack_3": "182",
                "Attack_4": "240",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Torrent"
            }
        ],
        "Crush Kin Mik & Kia": [
            {
                "name": "Machoke",
                "Level": 39,
                "Attack_1": "69",
                "Attack_2": "193",
                "Attack_3": "279",
                "Attack_4": "233",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Guts"
            },
            {
                "name": "Primeape",
                "Level": 39,
                "Attack_1": "116",
                "Attack_2": "69",
                "Attack_3": "99",
                "Attack_4": "238",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Vital Spirit"
            }
        ],
        "Black Belt Hugh": [
            {
                "name": "Machop",
                "Level": 37,
                "Attack_1": "193",
                "Attack_2": "279",
                "Attack_3": "233",
                "Attack_4": "66",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Guts"
            },
            {
                "name": "Machoke",
                "Level": 37,
                "Attack_1": "69",
                "Attack_2": "193",
                "Attack_3": "279",
                "Attack_4": "233",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Guts"
            }
        ],
        "Black Belt Shea": [
            {
                "name": "Machop",
                "Level": 38,
                "Attack_1": "193",
                "Attack_2": "279",
                "Attack_3": "233",
                "Attack_4": "66",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Guts"
            },
            {
                "name": "Machoke",
                "Level": 38,
                "Attack_1": "69",
                "Attack_2": "193",
                "Attack_3": "279",
                "Attack_4": "233",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Guts"
            }
        ],
        "Crush Girl Sharon": [
            {
                "name": "Mankey",
                "Level": 37,
                "Attack_1": "116",
                "Attack_2": "69",
                "Attack_3": "238",
                "Attack_4": "207",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Vital Spirit"
            },
            {
                "name": "Primeape",
                "Level": 37,
                "Attack_1": "116",
                "Attack_2": "69",
                "Attack_3": "99",
                "Attack_4": "238",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Vital Spirit"
            }
        ],
        "Swimmer Finn": [
            {
                "name": "Starmie",
                "Level": 38,
                "Attack_1": "229",
                "Attack_2": "105",
                "Attack_3": "129",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Illuminate"
            }
        ],
        "Swimmer Maria": [
            {
                "name": "Seadra",
                "Level": 37,
                "Attack_1": "108",
                "Attack_2": "43",
                "Attack_3": "55",
                "Attack_4": "239",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Poison Point"
            },
            {
                "name": "Seadra",
                "Level": 37,
                "Attack_1": "108",
                "Attack_2": "43",
                "Attack_3": "55",
                "Attack_4": "239",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Poison Point"
            }
        ],
        "Fisherman Tommy": [
            {
                "name": "Goldeen",
                "Level": 33,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Swift Swim"
            },
            {
                "name": "Goldeen",
                "Level": 33,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Swift Swim"
            },
            {
                "name": "Seaking",
                "Level": 35,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Swift Swim"
            },
            {
                "name": "Seaking",
                "Level": 35,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Swift Swim"
            },
            {
                "name": "Seaking",
                "Level": 35,
                "Attack_1": "48",
                "Attack_2": "30",
                "Attack_3": "175",
                "Attack_4": "31",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Swift Swim"
            }
        ],
        "Pokemon Ranger Beth": [
            {
                "name": "Bellsprout",
                "Level": 38,
                "Attack_1": "78",
                "Attack_2": "51",
                "Attack_3": "230",
                "Attack_4": "75",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Gloom",
                "Level": 38,
                "Attack_1": "78",
                "Attack_2": "79",
                "Attack_3": "51",
                "Attack_4": "236",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Gloom",
                "Level": 38,
                "Attack_1": "78",
                "Attack_2": "79",
                "Attack_3": "51",
                "Attack_4": "236",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Chlorophyll"
            }
        ],
        "Crush Girl Jocelyn": [
            {
                "name": "Hitmonchan",
                "Level": 38,
                "Attack_1": "8",
                "Attack_2": "7",
                "Attack_3": "327",
                "Attack_4": "5",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Keen Eye"
            },
            {
                "name": "Hitmonchan",
                "Level": 38,
                "Attack_1": "8",
                "Attack_2": "7",
                "Attack_3": "327",
                "Attack_4": "5",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Keen Eye"
            }
        ],
        "Pokemon Ranger Logan": [
            {
                "name": "Exeggcute",
                "Level": 37,
                "Attack_1": "79",
                "Attack_2": "77",
                "Attack_3": "78",
                "Attack_4": "93",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Exeggutor",
                "Level": 40,
                "Attack_1": "121",
                "Attack_2": "23",
                "Attack_3": "93",
                "Attack_4": "95",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Chlorophyll"
            }
        ],
        "Team Rocket Grunt 41": [
            {
                "name": "Cubone",
                "Level": 37,
                "Attack_1": "155",
                "Attack_2": "99",
                "Attack_3": "206",
                "Attack_4": "37",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Rock Head"
            },
            {
                "name": "Marowak",
                "Level": 37,
                "Attack_1": "43",
                "Attack_2": "116",
                "Attack_3": "155",
                "Attack_4": "99",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Rock Head"
            }
        ],
        "Team Rocket Grunt 42": [
            {
                "name": "Rattata",
                "Level": 35,
                "Attack_1": "158",
                "Attack_2": "116",
                "Attack_3": "228",
                "Attack_4": "162",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Run Away"
            },
            {
                "name": "Raticate",
                "Level": 35,
                "Attack_1": "98",
                "Attack_2": "158",
                "Attack_3": "184",
                "Attack_4": "228",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Run Away"
            },
            {
                "name": "Sandshrew",
                "Level": 35,
                "Attack_1": "28",
                "Attack_2": "40",
                "Attack_3": "163",
                "Attack_4": "129",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Sand Veil"
            },
            {
                "name": "Sandslash",
                "Level": 35,
                "Attack_1": "28",
                "Attack_2": "40",
                "Attack_3": "163",
                "Attack_4": "129",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Sand Veil"
            }
        ],
        "Biker Goon 1": [
            {
                "name": "Koffing",
                "Level": 37,
                "Attack_1": "114",
                "Attack_2": "108",
                "Attack_3": "124",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Levitate"
            },
            {
                "name": "Grimer",
                "Level": 37,
                "Attack_1": "151",
                "Attack_2": "103",
                "Attack_3": "107",
                "Attack_4": "124",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Stench"
            }
        ],
        "Biker Goon 2": [
            {
                "name": "Koffing",
                "Level": 38,
                "Attack_1": "114",
                "Attack_2": "108",
                "Attack_3": "124",
                "Attack_4": "33",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Levitate"
            }
        ],
        "Biker Goon 3": [
            {
                "name": "Grimer",
                "Level": 38,
                "Attack_1": "124",
                "Attack_2": "107",
                "Attack_3": "103",
                "Attack_4": "151",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Stench"
            }
        ],
        "Cue Ball Paxton": [
            {
                "name": "Weezing",
                "Level": 39,
                "Attack_1": "120",
                "Attack_2": "124",
                "Attack_3": "108",
                "Attack_4": "114",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Levitate"
            },
            {
                "name": "Muk",
                "Level": 39,
                "Attack_1": "124",
                "Attack_2": "107",
                "Attack_3": "103",
                "Attack_4": "151",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Stench"
            }
        ],
        "Twins Joy & Meg": [
            {
                "name": "Clefairy",
                "Level": 37,
                "Attack_1": "111",
                "Attack_2": "118",
                "Attack_3": "322",
                "Attack_4": "236",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Cute Charm"
            },
            {
                "name": "Clefairy",
                "Level": 37,
                "Attack_1": "111",
                "Attack_2": "118",
                "Attack_3": "322",
                "Attack_4": "236",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Cute Charm"
            }
        ],
        "Aroma Lady Violet": [
            {
                "name": "Bulbasaur",
                "Level": 36,
                "Attack_1": "79",
                "Attack_2": "75",
                "Attack_3": "230",
                "Attack_4": "74",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Overgrow"
            },
            {
                "name": "Ivysaur",
                "Level": 36,
                "Attack_1": "77",
                "Attack_2": "79",
                "Attack_3": "75",
                "Attack_4": "230",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Overgrow"
            },
            {
                "name": "Ivysaur",
                "Level": 36,
                "Attack_1": "77",
                "Attack_2": "79",
                "Attack_3": "75",
                "Attack_4": "230",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Overgrow"
            }
        ],
        "Tuber Alexis": [
            {
                "name": "Staryu",
                "Level": 34,
                "Attack_1": "293",
                "Attack_2": "129",
                "Attack_3": "61",
                "Attack_4": "107",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Illuminate"
            },
            {
                "name": "Staryu",
                "Level": 34,
                "Attack_1": "293",
                "Attack_2": "129",
                "Attack_3": "61",
                "Attack_4": "107",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Illuminate"
            },
            {
                "name": "Krabby",
                "Level": 34,
                "Attack_1": "106",
                "Attack_2": "341",
                "Attack_3": "23",
                "Attack_4": "12",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Hyper Cutter"
            },
            {
                "name": "Krabby",
                "Level": 34,
                "Attack_1": "106",
                "Attack_2": "341",
                "Attack_3": "23",
                "Attack_4": "12",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Hyper Cutter"
            }
        ],
        "Swimmer Tisha": [
            {
                "name": "Kingler",
                "Level": 38,
                "Attack_1": "106",
                "Attack_2": "341",
                "Attack_3": "23",
                "Attack_4": "12",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Hyper Cutter"
            }
        ],
        "Tuber Amira": [
            {
                "name": "Poliwag",
                "Level": 34,
                "Attack_1": "55",
                "Attack_2": "3",
                "Attack_3": "240",
                "Attack_4": "34",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Water Absorb"
            },
            {
                "name": "Poliwhirl",
                "Level": 35,
                "Attack_1": "55",
                "Attack_2": "3",
                "Attack_3": "240",
                "Attack_4": "34",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Water Absorb"
            },
            {
                "name": "Poliwag",
                "Level": 34,
                "Attack_1": "55",
                "Attack_2": "3",
                "Attack_3": "240",
                "Attack_4": "34",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Water Absorb"
            }
        ],
        "Aroma Lady Nikki": [
            {
                "name": "Bellsprout",
                "Level": 37,
                "Attack_1": "78",
                "Attack_2": "51",
                "Attack_3": "230",
                "Attack_4": "75",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Weepinbell",
                "Level": 37,
                "Attack_1": "77",
                "Attack_2": "78",
                "Attack_3": "51",
                "Attack_4": "230",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Chlorophyll"
            }
        ],
        "Tamer Cole": [
            {
                "name": "Arbok",
                "Level": 39,
                "Attack_1": "44",
                "Attack_2": "137",
                "Attack_3": "103",
                "Attack_4": "51",
                "IVS": {
                    "hp": 4,
                    "atk": 4,
                    "def": 4,
                    "spa": 4,
                    "spd": 4,
                    "spe": 4
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Intimidate"
            },
            {
                "name": "Tauros",
                "Level": 39,
                "Attack_1": "184",
                "Attack_2": "228",
                "Attack_3": "207",
                "Attack_4": "156",
                "IVS": {
                    "hp": 4,
                    "atk": 4,
                    "def": 4,
                    "spa": 4,
                    "spd": 4,
                    "spe": 4
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Intimidate"
            }
        ],
        "Black Belt Kiyo": [
            {
                "name": "Machoke",
                "Level": 43,
                "Attack_1": "193",
                "Attack_2": "279",
                "Attack_3": "233",
                "Attack_4": "66",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Guts"
            }
        ],
        "Tamer Jason": [
            {
                "name": "Rhyhorn",
                "Level": 43,
                "Attack_1": "184",
                "Attack_2": "350",
                "Attack_3": "32",
                "Attack_4": "36",
                "IVS": {
                    "hp": 4,
                    "atk": 4,
                    "def": 4,
                    "spa": 4,
                    "spd": 4,
                    "spe": 4
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Lightningrod"
            }
        ],
        "Black Belt Atsushi": [
            {
                "name": "Machop",
                "Level": 40,
                "Attack_1": "279",
                "Attack_2": "233",
                "Attack_3": "66",
                "Attack_4": "238",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Guts"
            },
            {
                "name": "Machoke",
                "Level": 40,
                "Attack_1": "69",
                "Attack_2": "193",
                "Attack_3": "279",
                "Attack_4": "233",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Guts"
            }
        ],
        "Cooltrainer Samuel": [
            {
                "name": "Sandslash",
                "Level": 37,
                "Attack_1": "163",
                "Attack_2": "129",
                "Attack_3": "28",
                "Attack_4": "40",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Sand Veil"
            },
            {
                "name": "Sandslash",
                "Level": 37,
                "Attack_1": "163",
                "Attack_2": "129",
                "Attack_3": "28",
                "Attack_4": "40",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Sand Veil"
            },
            {
                "name": "Rhyhorn",
                "Level": 38,
                "Attack_1": "36",
                "Attack_2": "350",
                "Attack_3": "31",
                "Attack_4": "184",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Lightningrod"
            },
            {
                "name": "Nidorino",
                "Level": 39,
                "Attack_1": "31",
                "Attack_2": "30",
                "Attack_3": "40",
                "Attack_4": "24",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Poison Point"
            },
            {
                "name": "Nidoking",
                "Level": 39,
                "Attack_1": "37",
                "Attack_2": "24",
                "Attack_3": "40",
                "Attack_4": "116",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Poison Point"
            }
        ],
        "Black Belt Takashi": [
            {
                "name": "Machoke",
                "Level": 38,
                "Attack_1": "69",
                "Attack_2": "193",
                "Attack_3": "279",
                "Attack_4": "233",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Guts"
            },
            {
                "name": "Machop",
                "Level": 38,
                "Attack_1": "193",
                "Attack_2": "279",
                "Attack_3": "233",
                "Attack_4": "66",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Guts"
            },
            {
                "name": "Machoke",
                "Level": 38,
                "Attack_1": "69",
                "Attack_2": "193",
                "Attack_3": "279",
                "Attack_4": "233",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Guts"
            }
        ],
        "Cooltrainer Yuji": [
            {
                "name": "Sandslash",
                "Level": 38,
                "Attack_1": "163",
                "Attack_2": "129",
                "Attack_3": "28",
                "Attack_4": "40",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Sand Veil"
            },
            {
                "name": "Graveler",
                "Level": 38,
                "Attack_1": "350",
                "Attack_2": "222",
                "Attack_3": "300",
                "Attack_4": "111",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Rock Head"
            },
            {
                "name": "Onix",
                "Level": 38,
                "Attack_1": "225",
                "Attack_2": "201",
                "Attack_3": "88",
                "Attack_4": "20",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Rock Head"
            },
            {
                "name": "Graveler",
                "Level": 38,
                "Attack_1": "350",
                "Attack_2": "222",
                "Attack_3": "205",
                "Attack_4": "111",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Rock Head"
            },
            {
                "name": "Marowak",
                "Level": 38,
                "Attack_1": "155",
                "Attack_2": "29",
                "Attack_3": "43",
                "Attack_4": "45",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Rock Head"
            }
        ],
        "Cooltrainer Warren": [
            {
                "name": "Marowak",
                "Level": 37,
                "Attack_1": "155",
                "Attack_2": "29",
                "Attack_3": "43",
                "Attack_4": "45",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Rock Head"
            },
            {
                "name": "Marowak",
                "Level": 37,
                "Attack_1": "155",
                "Attack_2": "29",
                "Attack_3": "43",
                "Attack_4": "45",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Rock Head"
            },
            {
                "name": "Rhyhorn",
                "Level": 38,
                "Attack_1": "36",
                "Attack_2": "350",
                "Attack_3": "31",
                "Attack_4": "184",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Lightningrod"
            },
            {
                "name": "Nidorina",
                "Level": 39,
                "Attack_1": "154",
                "Attack_2": "44",
                "Attack_3": "40",
                "Attack_4": "24",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Poison Point"
            },
            {
                "name": "Nidoqueen",
                "Level": 39,
                "Attack_1": "34",
                "Attack_2": "44",
                "Attack_3": "40",
                "Attack_4": "24",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Poison Point"
            }
        ],
        "Leader Giovanni": [
            {
                "name": "Rhyhorn",
                "Level": 45,
                "Attack_1": "36",
                "Attack_2": "350",
                "Attack_3": "184",
                "Attack_4": "89",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Lightningrod"
            },
            {
                "name": "Dugtrio",
                "Level": 42,
                "Attack_1": "163",
                "Attack_2": "328",
                "Attack_3": "189",
                "Attack_4": "89",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Sand Veil"
            },
            {
                "name": "Nidoqueen",
                "Level": 44,
                "Attack_1": "34",
                "Attack_2": "24",
                "Attack_3": "40",
                "Attack_4": "89",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Poison Point"
            },
            {
                "name": "Nidoking",
                "Level": 45,
                "Attack_1": "37",
                "Attack_2": "24",
                "Attack_3": "40",
                "Attack_4": "89",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Poison Point"
            },
            {
                "name": "Rhyhorn",
                "Level": 50,
                "Attack_1": "36",
                "Attack_2": "350",
                "Attack_3": "184",
                "Attack_4": "89",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Lightningrod"
            }
        ],
        "22 Rival 2(Bulbasaur)": [
            {
                "name": "Pidgeot",
                "Level": 47,
                "Attack_1": "297",
                "Attack_2": "17",
                "Attack_3": "16",
                "Attack_4": "98",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Keen Eye"
            },
            {
                "name": "Rhyhorn",
                "Level": 45,
                "Attack_1": "36",
                "Attack_2": "32",
                "Attack_3": "350",
                "Attack_4": "31",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Lightningrod"
            },
            {
                "name": "Gyarados",
                "Level": 45,
                "Attack_1": "56",
                "Attack_2": "239",
                "Attack_3": "43",
                "Attack_4": "240",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Intimidate"
            },
            {
                "name": "Exeggcute",
                "Level": 45,
                "Attack_1": "76",
                "Attack_2": "79",
                "Attack_3": "77",
                "Attack_4": "78",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Alakazam",
                "Level": 47,
                "Attack_1": "94",
                "Attack_2": "347",
                "Attack_3": "248",
                "Attack_4": "50",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Synchronize"
            },
            {
                "name": "Charizard",
                "Level": 53,
                "Attack_1": "53",
                "Attack_2": "17",
                "Attack_3": "163",
                "Attack_4": "184",
                "IVS": {
                    "hp": 30,
                    "atk": 30,
                    "def": 30,
                    "spa": 30,
                    "spd": 30,
                    "spe": 30
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Blaze"
            }
        ],
        "22 Rival 2(Squirtle)": [
            {
                "name": "Pidgeot",
                "Level": 47,
                "Attack_1": "297",
                "Attack_2": "17",
                "Attack_3": "16",
                "Attack_4": "98",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Keen Eye"
            },
            {
                "name": "Rhyhorn",
                "Level": 45,
                "Attack_1": "36",
                "Attack_2": "32",
                "Attack_3": "350",
                "Attack_4": "31",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Lightningrod"
            },
            {
                "name": "Growlithe",
                "Level": 45,
                "Attack_1": "172",
                "Attack_2": "36",
                "Attack_3": "43",
                "Attack_4": "97",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Intimidate"
            },
            {
                "name": "Gyarados",
                "Level": 45,
                "Attack_1": "76",
                "Attack_2": "79",
                "Attack_3": "77",
                "Attack_4": "78",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Intimidate"
            },
            {
                "name": "Alakazam",
                "Level": 47,
                "Attack_1": "94",
                "Attack_2": "347",
                "Attack_3": "248",
                "Attack_4": "50",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Synchronize"
            },
            {
                "name": "Venusaur",
                "Level": 53,
                "Attack_1": "75",
                "Attack_2": "230",
                "Attack_3": "74",
                "Attack_4": "235",
                "IVS": {
                    "hp": 30,
                    "atk": 30,
                    "def": 30,
                    "spa": 30,
                    "spd": 30,
                    "spe": 30
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Overgrow"
            }
        ],
        "22 Rival 2(Charmander)": [
            {
                "name": "Pidgeot",
                "Level": 47,
                "Attack_1": "297",
                "Attack_2": "17",
                "Attack_3": "16",
                "Attack_4": "98",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Keen Eye"
            },
            {
                "name": "Rhyhorn",
                "Level": 45,
                "Attack_1": "36",
                "Attack_2": "32",
                "Attack_3": "350",
                "Attack_4": "31",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Lightningrod"
            },
            {
                "name": "Growlithe",
                "Level": 45,
                "Attack_1": "172",
                "Attack_2": "36",
                "Attack_3": "43",
                "Attack_4": "97",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Intimidate"
            },
            {
                "name": "Exeggcute",
                "Level": 45,
                "Attack_1": "76",
                "Attack_2": "79",
                "Attack_3": "77",
                "Attack_4": "78",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Alakazam",
                "Level": 47,
                "Attack_1": "94",
                "Attack_2": "347",
                "Attack_3": "248",
                "Attack_4": "50",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Synchronize"
            },
            {
                "name": "Blastoise",
                "Level": 53,
                "Attack_1": "55",
                "Attack_2": "240",
                "Attack_3": "44",
                "Attack_4": "229",
                "IVS": {
                    "hp": 30,
                    "atk": 30,
                    "def": 30,
                    "spa": 30,
                    "spd": 30,
                    "spe": 30
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Torrent"
            }
        ],
        "Cooltrainer Naomi": [
            {
                "name": "Persian",
                "Level": 42,
                "Attack_1": "154",
                "Attack_2": "103",
                "Attack_3": "185",
                "Attack_4": "6",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Limber"
            },
            {
                "name": "Ponyta",
                "Level": 42,
                "Attack_1": "97",
                "Attack_2": "36",
                "Attack_3": "83",
                "Attack_4": "23",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Run Away"
            },
            {
                "name": "Rapidash",
                "Level": 42,
                "Attack_1": "31",
                "Attack_2": "83",
                "Attack_3": "23",
                "Attack_4": "45",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Run Away"
            },
            {
                "name": "Vulpix",
                "Level": 42,
                "Attack_1": "53",
                "Attack_2": "109",
                "Attack_3": "98",
                "Attack_4": "286",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Flash Fire"
            },
            {
                "name": "Ninetales",
                "Level": 42,
                "Attack_1": "219",
                "Attack_2": "261",
                "Attack_3": "109",
                "Attack_4": "83",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Flash Fire"
            }
        ],
        "Cooltrainer Rolando": [
            {
                "name": "Raticate",
                "Level": 42,
                "Attack_1": "162",
                "Attack_2": "228",
                "Attack_3": "184",
                "Attack_4": "98",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Run Away"
            },
            {
                "name": "Ivysaur",
                "Level": 42,
                "Attack_1": "75",
                "Attack_2": "79",
                "Attack_3": "230",
                "Attack_4": "235",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Overgrow"
            },
            {
                "name": "Wartortle",
                "Level": 42,
                "Attack_1": "55",
                "Attack_2": "44",
                "Attack_3": "229",
                "Attack_4": "39",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Torrent"
            },
            {
                "name": "Charmeleon",
                "Level": 42,
                "Attack_1": "53",
                "Attack_2": "163",
                "Attack_3": "108",
                "Attack_4": "184",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Blaze"
            },
            {
                "name": "Charizard",
                "Level": 42,
                "Attack_1": "53",
                "Attack_2": "17",
                "Attack_3": "108",
                "Attack_4": "184",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Blaze"
            }
        ],
        "Black Belt Daisuke": [
            {
                "name": "Machoke",
                "Level": 43,
                "Attack_1": "193",
                "Attack_2": "279",
                "Attack_3": "233",
                "Attack_4": "66",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Guts"
            },
            {
                "name": "Machop",
                "Level": 43,
                "Attack_1": "233",
                "Attack_2": "66",
                "Attack_3": "238",
                "Attack_4": "184",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Guts"
            },
            {
                "name": "Machoke",
                "Level": 43,
                "Attack_1": "193",
                "Attack_2": "279",
                "Attack_3": "233",
                "Attack_4": "66",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Guts"
            }
        ],
        "Juggler Nelson": [
            {
                "name": "Drowzee",
                "Level": 41,
                "Attack_1": "96",
                "Attack_2": "94",
                "Attack_3": "244",
                "Attack_4": "207",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Insomnia"
            },
            {
                "name": "Hypno",
                "Level": 41,
                "Attack_1": "29",
                "Attack_2": "139",
                "Attack_3": "96",
                "Attack_4": "94",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Insomnia"
            },
            {
                "name": "Kadabra",
                "Level": 41,
                "Attack_1": "105",
                "Attack_2": "248",
                "Attack_3": "272",
                "Attack_4": "94",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Synchronize"
            },
            {
                "name": "Kadabra",
                "Level": 41,
                "Attack_1": "105",
                "Attack_2": "248",
                "Attack_3": "272",
                "Attack_4": "94",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Synchronize"
            }
        ],
        "Tamer Vincent": [
            {
                "name": "Persian",
                "Level": 44,
                "Attack_1": "6",
                "Attack_2": "185",
                "Attack_3": "103",
                "Attack_4": "154",
                "IVS": {
                    "hp": 4,
                    "atk": 4,
                    "def": 4,
                    "spa": 4,
                    "spd": 4,
                    "spe": 4
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Limber"
            },
            {
                "name": "Golduck",
                "Level": 44,
                "Attack_1": "93",
                "Attack_2": "103",
                "Attack_3": "244",
                "Attack_4": "154",
                "IVS": {
                    "hp": 4,
                    "atk": 4,
                    "def": 4,
                    "spa": 4,
                    "spd": 4,
                    "spe": 4
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Damp"
            }
        ],
        "Juggler Gregory": [
            {
                "name": "Mr. Mime",
                "Level": 48,
                "Attack_1": "94",
                "Attack_2": "272",
                "Attack_3": "3",
                "Attack_4": "227",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Soundproof"
            }
        ],
        "Cooltrainer George": [
            {
                "name": "Exeggutor",
                "Level": 42,
                "Attack_1": "121",
                "Attack_2": "93",
                "Attack_3": "78",
                "Attack_4": "79",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Sandslash",
                "Level": 42,
                "Attack_1": "154",
                "Attack_2": "129",
                "Attack_3": "40",
                "Attack_4": "28",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Sand Veil"
            },
            {
                "name": "Cloyster",
                "Level": 42,
                "Attack_1": "131",
                "Attack_2": "191",
                "Attack_3": "62",
                "Attack_4": "48",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Shell Armor"
            },
            {
                "name": "Electrode",
                "Level": 42,
                "Attack_1": "209",
                "Attack_2": "49",
                "Attack_3": "103",
                "Attack_4": "113",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Soundproof"
            },
            {
                "name": "Arcanine",
                "Level": 42,
                "Attack_1": "172",
                "Attack_2": "46",
                "Attack_3": "44",
                "Attack_4": "36",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Intimidate"
            }
        ],
        "Pokemaniac Dawson": [
            {
                "name": "Charmeleon",
                "Level": 40,
                "Attack_1": "232",
                "Attack_2": "108",
                "Attack_3": "184",
                "Attack_4": "53",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Blaze"
            },
            {
                "name": "Lapras",
                "Level": 40,
                "Attack_1": "109",
                "Attack_2": "195",
                "Attack_3": "58",
                "Attack_4": "240",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Water Absorb"
            },
            {
                "name": "Lickitung",
                "Level": 40,
                "Attack_1": "23",
                "Attack_2": "35",
                "Attack_3": "50",
                "Attack_4": "21",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Own Tempo"
            }
        ],
        "Cooltrainer Alexa": [
            {
                "name": "Clefairy",
                "Level": 42,
                "Attack_1": "113",
                "Attack_2": "322",
                "Attack_3": "3",
                "Attack_4": "227",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Cute Charm"
            },
            {
                "name": "Jigglypuff",
                "Level": 42,
                "Attack_1": "47",
                "Attack_2": "34",
                "Attack_3": "205",
                "Attack_4": "50",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Cute Charm"
            },
            {
                "name": "Persian",
                "Level": 42,
                "Attack_1": "6",
                "Attack_2": "185",
                "Attack_3": "44",
                "Attack_4": "103",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Limber"
            },
            {
                "name": "Dewgong",
                "Level": 42,
                "Attack_1": "329",
                "Attack_2": "36",
                "Attack_3": "156",
                "Attack_4": "62",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Thick Fat"
            },
            {
                "name": "Chansey",
                "Level": 42,
                "Attack_1": "47",
                "Attack_2": "121",
                "Attack_3": "135",
                "Attack_4": "107",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Natural Cure"
            }
        ],
        "Cooltrainer Colby": [
            {
                "name": "Kingler",
                "Level": 41,
                "Attack_1": "12",
                "Attack_2": "23",
                "Attack_3": "341",
                "Attack_4": "145",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Hyper Cutter"
            },
            {
                "name": "Poliwhirl",
                "Level": 42,
                "Attack_1": "34",
                "Attack_2": "3",
                "Attack_3": "55",
                "Attack_4": "95",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Water Absorb"
            },
            {
                "name": "Tentacruel",
                "Level": 42,
                "Attack_1": "112",
                "Attack_2": "35",
                "Attack_3": "61",
                "Attack_4": "51",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Clear Body"
            },
            {
                "name": "Seadra",
                "Level": 42,
                "Attack_1": "55",
                "Attack_2": "108",
                "Attack_3": "239",
                "Attack_4": "43",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Poison Point"
            },
            {
                "name": "Blastoise",
                "Level": 43,
                "Attack_1": "55",
                "Attack_2": "44",
                "Attack_3": "229",
                "Attack_4": "240",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Torrent"
            }
        ],
        "Cooltrainer Caroline": [
            {
                "name": "Bellsprout",
                "Level": 42,
                "Attack_1": "75",
                "Attack_2": "51",
                "Attack_3": "78",
                "Attack_4": "77",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Weepinbell",
                "Level": 42,
                "Attack_1": "75",
                "Attack_2": "51",
                "Attack_3": "79",
                "Attack_4": "77",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Victreebel",
                "Level": 42,
                "Attack_1": "75",
                "Attack_2": "51",
                "Attack_3": "78",
                "Attack_4": "79",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Paras",
                "Level": 42,
                "Attack_1": "74",
                "Attack_2": "163",
                "Attack_3": "141",
                "Attack_4": "78",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Effect Spore"
            },
            {
                "name": "Parasect",
                "Level": 42,
                "Attack_1": "147",
                "Attack_2": "163",
                "Attack_3": "141",
                "Attack_4": "77",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Effect Spore"
            }
        ],
        "Cool Couple Ray & Tyra": [
            {
                "name": "Nidoking",
                "Level": 45,
                "Attack_1": "224",
                "Attack_2": "37",
                "Attack_3": "24",
                "Attack_4": "40",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Poison Point"
            },
            {
                "name": "Nidoqueen",
                "Level": 45,
                "Attack_1": "276",
                "Attack_2": "34",
                "Attack_3": "24",
                "Attack_4": "40",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Poison Point"
            }
        ],
        "Lorelei 1": [
            {
                "name": "Dewgong",
                "Level": 52,
                "Attack_1": "58",
                "Attack_2": "57",
                "Attack_3": "258",
                "Attack_4": "219",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Thick Fat"
            },
            {
                "name": "Cloyster",
                "Level": 51,
                "Attack_1": "191",
                "Attack_2": "182",
                "Attack_3": "258",
                "Attack_4": "291",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Shell Armor"
            },
            {
                "name": "Slowbro",
                "Level": 52,
                "Attack_1": "58",
                "Attack_2": "57",
                "Attack_3": "133",
                "Attack_4": "281",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Oblivious"
            },
            {
                "name": "Jynx",
                "Level": 54,
                "Attack_1": "8",
                "Attack_2": "3",
                "Attack_3": "142",
                "Attack_4": "213",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Oblivious"
            },
            {
                "name": "Lapras",
                "Level": 54,
                "Attack_1": "109",
                "Attack_2": "58",
                "Attack_3": "57",
                "Attack_4": "34",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Water Absorb"
            }
        ],
        "Bruno 1": [
            {
                "name": "Onix",
                "Level": 51,
                "Attack_1": "89",
                "Attack_2": "317",
                "Attack_3": "231",
                "Attack_4": "46",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Rock Head"
            },
            {
                "name": "Hitmonchan",
                "Level": 53,
                "Attack_1": "327",
                "Attack_2": "183",
                "Attack_3": "317",
                "Attack_4": "68",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Keen Eye"
            },
            {
                "name": "Hitmonlee",
                "Level": 53,
                "Attack_1": "25",
                "Attack_2": "193",
                "Attack_3": "280",
                "Attack_4": "263",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Limber"
            },
            {
                "name": "Onix",
                "Level": 54,
                "Attack_1": "38",
                "Attack_2": "89",
                "Attack_3": "231",
                "Attack_4": "328",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Rock Head"
            },
            {
                "name": "Machamp",
                "Level": 56,
                "Attack_1": "238",
                "Attack_2": "339",
                "Attack_3": "184",
                "Attack_4": "317",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Guts"
            }
        ],
        "Agatha 1": [
            {
                "name": "Gengar",
                "Level": 54,
                "Attack_1": "325",
                "Attack_2": "109",
                "Attack_3": "92",
                "Attack_4": "104",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Levitate"
            },
            {
                "name": "Golbat",
                "Level": 54,
                "Attack_1": "109",
                "Attack_2": "305",
                "Attack_3": "314",
                "Attack_4": "44",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Inner Focus"
            },
            {
                "name": "Haunter",
                "Level": 53,
                "Attack_1": "95",
                "Attack_2": "138",
                "Attack_3": "174",
                "Attack_4": "212",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Levitate"
            },
            {
                "name": "Arbok",
                "Level": 56,
                "Attack_1": "188",
                "Attack_2": "103",
                "Attack_3": "231",
                "Attack_4": "44",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Intimidate"
            },
            {
                "name": "Gengar",
                "Level": 58,
                "Attack_1": "247",
                "Attack_2": "188",
                "Attack_3": "95",
                "Attack_4": "171",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Levitate"
            }
        ],
        "Lance 1": [
            {
                "name": "Gyarados",
                "Level": 56,
                "Attack_1": "63",
                "Attack_2": "82",
                "Attack_3": "239",
                "Attack_4": "44",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Intimidate"
            },
            {
                "name": "Dragonair",
                "Level": 54,
                "Attack_1": "63",
                "Attack_2": "219",
                "Attack_3": "82",
                "Attack_4": "200",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Shed Skin"
            },
            {
                "name": "Dragonair",
                "Level": 54,
                "Attack_1": "63",
                "Attack_2": "219",
                "Attack_3": "86",
                "Attack_4": "200",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Shed Skin"
            },
            {
                "name": "Aerodactyl",
                "Level": 58,
                "Attack_1": "63",
                "Attack_2": "246",
                "Attack_3": "17",
                "Attack_4": "184",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Rock Head"
            },
            {
                "name": "Dragonite",
                "Level": 60,
                "Attack_1": "63",
                "Attack_2": "219",
                "Attack_3": "200",
                "Attack_4": "17",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Inner Focus"
            }
        ],
        "Champ 1(Bulbasaur)": [
            {
                "name": "Pidgeot",
                "Level": 59,
                "Attack_1": "332",
                "Attack_2": "297",
                "Attack_3": "28",
                "Attack_4": "18",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Keen Eye"
            },
            {
                "name": "Alakazam",
                "Level": 59,
                "Attack_1": "94",
                "Attack_2": "248",
                "Attack_3": "105",
                "Attack_4": "115",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Synchronize"
            },
            {
                "name": "Rhydon",
                "Level": 57,
                "Attack_1": "36",
                "Attack_2": "89",
                "Attack_3": "317",
                "Attack_4": "184",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Lightningrod"
            },
            {
                "name": "Gyarados",
                "Level": 61,
                "Attack_1": "56",
                "Attack_2": "82",
                "Attack_3": "44",
                "Attack_4": "37",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Intimidate"
            },
            {
                "name": "Exeggutor",
                "Level": 59,
                "Attack_1": "202",
                "Attack_2": "121",
                "Attack_3": "79",
                "Attack_4": "113",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Charizard",
                "Level": 63,
                "Attack_1": "126",
                "Attack_2": "332",
                "Attack_3": "163",
                "Attack_4": "83",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Blaze"
            }
        ],
        "Champ 1(Squirtle)": [
            {
                "name": "Pidgeot",
                "Level": 59,
                "Attack_1": "332",
                "Attack_2": "297",
                "Attack_3": "28",
                "Attack_4": "18",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Keen Eye"
            },
            {
                "name": "Alakazam",
                "Level": 59,
                "Attack_1": "94",
                "Attack_2": "248",
                "Attack_3": "105",
                "Attack_4": "115",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Synchronize"
            },
            {
                "name": "Rhydon",
                "Level": 57,
                "Attack_1": "36",
                "Attack_2": "89",
                "Attack_3": "317",
                "Attack_4": "184",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Lightningrod"
            },
            {
                "name": "Arcanine",
                "Level": 61,
                "Attack_1": "245",
                "Attack_2": "53",
                "Attack_3": "46",
                "Attack_4": "44",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Intimidate"
            },
            {
                "name": "Gyarados",
                "Level": 59,
                "Attack_1": "56",
                "Attack_2": "82",
                "Attack_3": "44",
                "Attack_4": "37",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Intimidate"
            },
            {
                "name": "Venusaur",
                "Level": 63,
                "Attack_1": "76",
                "Attack_2": "235",
                "Attack_3": "241",
                "Attack_4": "74",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Overgrow"
            }
        ],
        "Champ 1(Charmander)": [
            {
                "name": "Pidgeot",
                "Level": 59,
                "Attack_1": "332",
                "Attack_2": "297",
                "Attack_3": "28",
                "Attack_4": "18",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Keen Eye"
            },
            {
                "name": "Alakazam",
                "Level": 59,
                "Attack_1": "94",
                "Attack_2": "248",
                "Attack_3": "105",
                "Attack_4": "115",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Synchronize"
            },
            {
                "name": "Rhydon",
                "Level": 57,
                "Attack_1": "36",
                "Attack_2": "89",
                "Attack_3": "317",
                "Attack_4": "184",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Lightningrod"
            },
            {
                "name": "Arcanine",
                "Level": 61,
                "Attack_1": "245",
                "Attack_2": "53",
                "Attack_3": "46",
                "Attack_4": "44",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Intimidate"
            },
            {
                "name": "Exeggutor",
                "Level": 59,
                "Attack_1": "202",
                "Attack_2": "121",
                "Attack_3": "79",
                "Attack_4": "113",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Blastoise",
                "Level": 63,
                "Attack_1": "56",
                "Attack_2": "240",
                "Attack_3": "130",
                "Attack_4": "44",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Torrent"
            }
        ],
        "Team Rocket Grunt 43": [
            {
                "name": "Zubat",
                "Level": 38,
                "Attack_1": "17",
                "Attack_2": "109",
                "Attack_3": "314",
                "Attack_4": "212",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Inner Focus"
            },
            {
                "name": "Zubat",
                "Level": 38,
                "Attack_1": "17",
                "Attack_2": "109",
                "Attack_3": "314",
                "Attack_4": "212",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Inner Focus"
            },
            {
                "name": "Golbat",
                "Level": 38,
                "Attack_1": "44",
                "Attack_2": "17",
                "Attack_3": "109",
                "Attack_4": "314",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Inner Focus"
            }
        ],
        "Pokemon Breeder Alize": [
            {
                "name": "Pikachu",
                "Level": 48,
                "Attack_1": "21",
                "Attack_2": "85",
                "Attack_3": "97",
                "Attack_4": "87",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Static"
            },
            {
                "name": "Clefairy",
                "Level": 48,
                "Attack_1": "322",
                "Attack_2": "236",
                "Attack_3": "113",
                "Attack_4": "309",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Cute Charm"
            },
            {
                "name": "Marill",
                "Level": 48,
                "Attack_1": "61",
                "Attack_2": "38",
                "Attack_3": "240",
                "Attack_4": "56",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Thick Fat"
            }
        ],
        "Painter Rayna": [
            {
                "name": "Smeargle",
                "Level": 50,
                "Attack_1": "238",
                "Attack_2": "224",
                "Attack_3": "38",
                "Attack_4": "120",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Own Tempo"
            }
        ],
        "Lady Jacki": [
            {
                "name": "Hoppip",
                "Level": 48,
                "Attack_1": "79",
                "Attack_2": "73",
                "Attack_3": "178",
                "Attack_4": "72",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Skiploom",
                "Level": 50,
                "Attack_1": "79",
                "Attack_2": "73",
                "Attack_3": "178",
                "Attack_4": "72",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Chlorophyll"
            }
        ],
        "Painter Celina": [
            {
                "name": "Smeargle",
                "Level": 50,
                "Attack_1": "19",
                "Attack_2": "91",
                "Attack_3": "291",
                "Attack_4": "340",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Own Tempo"
            }
        ],
        "Lady Gillian": [
            {
                "name": "Mareep",
                "Level": 47,
                "Attack_1": "86",
                "Attack_2": "178",
                "Attack_3": "113",
                "Attack_4": "87",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Static"
            },
            {
                "name": "Mareep",
                "Level": 48,
                "Attack_1": "86",
                "Attack_2": "178",
                "Attack_3": "113",
                "Attack_4": "87",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Static"
            },
            {
                "name": "Flaaffy",
                "Level": 49,
                "Attack_1": "86",
                "Attack_2": "178",
                "Attack_3": "113",
                "Attack_4": "87",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Static"
            }
        ],
        "Youngster Destin": [
            {
                "name": "Raticate",
                "Level": 48,
                "Attack_1": "158",
                "Attack_2": "184",
                "Attack_3": "228",
                "Attack_4": "162",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Run Away"
            },
            {
                "name": "Pidgeotto",
                "Level": 48,
                "Attack_1": "18",
                "Attack_2": "17",
                "Attack_3": "297",
                "Attack_4": "97",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Keen Eye"
            }
        ],
        "Painter Daisy": [
            {
                "name": "Smeargle",
                "Level": 50,
                "Attack_1": "223",
                "Attack_2": "146",
                "Attack_3": "264",
                "Attack_4": "5",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Own Tempo"
            }
        ],
        "Swimmer Toby": [
            {
                "name": "Poliwhirl",
                "Level": 48,
                "Attack_1": "3",
                "Attack_2": "240",
                "Attack_3": "34",
                "Attack_4": "187",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Water Absorb"
            },
            {
                "name": "Tentacool",
                "Level": 48,
                "Attack_1": "61",
                "Attack_2": "35",
                "Attack_3": "112",
                "Attack_4": "103",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Clear Body"
            },
            {
                "name": "Tentacruel",
                "Level": 48,
                "Attack_1": "61",
                "Attack_2": "35",
                "Attack_3": "112",
                "Attack_4": "103",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Clear Body"
            }
        ],
        "Ruin Maniac Lawson": [
            {
                "name": "Onix",
                "Level": 47,
                "Attack_1": "225",
                "Attack_2": "201",
                "Attack_3": "258",
                "Attack_4": "231",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Rock Head"
            },
            {
                "name": "Graveler",
                "Level": 48,
                "Attack_1": "120",
                "Attack_2": "205",
                "Attack_3": "350",
                "Attack_4": "89",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Rock Head"
            },
            {
                "name": "Marowak",
                "Level": 49,
                "Attack_1": "155",
                "Attack_2": "99",
                "Attack_3": "206",
                "Attack_4": "37",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Rock Head"
            }
        ],
        "Psychic Laura": [
            {
                "name": "Natu",
                "Level": 48,
                "Attack_1": "100",
                "Attack_2": "273",
                "Attack_3": "248",
                "Attack_4": "109",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Synchronize"
            },
            {
                "name": "Natu",
                "Level": 48,
                "Attack_1": "100",
                "Attack_2": "273",
                "Attack_3": "248",
                "Attack_4": "109",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Synchronize"
            },
            {
                "name": "Xatu",
                "Level": 49,
                "Attack_1": "101",
                "Attack_2": "100",
                "Attack_3": "273",
                "Attack_4": "248",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Synchronize"
            }
        ],
        "Lady Selphy": [
            {
                "name": "Persian",
                "Level": 49,
                "Attack_1": "6",
                "Attack_2": "44",
                "Attack_3": "269",
                "Attack_4": "259",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Limber"
            },
            {
                "name": "Persian",
                "Level": 49,
                "Attack_1": "6",
                "Attack_2": "10",
                "Attack_3": "259",
                "Attack_4": "269",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Limber"
            }
        ],
        "Team Rocket Grunt 44": [
            {
                "name": "Rattata",
                "Level": 48,
                "Attack_1": "116",
                "Attack_2": "228",
                "Attack_3": "162",
                "Attack_4": "283",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Run Away"
            },
            {
                "name": "Grimer",
                "Level": 48,
                "Attack_1": "107",
                "Attack_2": "103",
                "Attack_3": "151",
                "Attack_4": "188",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Stench"
            },
            {
                "name": "Muk",
                "Level": 48,
                "Attack_1": "107",
                "Attack_2": "103",
                "Attack_3": "151",
                "Attack_4": "188",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Stench"
            }
        ],
        "Team Rocket Grunt 45": [
            {
                "name": "Ekans",
                "Level": 48,
                "Attack_1": "254",
                "Attack_2": "256",
                "Attack_3": "255",
                "Attack_4": "114",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Intimidate"
            },
            {
                "name": "Gloom",
                "Level": 48,
                "Attack_1": "79",
                "Attack_2": "51",
                "Attack_3": "236",
                "Attack_4": "80",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Gloom",
                "Level": 48,
                "Attack_1": "79",
                "Attack_2": "51",
                "Attack_3": "236",
                "Attack_4": "80",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Chlorophyll"
            }
        ],
        "Team Rocket Grunt 46": [
            {
                "name": "Koffing",
                "Level": 49,
                "Attack_1": "262",
                "Attack_2": "114",
                "Attack_3": "108",
                "Attack_4": "124",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Levitate"
            },
            {
                "name": "Weezing",
                "Level": 49,
                "Attack_1": "114",
                "Attack_2": "108",
                "Attack_3": "124",
                "Attack_4": "120",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Levitate"
            }
        ],
        "Bird Keeper Milo": [
            {
                "name": "Pidgey",
                "Level": 47,
                "Attack_1": "17",
                "Attack_2": "297",
                "Attack_3": "97",
                "Attack_4": "119",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Keen Eye"
            },
            {
                "name": "Pidgeotto",
                "Level": 49,
                "Attack_1": "18",
                "Attack_2": "17",
                "Attack_3": "297",
                "Attack_4": "97",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Keen Eye"
            }
        ],
        "Bird Keeper Chaz": [
            {
                "name": "Spearow",
                "Level": 47,
                "Attack_1": "332",
                "Attack_2": "119",
                "Attack_3": "65",
                "Attack_4": "97",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Keen Eye"
            },
            {
                "name": "Fearow",
                "Level": 49,
                "Attack_1": "228",
                "Attack_2": "119",
                "Attack_3": "65",
                "Attack_4": "97",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Keen Eye"
            }
        ],
        "Bird Keeper Harold": [
            {
                "name": "Hoothoot",
                "Level": 47,
                "Attack_1": "95",
                "Attack_2": "115",
                "Attack_3": "36",
                "Attack_4": "93",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Insomnia"
            },
            {
                "name": "Noctowl",
                "Level": 49,
                "Attack_1": "95",
                "Attack_2": "115",
                "Attack_3": "36",
                "Attack_4": "93",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Insomnia"
            }
        ],
        "Juggler Edward": [
            {
                "name": "Voltorb",
                "Level": 46,
                "Attack_1": "129",
                "Attack_2": "113",
                "Attack_3": "209",
                "Attack_4": "49",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Soundproof"
            },
            {
                "name": "Voltorb",
                "Level": 46,
                "Attack_1": "129",
                "Attack_2": "113",
                "Attack_3": "209",
                "Attack_4": "49",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Soundproof"
            },
            {
                "name": "Electrode",
                "Level": 47,
                "Attack_1": "129",
                "Attack_2": "209",
                "Attack_3": "120",
                "Attack_4": "49",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Soundproof"
            },
            {
                "name": "Mr. Mime",
                "Level": 48,
                "Attack_1": "94",
                "Attack_2": "272",
                "Attack_3": "115",
                "Attack_4": "227",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Soundproof"
            }
        ],
        "Swimmer Denise": [
            {
                "name": "Chinchou",
                "Level": 49,
                "Attack_1": "109",
                "Attack_2": "36",
                "Attack_3": "56",
                "Attack_4": "268",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Volt Absorb"
            },
            {
                "name": "Lanturn",
                "Level": 49,
                "Attack_1": "55",
                "Attack_2": "209",
                "Attack_3": "109",
                "Attack_4": "36",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Volt Absorb"
            }
        ],
        "Swimmer Samir": [
            {
                "name": "Gyarados",
                "Level": 50,
                "Attack_1": "239",
                "Attack_2": "56",
                "Attack_3": "240",
                "Attack_4": "349",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Intimidate"
            }
        ],
        "Twins Miu & Mia": [
            {
                "name": "Pikachu",
                "Level": 50,
                "Attack_1": "85",
                "Attack_2": "97",
                "Attack_3": "87",
                "Attack_4": "113",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Static"
            },
            {
                "name": "Pikachu",
                "Level": 50,
                "Attack_1": "85",
                "Attack_2": "97",
                "Attack_3": "87",
                "Attack_4": "113",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Static"
            }
        ],
        "Aroma Lady Rose": [
            {
                "name": "Sunkern",
                "Level": 49,
                "Attack_1": "283",
                "Attack_2": "241",
                "Attack_3": "235",
                "Attack_4": "202",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Sunflora",
                "Level": 49,
                "Attack_1": "331",
                "Attack_2": "241",
                "Attack_3": "80",
                "Attack_4": "76",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Chlorophyll"
            }
        ],
        "Hiker Earl": [
            {
                "name": "Onix",
                "Level": 49,
                "Attack_1": "225",
                "Attack_2": "201",
                "Attack_3": "21",
                "Attack_4": "231",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Rock Head"
            },
            {
                "name": "Machoke",
                "Level": 49,
                "Attack_1": "279",
                "Attack_2": "233",
                "Attack_3": "66",
                "Attack_4": "238",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Guts"
            }
        ],
        "Youngster Cordell": [
            {
                "name": "Farfetch'd",
                "Level": 48,
                "Attack_1": "14",
                "Attack_2": "97",
                "Attack_3": "163",
                "Attack_4": "206",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Keen Eye"
            },
            {
                "name": "Farfetch'd",
                "Level": 48,
                "Attack_1": "14",
                "Attack_2": "97",
                "Attack_3": "163",
                "Attack_4": "206",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Keen Eye"
            }
        ],
        "Pokemon Breeder Bethany": [
            {
                "name": "Chansey",
                "Level": 50,
                "Attack_1": "47",
                "Attack_2": "121",
                "Attack_3": "111",
                "Attack_4": "113",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Natural Cure"
            }
        ],
        "Bug Catcher Garret": [
            {
                "name": "Heracross",
                "Level": 49,
                "Attack_1": "280",
                "Attack_2": "68",
                "Attack_3": "36",
                "Attack_4": "179",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Swarm"
            }
        ],
        "Lass Joana": [
            {
                "name": "Snubbull",
                "Level": 49,
                "Attack_1": "122",
                "Attack_2": "46",
                "Attack_3": "99",
                "Attack_4": "36",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Intimidate"
            }
        ],
        "Youngster Nash": [
            {
                "name": "Weepinbell",
                "Level": 47,
                "Attack_1": "78",
                "Attack_2": "51",
                "Attack_3": "230",
                "Attack_4": "75",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Weepinbell",
                "Level": 47,
                "Attack_1": "78",
                "Attack_2": "51",
                "Attack_3": "230",
                "Attack_4": "75",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Victreebel",
                "Level": 49,
                "Attack_1": "22",
                "Attack_2": "79",
                "Attack_3": "230",
                "Attack_4": "75",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Chlorophyll"
            }
        ],
        "Bug Catcher Vance": [
            {
                "name": "Venonat",
                "Level": 48,
                "Attack_1": "78",
                "Attack_2": "60",
                "Attack_3": "79",
                "Attack_4": "94",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Compoundeyes"
            },
            {
                "name": "Venomoth",
                "Level": 48,
                "Attack_1": "78",
                "Attack_2": "16",
                "Attack_3": "60",
                "Attack_4": "79",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Shield Dust"
            }
        ],
        "Ruin Maniac Layton": [
            {
                "name": "Sandslash",
                "Level": 48,
                "Attack_1": "40",
                "Attack_2": "163",
                "Attack_3": "129",
                "Attack_4": "154",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Sand Veil"
            },
            {
                "name": "Onix",
                "Level": 48,
                "Attack_1": "225",
                "Attack_2": "201",
                "Attack_3": "21",
                "Attack_4": "231",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Rock Head"
            },
            {
                "name": "Sandslash",
                "Level": 48,
                "Attack_1": "40",
                "Attack_2": "163",
                "Attack_3": "129",
                "Attack_4": "154",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Sand Veil"
            }
        ],
        "Picknicker Marcy": [
            {
                "name": "Paras",
                "Level": 48,
                "Attack_1": "147",
                "Attack_2": "163",
                "Attack_3": "74",
                "Attack_4": "202",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Effect Spore"
            },
            {
                "name": "Paras",
                "Level": 48,
                "Attack_1": "147",
                "Attack_2": "163",
                "Attack_3": "74",
                "Attack_4": "202",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Effect Spore"
            },
            {
                "name": "Parasect",
                "Level": 49,
                "Attack_1": "141",
                "Attack_2": "147",
                "Attack_3": "163",
                "Attack_4": "74",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Effect Spore"
            }
        ],
        "Bug Catcher Jonah": [
            {
                "name": "Yanma",
                "Level": 45,
                "Attack_1": "197",
                "Attack_2": "253",
                "Attack_3": "17",
                "Attack_4": "48",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Speed Boost"
            },
            {
                "name": "Beedrill",
                "Level": 45,
                "Attack_1": "228",
                "Attack_2": "42",
                "Attack_3": "97",
                "Attack_4": "283",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Swarm"
            },
            {
                "name": "Yanma",
                "Level": 46,
                "Attack_1": "197",
                "Attack_2": "253",
                "Attack_3": "17",
                "Attack_4": "48",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Speed Boost"
            },
            {
                "name": "Beedrill",
                "Level": 47,
                "Attack_1": "228",
                "Attack_2": "42",
                "Attack_3": "97",
                "Attack_4": "283",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Swarm"
            }
        ],
        "Lass Dalia": [
            {
                "name": "Hoppip",
                "Level": 46,
                "Attack_1": "79",
                "Attack_2": "73",
                "Attack_3": "178",
                "Attack_4": "72",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Hoppip",
                "Level": 47,
                "Attack_1": "79",
                "Attack_2": "73",
                "Attack_3": "178",
                "Attack_4": "72",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Skiploom",
                "Level": 47,
                "Attack_1": "79",
                "Attack_2": "73",
                "Attack_3": "178",
                "Attack_4": "72",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Skiploom",
                "Level": 48,
                "Attack_1": "79",
                "Attack_2": "73",
                "Attack_3": "178",
                "Attack_4": "72",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Chlorophyll"
            }
        ],
        "Pokemon Breeder Allison": [
            {
                "name": "Clefairy",
                "Level": 48,
                "Attack_1": "322",
                "Attack_2": "236",
                "Attack_3": "113",
                "Attack_4": "309",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Cute Charm"
            },
            {
                "name": "Clefairy",
                "Level": 48,
                "Attack_1": "322",
                "Attack_2": "236",
                "Attack_3": "113",
                "Attack_4": "309",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Cute Charm"
            },
            {
                "name": "Clefable",
                "Level": 48,
                "Attack_1": "47",
                "Attack_2": "3",
                "Attack_3": "107",
                "Attack_4": "118",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Cute Charm"
            }
        ],
        "Camper Riley": [
            {
                "name": "Pinsir",
                "Level": 49,
                "Attack_1": "280",
                "Attack_2": "12",
                "Attack_3": "66",
                "Attack_4": "14",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Hyper Cutter"
            },
            {
                "name": "Heracross",
                "Level": 50,
                "Attack_1": "280",
                "Attack_2": "68",
                "Attack_3": "36",
                "Attack_4": "179",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Swarm"
            }
        ],
        "Psychic Jaclyn": [
            {
                "name": "Natu",
                "Level": 48,
                "Attack_1": "101",
                "Attack_2": "109",
                "Attack_3": "248",
                "Attack_4": "273",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Synchronize"
            },
            {
                "name": "Slowbro",
                "Level": 48,
                "Attack_1": "94",
                "Attack_2": "29",
                "Attack_3": "133",
                "Attack_4": "281",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Oblivious"
            },
            {
                "name": "Kadabra",
                "Level": 49,
                "Attack_1": "94",
                "Attack_2": "248",
                "Attack_3": "105",
                "Attack_4": "115",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Synchronize"
            }
        ],
        "Swimmer Nicole": [
            {
                "name": "Marill",
                "Level": 50,
                "Attack_1": "61",
                "Attack_2": "38",
                "Attack_3": "240",
                "Attack_4": "56",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Thick Fat"
            }
        ],
        "Sis and Bro Ava & Geb": [
            {
                "name": "Starmie",
                "Level": 50,
                "Attack_1": "229",
                "Attack_2": "105",
                "Attack_3": "129",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Illuminate"
            },
            {
                "name": "Poliwhirl",
                "Level": 50,
                "Attack_1": "3",
                "Attack_2": "240",
                "Attack_3": "34",
                "Attack_4": "187",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Water Absorb"
            }
        ],
        "Swimmer Mymo": [
            {
                "name": "Kingler",
                "Level": 49,
                "Attack_1": "341",
                "Attack_2": "23",
                "Attack_3": "12",
                "Attack_4": "182",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Hyper Cutter"
            },
            {
                "name": "Wartortle",
                "Level": 49,
                "Attack_1": "229",
                "Attack_2": "182",
                "Attack_3": "240",
                "Attack_4": "130",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Torrent"
            }
        ],
        "Fisherman Tylor": [
            {
                "name": "Qwilfish",
                "Level": 49,
                "Attack_1": "279",
                "Attack_2": "36",
                "Attack_3": "56",
                "Attack_4": "194",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Poison Point"
            },
            {
                "name": "Qwilfish",
                "Level": 49,
                "Attack_1": "279",
                "Attack_2": "36",
                "Attack_3": "56",
                "Attack_4": "194",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Poison Point"
            }
        ],
        "Team Rocket Grunt 47": [
            {
                "name": "Muk",
                "Level": 48,
                "Attack_1": "107",
                "Attack_2": "103",
                "Attack_3": "151",
                "Attack_4": "188",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Stench"
            },
            {
                "name": "Golbat",
                "Level": 48,
                "Attack_1": "17",
                "Attack_2": "109",
                "Attack_3": "314",
                "Attack_4": "212",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Inner Focus"
            },
            {
                "name": "Raticate",
                "Level": 48,
                "Attack_1": "158",
                "Attack_2": "184",
                "Attack_3": "228",
                "Attack_4": "162",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Run Away"
            }
        ],
        "Hiker Daryl": [
            {
                "name": "Sudowoodo",
                "Level": 50,
                "Attack_1": "157",
                "Attack_2": "335",
                "Attack_3": "185",
                "Attack_4": "21",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Sturdy"
            }
        ],
        "Pokemaniac Hector": [
            {
                "name": "Rhyhorn",
                "Level": 49,
                "Attack_1": "184",
                "Attack_2": "350",
                "Attack_3": "32",
                "Attack_4": "36",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Lightningrod"
            },
            {
                "name": "Kangaskhan",
                "Level": 49,
                "Attack_1": "99",
                "Attack_2": "203",
                "Attack_3": "146",
                "Attack_4": "179",
                "IVS": {
                    "hp": 3,
                    "atk": 3,
                    "def": 3,
                    "spa": 3,
                    "spd": 3,
                    "spe": 3
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Early Bird"
            }
        ],
        "Ruin Maniac Stanly": [
            {
                "name": "Graveler",
                "Level": 48,
                "Attack_1": "120",
                "Attack_2": "205",
                "Attack_3": "350",
                "Attack_4": "89",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Rock Head"
            },
            {
                "name": "Onix",
                "Level": 48,
                "Attack_1": "225",
                "Attack_2": "201",
                "Attack_3": "21",
                "Attack_4": "231",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Rock Head"
            },
            {
                "name": "Graveler",
                "Level": 48,
                "Attack_1": "120",
                "Attack_2": "205",
                "Attack_3": "350",
                "Attack_4": "89",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Rock Head"
            }
        ],
        "Ruin Maniac Foster": [
            {
                "name": "Golem",
                "Level": 50,
                "Attack_1": "120",
                "Attack_2": "205",
                "Attack_3": "350",
                "Attack_4": "89",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Rock Head"
            }
        ],
        "Ruin Maniac Larry": [
            {
                "name": "Machoke",
                "Level": 49,
                "Attack_1": "279",
                "Attack_2": "233",
                "Attack_3": "66",
                "Attack_4": "238",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Guts"
            },
            {
                "name": "Machoke",
                "Level": 49,
                "Attack_1": "279",
                "Attack_2": "233",
                "Attack_3": "66",
                "Attack_4": "238",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Guts"
            }
        ],
        "Psychic Rodette": [
            {
                "name": "Natu",
                "Level": 48,
                "Attack_1": "101",
                "Attack_2": "109",
                "Attack_3": "273",
                "Attack_4": "248",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Synchronize"
            },
            {
                "name": "Drowzee",
                "Level": 48,
                "Attack_1": "94",
                "Attack_2": "50",
                "Attack_3": "244",
                "Attack_4": "248",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Insomnia"
            },
            {
                "name": "Hypno",
                "Level": 50,
                "Attack_1": "94",
                "Attack_2": "95",
                "Attack_3": "244",
                "Attack_4": "248",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Insomnia"
            }
        ],
        "Psychic Dario": [
            {
                "name": "Girafarig",
                "Level": 52,
                "Attack_1": "242",
                "Attack_2": "60",
                "Attack_3": "316",
                "Attack_4": "97",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Inner Focus"
            }
        ],
        "Aroma Lady Miah": [
            {
                "name": "Bellossom",
                "Level": 50,
                "Attack_1": "230",
                "Attack_2": "78",
                "Attack_3": "345",
                "Attack_4": "80",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Bellossom",
                "Level": 50,
                "Attack_1": "230",
                "Attack_2": "78",
                "Attack_3": "345",
                "Attack_4": "80",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Chlorophyll"
            }
        ],
        "Juggler Mason": [
            {
                "name": "Voltorb",
                "Level": 47,
                "Attack_1": "129",
                "Attack_2": "113",
                "Attack_3": "209",
                "Attack_4": "49",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Soundproof"
            },
            {
                "name": "Voltorb",
                "Level": 47,
                "Attack_1": "129",
                "Attack_2": "113",
                "Attack_3": "209",
                "Attack_4": "49",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Soundproof"
            },
            {
                "name": "Pineco",
                "Level": 47,
                "Attack_1": "191",
                "Attack_2": "117",
                "Attack_3": "229",
                "Attack_4": "36",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Sturdy"
            },
            {
                "name": "Pineco",
                "Level": 47,
                "Attack_1": "191",
                "Attack_2": "117",
                "Attack_3": "229",
                "Attack_4": "36",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Sturdy"
            }
        ],
        "Pokemon Ranger Nicolas": [
            {
                "name": "Weepinbell",
                "Level": 51,
                "Attack_1": "75",
                "Attack_2": "51",
                "Attack_3": "230",
                "Attack_4": "35",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Victreebel",
                "Level": 51,
                "Attack_1": "75",
                "Attack_2": "51",
                "Attack_3": "79",
                "Attack_4": "78",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bold",
                "Ability": "Chlorophyll"
            }
        ],
        "Pokemon Ranger Madeline": [
            {
                "name": "Gloom",
                "Level": 51,
                "Attack_1": "80",
                "Attack_2": "51",
                "Attack_3": "230",
                "Attack_4": "77",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Vileplume",
                "Level": 51,
                "Attack_1": "80",
                "Attack_2": "236",
                "Attack_3": "51",
                "Attack_4": "78",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Chlorophyll"
            }
        ],
        "Young Couple Eve & Jon": [
            {
                "name": "Golduck",
                "Level": 50,
                "Attack_1": "93",
                "Attack_2": "103",
                "Attack_3": "244",
                "Attack_4": "154",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Damp"
            },
            {
                "name": "Psyduck",
                "Level": 50,
                "Attack_1": "103",
                "Attack_2": "244",
                "Attack_3": "154",
                "Attack_4": "56",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Bashful",
                "Ability": "Damp"
            }
        ],
        "Cool Couple Lex & Nya": [
            {
                "name": "Tauros",
                "Level": 52,
                "Attack_1": "34",
                "Attack_2": "208",
                "Attack_3": "45",
                "Attack_4": "111",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Modest",
                "Ability": "Intimidate"
            },
            {
                "name": "Miltank",
                "Level": 52,
                "Attack_1": "37",
                "Attack_2": "30",
                "Attack_3": "228",
                "Attack_4": "207",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Thick Fat"
            }
        ],
        "Tamer Evan": [
            {
                "name": "Sandslash",
                "Level": 48,
                "Attack_1": "40",
                "Attack_2": "163",
                "Attack_3": "129",
                "Attack_4": "154",
                "IVS": {
                    "hp": 4,
                    "atk": 4,
                    "def": 4,
                    "spa": 4,
                    "spd": 4,
                    "spe": 4
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Sand Veil"
            },
            {
                "name": "Lickitung",
                "Level": 48,
                "Attack_1": "35",
                "Attack_2": "50",
                "Attack_3": "21",
                "Attack_4": "103",
                "IVS": {
                    "hp": 4,
                    "atk": 4,
                    "def": 4,
                    "spa": 4,
                    "spd": 4,
                    "spe": 4
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quirky",
                "Ability": "Own Tempo"
            },
            {
                "name": "Ursaring",
                "Level": 49,
                "Attack_1": "156",
                "Attack_2": "163",
                "Attack_3": "173",
                "Attack_4": "37",
                "IVS": {
                    "hp": 4,
                    "atk": 4,
                    "def": 4,
                    "spa": 4,
                    "spd": 4,
                    "spe": 4
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Guts"
            }
        ],
        "Pokemon Ranger Jackson": [
            {
                "name": "Tangela",
                "Level": 49,
                "Attack_1": "21",
                "Attack_2": "72",
                "Attack_3": "20",
                "Attack_4": "275",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Exeggcute",
                "Level": 49,
                "Attack_1": "93",
                "Attack_2": "77",
                "Attack_3": "140",
                "Attack_4": "115",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Exeggutor",
                "Level": 49,
                "Attack_1": "93",
                "Attack_2": "121",
                "Attack_3": "79",
                "Attack_4": "23",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Chlorophyll"
            }
        ],
        "Pokemon Ranger Katelyn": [
            {
                "name": "Chansey",
                "Level": 52,
                "Attack_1": "121",
                "Attack_2": "111",
                "Attack_3": "107",
                "Attack_4": "135",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Natural Cure"
            }
        ],
        "Crush Girl Cyndy": [
            {
                "name": "Primeape",
                "Level": 48,
                "Attack_1": "69",
                "Attack_2": "99",
                "Attack_3": "238",
                "Attack_4": "207",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Vital Spirit"
            },
            {
                "name": "Hitmontop",
                "Level": 48,
                "Attack_1": "229",
                "Attack_2": "68",
                "Attack_3": "97",
                "Attack_4": "197",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Intimidate"
            },
            {
                "name": "Machoke",
                "Level": 48,
                "Attack_1": "279",
                "Attack_2": "233",
                "Attack_3": "66",
                "Attack_4": "238",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Guts"
            }
        ],
        "Cooltrainer Leroy": [
            {
                "name": "Rhydon",
                "Level": 47,
                "Attack_1": "36",
                "Attack_2": "32",
                "Attack_3": "350",
                "Attack_4": "184",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Lightningrod"
            },
            {
                "name": "Slowbro",
                "Level": 48,
                "Attack_1": "94",
                "Attack_2": "29",
                "Attack_3": "133",
                "Attack_4": "50",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Oblivious"
            },
            {
                "name": "Machoke",
                "Level": 48,
                "Attack_1": "238",
                "Attack_2": "233",
                "Attack_3": "279",
                "Attack_4": "69",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Timid",
                "Ability": "Guts"
            },
            {
                "name": "Kangaskhan",
                "Level": 47,
                "Attack_1": "146",
                "Attack_2": "44",
                "Attack_3": "203",
                "Attack_4": "179",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Early Bird"
            },
            {
                "name": "Ursaring",
                "Level": 50,
                "Attack_1": "163",
                "Attack_2": "185",
                "Attack_3": "173",
                "Attack_4": "156",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Guts"
            }
        ],
        "Cooltrainer Michelle": [
            {
                "name": "Persian",
                "Level": 47,
                "Attack_1": "163",
                "Attack_2": "103",
                "Attack_3": "185",
                "Attack_4": "44",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Limber"
            },
            {
                "name": "Dewgong",
                "Level": 47,
                "Attack_1": "58",
                "Attack_2": "36",
                "Attack_3": "196",
                "Attack_4": "45",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Thick Fat"
            },
            {
                "name": "Ninetales",
                "Level": 48,
                "Attack_1": "53",
                "Attack_2": "109",
                "Attack_3": "261",
                "Attack_4": "288",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Flash Fire"
            },
            {
                "name": "Rapidash",
                "Level": 48,
                "Attack_1": "340",
                "Attack_2": "97",
                "Attack_3": "83",
                "Attack_4": "36",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Run Away"
            },
            {
                "name": "Girafarig",
                "Level": 50,
                "Attack_1": "242",
                "Attack_2": "60",
                "Attack_3": "23",
                "Attack_4": "316",
                "IVS": {
                    "hp": 12,
                    "atk": 12,
                    "def": 12,
                    "spa": 12,
                    "spd": 12,
                    "spe": 12
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Impish",
                "Ability": "Inner Focus"
            }
        ],
        "Ruin Maniac Brandon": [
            {
                "name": "Onix",
                "Level": 50,
                "Attack_1": "225",
                "Attack_2": "201",
                "Attack_3": "21",
                "Attack_4": "231",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Rock Head"
            }
        ],
        "Gentleman Clifford": [
            {
                "name": "Marowak",
                "Level": 49,
                "Attack_1": "155",
                "Attack_2": "99",
                "Attack_3": "206",
                "Attack_4": "37",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Rock Head"
            },
            {
                "name": "Golduck",
                "Level": 49,
                "Attack_1": "93",
                "Attack_2": "103",
                "Attack_3": "244",
                "Attack_4": "154",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Damp"
            }
        ],
        "Painter Edna": [
            {
                "name": "Smeargle",
                "Level": 50,
                "Attack_1": "252",
                "Attack_2": "245",
                "Attack_3": "182",
                "Attack_4": "98",
                "IVS": {
                    "hp": 6,
                    "atk": 6,
                    "def": 6,
                    "spa": 6,
                    "spd": 6,
                    "spe": 6
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Own Tempo"
            }
        ],
        "Ruin Maniac Benjamin": [
            {
                "name": "Geodude",
                "Level": 48,
                "Attack_1": "89",
                "Attack_2": "350",
                "Attack_3": "205",
                "Attack_4": "120",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Rock Head"
            },
            {
                "name": "Graveler",
                "Level": 48,
                "Attack_1": "89",
                "Attack_2": "350",
                "Attack_3": "88",
                "Attack_4": "120",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Rock Head"
            },
            {
                "name": "Graveler",
                "Level": 48,
                "Attack_1": "89",
                "Attack_2": "350",
                "Attack_3": "88",
                "Attack_4": "120",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lax",
                "Ability": "Rock Head"
            }
        ],
        "Team Rocket Grunt 48": [
            {
                "name": "Houndour",
                "Level": 49,
                "Attack_1": "316",
                "Attack_2": "185",
                "Attack_3": "53",
                "Attack_4": "242",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Early Bird"
            },
            {
                "name": "Houndour",
                "Level": 49,
                "Attack_1": "316",
                "Attack_2": "185",
                "Attack_3": "53",
                "Attack_4": "242",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naughty",
                "Ability": "Early Bird"
            }
        ],
        "Team Rocket Grunt 49": [
            {
                "name": "Machop",
                "Level": 48,
                "Attack_1": "233",
                "Attack_2": "66",
                "Attack_3": "238",
                "Attack_4": "184",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Guts"
            },
            {
                "name": "Machop",
                "Level": 48,
                "Attack_1": "233",
                "Attack_2": "66",
                "Attack_3": "238",
                "Attack_4": "184",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Guts"
            },
            {
                "name": "Machoke",
                "Level": 48,
                "Attack_1": "279",
                "Attack_2": "233",
                "Attack_3": "66",
                "Attack_4": "238",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Guts"
            }
        ],
        "Team Rocket Grunt 50": [
            {
                "name": "Hypno",
                "Level": 49,
                "Attack_1": "96",
                "Attack_2": "94",
                "Attack_3": "244",
                "Attack_4": "207",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Insomnia"
            },
            {
                "name": "Hypno",
                "Level": 49,
                "Attack_1": "96",
                "Attack_2": "94",
                "Attack_3": "244",
                "Attack_4": "207",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Insomnia"
            }
        ],
        "Team Rocket Admin 1": [
            {
                "name": "Muk",
                "Level": 52,
                "Attack_1": "188",
                "Attack_2": "103",
                "Attack_3": "107",
                "Attack_4": "317",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Stench"
            },
            {
                "name": "Arbok",
                "Level": 53,
                "Attack_1": "188",
                "Attack_2": "44",
                "Attack_3": "89",
                "Attack_4": "231",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Intimidate"
            },
            {
                "name": "Vileplume",
                "Level": 54,
                "Attack_1": "188",
                "Attack_2": "202",
                "Attack_3": "79",
                "Attack_4": "78",
                "IVS": {
                    "hp": 18,
                    "atk": 18,
                    "def": 18,
                    "spa": 18,
                    "spd": 18,
                    "spe": 18
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Chlorophyll"
            }
        ],
        "Team Rocket Admin 2": [
            {
                "name": "Golbat",
                "Level": 53,
                "Attack_1": "109",
                "Attack_2": "188",
                "Attack_3": "314",
                "Attack_4": "247",
                "IVS": {
                    "hp": 24,
                    "atk": 24,
                    "def": 24,
                    "spa": 24,
                    "spd": 24,
                    "spe": 24
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Inner Focus"
            },
            {
                "name": "Weezing",
                "Level": 54,
                "Attack_1": "188",
                "Attack_2": "85",
                "Attack_3": "153",
                "Attack_4": "247",
                "IVS": {
                    "hp": 24,
                    "atk": 24,
                    "def": 24,
                    "spa": 24,
                    "spd": 24,
                    "spe": 24
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Levitate"
            },
            {
                "name": "Houndoom",
                "Level": 55,
                "Attack_1": "53",
                "Attack_2": "242",
                "Attack_3": "231",
                "Attack_4": "247",
                "IVS": {
                    "hp": 24,
                    "atk": 24,
                    "def": 24,
                    "spa": 24,
                    "spd": 24,
                    "spe": 24
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Early Bird"
            }
        ],
        "Scientist Gideon": [
            {
                "name": "Voltorb",
                "Level": 46,
                "Attack_1": "129",
                "Attack_2": "103",
                "Attack_3": "209",
                "Attack_4": "49",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Soundproof"
            },
            {
                "name": "Electrode",
                "Level": 46,
                "Attack_1": "209",
                "Attack_2": "49",
                "Attack_3": "103",
                "Attack_4": "268",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Soundproof"
            },
            {
                "name": "Magnemite",
                "Level": 46,
                "Attack_1": "103",
                "Attack_2": "129",
                "Attack_3": "209",
                "Attack_4": "86",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Magnet Pull"
            },
            {
                "name": "Magneton",
                "Level": 46,
                "Attack_1": "161",
                "Attack_2": "209",
                "Attack_3": "86",
                "Attack_4": "49",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Magnet Pull"
            },
            {
                "name": "Porygon",
                "Level": 46,
                "Attack_1": "161",
                "Attack_2": "160",
                "Attack_3": "105",
                "Attack_4": "60",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Trace"
            }
        ],
        "Lorelei 2": [
            {
                "name": "Dewgong",
                "Level": 64,
                "Attack_1": "58",
                "Attack_2": "57",
                "Attack_3": "324",
                "Attack_4": "104",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hardy",
                "Ability": "Thick Fat"
            },
            {
                "name": "Cloyster",
                "Level": 63,
                "Attack_1": "58",
                "Attack_2": "57",
                "Attack_3": "48",
                "Attack_4": "240",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Careful",
                "Ability": "Shell Armor"
            },
            {
                "name": "Piloswine",
                "Level": 63,
                "Attack_1": "59",
                "Attack_2": "89",
                "Attack_3": "38",
                "Attack_4": "157",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Oblivious"
            },
            {
                "name": "Jynx",
                "Level": 66,
                "Attack_1": "58",
                "Attack_2": "94",
                "Attack_3": "142",
                "Attack_4": "213",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Oblivious"
            },
            {
                "name": "Lapras",
                "Level": 66,
                "Attack_1": "58",
                "Attack_2": "57",
                "Attack_3": "94",
                "Attack_4": "87",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Water Absorb"
            }
        ],
        "Bruno 2": [
            {
                "name": "Steelix",
                "Level": 65,
                "Attack_1": "89",
                "Attack_2": "231",
                "Attack_3": "242",
                "Attack_4": "317",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Rock Head"
            },
            {
                "name": "Hitmonchan",
                "Level": 65,
                "Attack_1": "327",
                "Attack_2": "183",
                "Attack_3": "157",
                "Attack_4": "68",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Keen Eye"
            },
            {
                "name": "Hitmonlee",
                "Level": 65,
                "Attack_1": "25",
                "Attack_2": "193",
                "Attack_3": "89",
                "Attack_4": "157",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Limber"
            },
            {
                "name": "Steelix",
                "Level": 66,
                "Attack_1": "89",
                "Attack_2": "231",
                "Attack_3": "242",
                "Attack_4": "225",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Rock Head"
            },
            {
                "name": "Machamp",
                "Level": 68,
                "Attack_1": "238",
                "Attack_2": "89",
                "Attack_3": "280",
                "Attack_4": "157",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Calm",
                "Ability": "Guts"
            }
        ],
        "Agatha 2": [
            {
                "name": "Gengar",
                "Level": 66,
                "Attack_1": "247",
                "Attack_2": "94",
                "Attack_3": "109",
                "Attack_4": "95",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Brave",
                "Ability": "Levitate"
            },
            {
                "name": "Crobat",
                "Level": 66,
                "Attack_1": "188",
                "Attack_2": "314",
                "Attack_3": "247",
                "Attack_4": "109",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Lonely",
                "Ability": "Inner Focus"
            },
            {
                "name": "Misdreavus",
                "Level": 65,
                "Attack_1": "247",
                "Attack_2": "94",
                "Attack_3": "85",
                "Attack_4": "213",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Levitate"
            },
            {
                "name": "Arbok",
                "Level": 68,
                "Attack_1": "188",
                "Attack_2": "89",
                "Attack_3": "202",
                "Attack_4": "104",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Intimidate"
            },
            {
                "name": "Gengar",
                "Level": 70,
                "Attack_1": "247",
                "Attack_2": "94",
                "Attack_3": "85",
                "Attack_4": "188",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Levitate"
            }
        ],
        "Lance 2": [
            {
                "name": "Gyarados",
                "Level": 68,
                "Attack_1": "63",
                "Attack_2": "349",
                "Attack_3": "89",
                "Attack_4": "86",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Rash",
                "Ability": "Intimidate"
            },
            {
                "name": "Dragonite",
                "Level": 66,
                "Attack_1": "63",
                "Attack_2": "89",
                "Attack_3": "337",
                "Attack_4": "53",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Hasty",
                "Ability": "Inner Focus"
            },
            {
                "name": "Kingdra",
                "Level": 66,
                "Attack_1": "63",
                "Attack_2": "349",
                "Attack_3": "57",
                "Attack_4": "58",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Sassy",
                "Ability": "Swift Swim"
            },
            {
                "name": "Aerodactyl",
                "Level": 70,
                "Attack_1": "63",
                "Attack_2": "246",
                "Attack_3": "332",
                "Attack_4": "89",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Gentle",
                "Ability": "Rock Head"
            },
            {
                "name": "Dragonite",
                "Level": 72,
                "Attack_1": "63",
                "Attack_2": "200",
                "Attack_3": "85",
                "Attack_4": "58",
                "IVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Jolly",
                "Ability": "Inner Focus"
            }
        ],
        "Champion 2(Squirtle)": [
            {
                "name": "Heracross",
                "Level": 72,
                "Attack_1": "224",
                "Attack_2": "89",
                "Attack_3": "68",
                "Attack_4": "317",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Swarm"
            },
            {
                "name": "Alakazam",
                "Level": 73,
                "Attack_1": "94",
                "Attack_2": "247",
                "Attack_3": "347",
                "Attack_4": "115",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Synchronize"
            },
            {
                "name": "Tyranitar",
                "Level": 72,
                "Attack_1": "242",
                "Attack_2": "89",
                "Attack_3": "85",
                "Attack_4": "332",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Sand Stream"
            },
            {
                "name": "Arcanine",
                "Level": 73,
                "Attack_1": "245",
                "Attack_2": "315",
                "Attack_3": "332",
                "Attack_4": "231",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Intimidate"
            },
            {
                "name": "Gyarados",
                "Level": 73,
                "Attack_1": "56",
                "Attack_2": "349",
                "Attack_3": "89",
                "Attack_4": "63",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Intimidate"
            },
            {
                "name": "Venusaur",
                "Level": 75,
                "Attack_1": "76",
                "Attack_2": "188",
                "Attack_3": "89",
                "Attack_4": "241",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Docile",
                "Ability": "Overgrow"
            }
        ],
        "Champion 2(Charmander)": [
            {
                "name": "Heracross",
                "Level": 72,
                "Attack_1": "224",
                "Attack_2": "89",
                "Attack_3": "68",
                "Attack_4": "317",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Serious",
                "Ability": "Swarm"
            },
            {
                "name": "Alakazam",
                "Level": 73,
                "Attack_1": "94",
                "Attack_2": "247",
                "Attack_3": "347",
                "Attack_4": "115",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Relaxed",
                "Ability": "Synchronize"
            },
            {
                "name": "Tyranitar",
                "Level": 72,
                "Attack_1": "242",
                "Attack_2": "89",
                "Attack_3": "85",
                "Attack_4": "332",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Adamant",
                "Ability": "Sand Stream"
            },
            {
                "name": "Arcanine",
                "Level": 73,
                "Attack_1": "245",
                "Attack_2": "315",
                "Attack_3": "332",
                "Attack_4": "231",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Quiet",
                "Ability": "Intimidate"
            },
            {
                "name": "Exeggutor",
                "Level": 73,
                "Attack_1": "202",
                "Attack_2": "94",
                "Attack_3": "79",
                "Attack_4": "113",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Naive",
                "Ability": "Chlorophyll"
            },
            {
                "name": "Blastoise",
                "Level": 75,
                "Attack_1": "56",
                "Attack_2": "58",
                "Attack_3": "89",
                "Attack_4": "240",
                "IVS": {
                    "hp": 31,
                    "atk": 31,
                    "def": 31,
                    "spa": 31,
                    "spd": 31,
                    "spe": 31
                },
                "EVS": {
                    "hp": 0,
                    "atk": 0,
                    "def": 0,
                    "spa": 0,
                    "spd": 0,
                    "spe": 0
                },
                "Nature": "Mild",
                "Ability": "Torrent"
            }
        ]
    }

In [ ]:
import json
frlg_pokemon_data = {}
for i in frlg_trainer_data.keys():
    for j in frlg_trainer_data[i]:
        frlg_pokemon_data[j['name']+'-'+i] = j
json.dump(frlg_pokemon_data, open('frlg_pokemon_data.json', 'w'))